In [ ]:
import geopandas as gpd
import pandas as pd
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

In [ ]:
# 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

print("LIA筛选完成: {} 条".format(len(lia_matched)))

In [ ]:
# 筛选MainRGI对应的RGI冰川
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()

# 转WGS84
lia_matched = lia_matched.to_crs("EPSG:4326")
rgi_to_calibrate = rgi_to_calibrate.to_crs("EPSG:4326")

# 建立映射字典: LIA的UniqueID → MainRGI
lia_to_main_rgi = dict(zip(
    lia_matched['UniqueID'].astype(int),
    lia_matched['MainRGI']
))

print("RGI待校准: {} 条".format(len(rgi_to_calibrate)))
print("映射关系: {} 条".format(len(lia_to_main_rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, utils

WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_13/dh_13_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"
# 读取MB数据
dfs = []
for path in MB_PATHS:
    print("读取: {}".format(path))
    df = pd.read_csv(path)
    dfs.append(df)
mb_all = pd.concat(dfs, ignore_index=True)
mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')
print("MB数据: {} 条".format(len(mb_data)))
# 清理RGI数据
rgi = rgi_to_calibrate.copy()
rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print("有MB数据的RGI冰川: {}".format(len(rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, GlacierDirectory
cfg.initialize()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_rgi = workflow.init_glacier_directories(rgi, reset=False)
print("初始化完成: {}".format(len(gdirs_rgi)))

In [ ]:
for gdir in gdirs_rgi:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_rgi)))

In [ ]:
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_rgi:
    tasks.process_cru_data(gdir)
print("气候数据处理完成")

In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True

execute_entity_task(tasks.glacier_masks, gdirs_rgi)
execute_entity_task(tasks.compute_centerlines, gdirs_rgi)
execute_entity_task(tasks.initialize_flowlines, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_line, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_rgi)
execute_entity_task(tasks.catchment_area, gdirs_rgi)
execute_entity_task(tasks.catchment_intersections, gdirs_rgi)
execute_entity_task(tasks.catchment_width_geom, gdirs_rgi)
execute_entity_task(tasks.catchment_width_correction, gdirs_rgi)
print("预处理完成")

In [ ]:
cfg.PARAMS['use_winter_prcp_fac'] = False
cfg.PARAMS['prcp_fac'] = 8
from oggm.core.massbalance import mb_calibration_from_scalar_mb
import os
import json
success = 0
fail = 0
for gdir in gdirs_rgi:
    try:
        ref_mb = float(mb_data.loc[gdir.rgi_id, 'dmdtda_kg'])
        mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            ref_mb_years=(2000, 2020),
            overwrite_gdir=True,
            calibrate_param1='melt_f',
            calibrate_param2='temp_bias',
        )
        success += 1
    except Exception as e:
        fail += 1
        if fail <= 3:
            print("失败: {} - {}".format(gdir.rgi_id, str(e)[:100]))

print("\nCRU校准完成: 成功 {}, 失败 {}".format(success, fail))

# 验证几条
for gdir in gdirs_rgi[:5]:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    print("{}: melt_f={:.3f} temp_bias={:.3f} baseline={}".format(
        gdir.rgi_id, p['melt_f'], p['temp_bias'], p['baseline_climate_source']))

In [ ]:
lia_sorted = lia_matched.sort_values('UniqueID').copy()
lia_cooked = utils.cook_rgidf(
    lia_sorted,
    o1_region='15',
    bgndate='16500101',
    ids=lia_sorted['UniqueID'].astype(int).tolist(),
    assign_column_values={'LIAArea': 'Area'}
)

lia_cooked['UniqueID'] = lia_sorted['UniqueID'].values
lia_cooked['MainRGI'] = lia_sorted['MainRGI'].values

print("cook完成: {} 条".format(len(lia_cooked)))
print(lia_cooked[['RGIId', 'UniqueID', 'Area']].head())

In [ ]:
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_lia = workflow.init_glacier_directories(lia_cooked,reset= False)
print("LIA初始化完成: {} 条".format(len(gdirs_lia)))

In [ ]:
# 1. 重建lia_mb_params（用新的CRU校准参数）
import json, os, numpy as np

rgi_params_cru = {}
for gdir in gdirs_rgi:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        rgi_params_cru[gdir.rgi_id] = json.load(f)

lia_mb_params_cru = {}
missing = []
for idx, row in lia_matched.iterrows():
    unique_id = int(row['UniqueID'])
    main_rgi = row['MainRGI']
    if main_rgi in rgi_params_cru:
        p = rgi_params_cru[main_rgi]
        lia_mb_params_cru[unique_id] = {
            'melt_f': p['melt_f'],
            'prcp_fac': p['prcp_fac'],
            'temp_bias': p['temp_bias'],
        }
    else:
        missing.append((unique_id, main_rgi))

# 缺失的用区域平均
avg_melt_f = np.mean([p['melt_f'] for p in rgi_params_cru.values()])
avg_temp_bias = np.mean([p['temp_bias'] for p in rgi_params_cru.values()])
for uid, rgi in missing:
    lia_mb_params_cru[uid] = {'melt_f': avg_melt_f, 'prcp_fac': 8.0, 'temp_bias': avg_temp_bias}

print("LIA参数映射: {} 条, 缺失补充: {} 条".format(len(lia_mb_params_cru), len(missing)))
print("melt_f范围: {:.2f} ~ {:.2f}".format(
    min(p['melt_f'] for p in lia_mb_params_cru.values()),
    max(p['melt_f'] for p in lia_mb_params_cru.values())))

# 2. 写入LIA冰川的mb_calib.json
for gdir in gdirs_lia:
    unique_id = int(gdir.rgi_id.split('.')[-1])
    params = lia_mb_params_cru[unique_id]

    mb_calib = {
        'rgi_id': gdir.rgi_id,
        'bias': 0,
        'melt_f': params['melt_f'],
        'prcp_fac': params['prcp_fac'],
        'temp_bias': params['temp_bias'],
        'reference_mb': 0,
        'reference_mb_err': None,
        'reference_period': '2000-01-01_2020-01-01',
        'mb_global_params': {
            'temp_default_gradient': -0.0065,
            'temp_all_solid': 0.0,
            'temp_all_liq': 2.0,
            'temp_melt': -1.0,
        },
        'baseline_climate_source': 'CRU TS4.04',
    }

    fpath = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fpath, 'w') as f:
        json.dump(mb_calib, f, indent=2)

print("LIA冰川mb_calib写入完成: {} 条".format(len(gdirs_lia)))

# 验证IK
with open(os.path.join(gdirs_lia[0].dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
print("样本: melt_f={:.3f} baseline={}".format(p['melt_f'], p['baseline_climate_source']))

In [ ]:
for gdir in gdirs_lia:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_lia)))

In [ ]:
from oggm.workflow import execute_entity_task

execute_entity_task(tasks.glacier_masks, gdirs_lia)
execute_entity_task(tasks.compute_centerlines, gdirs_lia)
execute_entity_task(tasks.initialize_flowlines, gdirs_lia)
execute_entity_task(tasks.compute_downstream_line, gdirs_lia)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_lia)
execute_entity_task(tasks.catchment_area, gdirs_lia)
execute_entity_task(tasks.catchment_intersections, gdirs_lia)
execute_entity_task(tasks.catchment_width_geom, gdirs_lia)
execute_entity_task(tasks.catchment_width_correction, gdirs_lia)
print("预处理完成: {}".format(len(gdirs_lia)))

In [ ]:
from tqdm import tqdm

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'


# cfg.PARAMS['baseline_climate'] = 'CRU'
LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20
#
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_lia:
    tasks.process_cru_data(gdir)


In [ ]:
# 处理重建的气候数据
import xarray as xr
import os
from oggm.shop import gcm_climate

ensemble_dir = "/mnt/e/3spyderProject/HMA_500/Down_scale_new/out_data/oggm_climate/Ensemble"
climate_files = sorted([f for f in os.listdir(ensemble_dir) if f.endswith('.nc')])

for ci, cfile in enumerate(climate_files):
    suffix = '_' + cfile.replace('climate_oggm_', '').replace('.nc', '')
    fpath = os.path.join(ensemble_dir, cfile)

    print("\n[{}/{}] {} -> suffix: {}".format(ci+1, len(climate_files), cfile, suffix))

    ds_clim = xr.open_dataset(fpath, use_cftime=True)
    temp_k = ds_clim['temp'] + 273.15
    prcp = ds_clim['prcp']

    success = 0
    fail = 0

    for gdir in gdirs_lia:
        try:
            temp_sel = temp_k.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')
            prcp_sel = prcp.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')

            gcm_climate.process_gcm_data(
                gdir,
                temp=temp_sel,
                prcp=prcp_sel,
                year_range=('1961', '1990'),
                source='rec' + suffix,
                output_filesuffix=suffix,
            )
            success += 1
        except Exception as e:
            fail += 1
            if fail <= 3:
                print("  失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

    ds_clim.close()
    print("  完成: 成功 {}, 失败 {}".format(success, fail))

print("\n全部处理完成!")

In [ ]:
import json
import os

mb_global_params = {
    'temp_default_gradient': -0.0065,
    'temp_all_solid': 0.0,
    'temp_all_liq': 2.0,
    'temp_melt': -1.0
}

success = 0
for gdir in gdirs_lia:
    try:
        unique_id = int(gdir.rgi_id.split('.')[-1])
        params = lia_mb_params_cru[unique_id]

        mb_calib = {
            'rgi_id': gdir.rgi_id,
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
            'temp_bias': params['temp_bias'],
            'bias': 0,
            'reference_mb': 0,
            'reference_mb_err': None,
            'reference_period': '2000-01-01_2020-01-01',
            'mb_global_params': mb_global_params,
            'baseline_climate_source': 'CRU TS4.04',
        }

        fpath = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fpath, 'w') as f:
            json.dump(mb_calib, f, indent=2)
        success += 1
    except Exception as e:
        print("失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

print("mb_calib更新完成: {}".format(success))


# 2. 验证一下climate_historical和mb_calib的source是否匹配
import xarray as xr
gdir = gdirs_lia[0]
ds = xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc'))
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    mb = json.load(f)
print("\nclimate_historical source: {}".format(ds.attrs.get('climate_source')))
print("mb_calib source: {}".format(mb['baseline_climate_source']))
ds.close()

In [ ]:
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import tasks, cfg
import json, os, numpy as np, xarray as xr
from scipy.interpolate import interp1d
import pandas as pd
import pickle

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

windows = [1300, 1350, 1400, 1450, 1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]
offsets = [0, -1, -2, -3, -4]

# 选一条冰川
gdir = gdirs_lia[0]
uid = int(gdir.rgi_id.split('.')[-1])
original_tb = lia_mb_params_cru[uid]['temp_bias']
print("测试冰川: {}, original temp_bias={:.2f}".format(gdir.rgi_id, original_tb))

# ========== 第一部分：运行所有窗口×offset ==========
for y0 in windows:
    for offset in offsets:
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
            p = json.load(f)
        p['temp_bias'] = original_tb + offset
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
            json.dump(p, f, indent=2)

        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            execute_entity_task(
                tasks.run_constant_climate,
                [gdir],
                nyears=300,
                y0=y0,
                halfsize=30,
                climate_filename='gcm_data',
                climate_input_filesuffix='_ensemble_rec_ref',
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix,
                store_model_geometry=True,
                store_fl_diagnostics=True
            )
            ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix))
            pct = (ds['area_m2'].values[-1] - ds['area_m2'].values[0]) / ds['area_m2'].values[0] * 100
            print("y0={}, offset={}: area pct={:.1f}%".format(y0, offset, pct))
            ds.close()
        except Exception as e:
            print("y0={}, offset={}: FAILED - {}".format(y0, offset, e))

# 恢复temp_bias
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
p['temp_bias'] = original_tb
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
    json.dump(p, f, indent=2)
print("\n第一部分完成，temp_bias已恢复")

# ========== 第二部分：提取数据 ==========
save_dir = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single')
os.makedirs(save_dir, exist_ok=True)

# 2a. 保存时间序列（netCDF per window）
for y0 in windows:
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            f = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            ds = xr.open_dataset(f)
            ds_out = xr.Dataset({
                'area_m2': ('time', ds['area_m2'].values),
                'volume_m3': ('time', ds['volume_m3'].values),
                'length_m': ('time', ds['length_m'].values),
            }, coords={'time': np.arange(len(ds['area_m2'].values))})
            ds_out.attrs['rgi_id'] = gdir.rgi_id
            ds_out.attrs['y0'] = y0
            ds_out.attrs['offset'] = offset
            nc_path = os.path.join(save_dir, 'ts_w{}_offset{}.nc'.format(
                y0, str(offset).replace('.', '_').replace('-', 'n')))
            ds_out.to_netcdf(nc_path)
            ds.close()
        except:
            pass
    print("y0={} 时间序列保存完成".format(y0))

# 2b. 提取气候信息
climate_info = {}
for y0 in windows:
    try:
        fpath = gdir.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
        ds_clim = xr.open_dataset(fpath)
        t_start = '{}-01'.format(y0 - 30)
        t_end = '{}-12'.format(y0 + 30)
        ds_sel = ds_clim.sel(time=slice(t_start, t_end))
        climate_info[y0] = {
            'mean_temp': float(ds_sel['temp'].mean()),
            'mean_prcp': float(ds_sel['prcp'].mean()),
        }
        ds_clim.close()
    except Exception as e:
        climate_info[y0] = {'mean_temp': np.nan, 'mean_prcp': np.nan, 'error': str(e)}
print("气候信息提取完成")

# 2c. 冰川基础属性
params = lia_mb_params_cru[uid]
try:
    suffix0 = '_w1650_offset0'
    ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix0))
    lia_area = float(ds['area_m2'].values[0])
    lia_volume = float(ds['volume_m3'].values[0])
    lia_length = float(ds['length_m'].values[0])
    ds.close()
except:
    lia_area = lia_volume = lia_length = np.nan

glacier_info = {
    gdir.rgi_id: {
        'melt_f': params['melt_f'],
        'temp_bias_original': params['temp_bias'],
        'prcp_fac': params['prcp_fac'],
        'lia_area_m2': lia_area,
        'lia_volume_m3': lia_volume,
        'lia_length_m': lia_length,
    }
}

# 2d. 计算best_offset
all_results = {}
for y0 in windows:
    area_changes = {}
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix))
            area_init = float(ds['area_m2'].values[0])
            area_end = float(ds['area_m2'].values[-1])
            pct = (area_end - area_init) / area_init * 100
            ds.close()
            area_changes[offset] = pct
        except:
            pass

    if len(area_changes) < 2:
        all_results[y0] = {'best_offset': np.nan, 'note': 'insufficient_data'}
        continue

    offs = sorted(area_changes.keys())
    pcts = [area_changes[o] for o in offs]

    best_offset = np.nan
    for i in range(len(pcts) - 1):
        if pcts[i] * pcts[i+1] <= 0:
            f_interp = interp1d([pcts[i], pcts[i+1]], [offs[i], offs[i+1]])
            best_offset = float(f_interp(0))
            break

    if np.isnan(best_offset):
        if all(p < 0 for p in pcts):
            note = 'all_shrink'
        elif all(p > 0 for p in pcts):
            note = 'all_grow'
        else:
            note = 'no_crossing'
        all_results[y0] = {'best_offset': np.nan, 'note': note}
    else:
        all_results[y0] = {'best_offset': best_offset, 'note': 'success'}
    print("y0={}: best_offset={}, note={}".format(
        y0, all_results[y0]['best_offset'], all_results[y0]['note']))

# ========== 第三部分：汇总CSV ==========
rows = []
rgi_id = gdir.rgi_id
gi = glacier_info[rgi_id]

for y0 in windows:
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix))
            row = {
                'y0': y0,
                'rgi_id': rgi_id,
                'offset': offset,
                'area_init': float(ds['area_m2'].values[0]),
                'area_end': float(ds['area_m2'].values[-1]),
                'area_pct': float((ds['area_m2'].values[-1] - ds['area_m2'].values[0]) / ds['area_m2'].values[0] * 100),
                'volume_init': float(ds['volume_m3'].values[0]),
                'volume_end': float(ds['volume_m3'].values[-1]),
                'length_init': float(ds['length_m'].values[0]),
                'length_end': float(ds['length_m'].values[-1]),
                'melt_f': gi['melt_f'],
                'temp_bias_original': gi['temp_bias_original'],
                'prcp_fac': gi['prcp_fac'],
                'lia_area_m2': gi['lia_area_m2'],
                'actual_temp_bias': gi['temp_bias_original'] + offset,
                'best_offset': all_results[y0]['best_offset'],
                'note': all_results[y0]['note'],
                'window_mean_temp': climate_info.get(y0, {}).get('mean_temp'),
                'window_mean_prcp': climate_info.get(y0, {}).get('mean_prcp'),
            }
            ds.close()
            rows.append(row)
        except:
            pass

df = pd.DataFrame(rows)
csv_path = os.path.join(save_dir, 'window_offset_single_{}.csv'.format(rgi_id.replace('.', '_')))
df.to_csv(csv_path, index=False)
print("\nCSV保存到: {}".format(csv_path))

# 保存pickle
pkl_path = os.path.join(save_dir, 'window_offset_single_{}.pkl'.format(rgi_id.replace('.', '_')))
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'all_results': all_results,
        'glacier_info': glacier_info,
        'climate_info': climate_info,
        'windows': windows,
        'offsets': offsets,
        'rgi_id': rgi_id,
    }, f)
print("Pickle保存到: {}".format(pkl_path))
print("\n全部完成！{}".format(rgi_id))

In [ ]:
import xarray as xr
import numpy as np

fpath = gdir.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
ds = xr.open_dataset(fpath, use_cftime=True)

temp = ds['temp'].values
time = ds['time'].values
years = np.array([t.year for t in time])

unique_years = np.unique(years)
temp_annual = []
for yr in unique_years:
    temp_annual.append(temp[years == yr].mean())
temp_annual = np.array(temp_annual)

# 参考期
ref_mask = (unique_years >= 1961) & (unique_years <= 1990)
t_ref = temp_annual[ref_mask].mean()

# 各时期距平
periods = [(1270, 1330), (1320, 1380), (1370, 1430), (1420, 1480),
           (1470, 1530), (1520, 1580), (1570, 1630), (1620, 1680),
           (1670, 1730), (1720, 1780), (1770, 1830), (1820, 1880),
           (1870, 1930), (1920, 1980), (1960, 2000)]

print("参考期(1961-1990)平均温度: {:.2f}°C".format(t_ref))
print("\n{:<16} {:>10} {:>10}".format("时期", "平均温度", "距平"))
print("-" * 38)
for y1, y2 in periods:
    mask = (unique_years >= y1) & (unique_years <= y2)
    if mask.sum() > 0:
        t_mean = temp_annual[mask].mean()
        print("{:<16} {:>10.2f} {:>+10.2f}".format("{}-{}".format(y1, y2), t_mean, t_mean - t_ref))

ds.close()

In [ ]:
import json, os
import xarray as xr
import numpy as np
import pickle
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import tasks
import pandas as pd

# 加载之前的结果
pkl_path = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single',
                        'window_offset_single_RGI60-15_00007.pkl')
with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

all_results = data['all_results']
windows = data['windows']

gdir = gdirs_lia[0]
uid = int(gdir.rgi_id.split('.')[-1])
original_tb = lia_mb_params_cru[uid]['temp_bias']
rgi_area = gdir.rgi_area_m2 / 1e6

print("冰川: {}, RGI观测面积: {:.2f} km2, original temp_bias: {:.2f}\n".format(
    gdir.rgi_id, rgi_area, original_tb))

results_rows = []

print("{:<8} {:>12} {:>12} {:>12} {:>10} {:>10}".format(
    "t0", "best_offset", "LIA_area", "2000_area", "change%", "sim/RGI"))
print("-" * 68)

for t0 in windows:
    bo = all_results[t0]['best_offset']
    if np.isnan(bo):
        print("{:<8} skipped (no best_offset)".format(t0))
        continue

    # 设置best_offset
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    p['temp_bias'] = original_tb + bo
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
        json.dump(p, f, indent=2)

    # spin-up (带store_model_geometry)
    try:
        execute_entity_task(
            tasks.run_constant_climate,
            [gdir],
            nyears=300,
            y0=t0,
            halfsize=30,
            climate_filename='gcm_data',
            climate_input_filesuffix='_ensemble_rec_ref',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            output_filesuffix='_spinup_t0_{}'.format(t0),
            store_model_geometry=True,
        )
    except Exception as e:
        print("{:<8} spinup FAILED: {}".format(t0, e))
        continue

    # 正演到2000
    try:
        execute_entity_task(
            tasks.run_from_climate_data,
            [gdir],
            min_ys=t0,
            max_ys=2000,
            climate_filename='gcm_data',
            climate_input_filesuffix='_ensemble_rec_ref',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix='_spinup_t0_{}'.format(t0),
            output_filesuffix='_forward_t0_{}'.format(t0),
        )
    except Exception as e:
        print("{:<8} forward FAILED: {}".format(t0, e))
        continue

    # 提取结果
    try:
        ds = xr.open_dataset(gdir.get_filepath('model_diagnostics',
                                                filesuffix='_forward_t0_{}'.format(t0)))
        a_init = float(ds['area_m2'].values[0]) / 1e6
        a_end = float(ds['area_m2'].values[-1]) / 1e6
        v_init = float(ds['volume_m3'].values[0])
        v_end = float(ds['volume_m3'].values[-1])
        l_init = float(ds['length_m'].values[0])
        l_end = float(ds['length_m'].values[-1])
        change = (a_end - a_init) / a_init * 100
        ratio = a_end / rgi_area
        ds.close()

        print("{:<8} {:>+12.3f} {:>12.2f} {:>12.2f} {:>+10.1f} {:>10.2f}".format(
            t0, bo, a_init, a_end, change, ratio))

        results_rows.append({
            't0': t0,
            'best_offset': bo,
            'actual_temp_bias': original_tb + bo,
            'lia_area_km2': a_init,
            'end_area_km2': a_end,
            'area_change_pct': change,
            'sim_rgi_ratio': ratio,
            'rgi_area_km2': rgi_area,
            'lia_volume_m3': v_init,
            'end_volume_m3': v_end,
            'lia_length_m': l_init,
            'end_length_m': l_end,
        })
    except Exception as e:
        print("{:<8} extract FAILED: {}".format(t0, e))

# 恢复temp_bias
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
p['temp_bias'] = original_tb
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
    json.dump(p, f, indent=2)
print("\ntemp_bias已恢复")

# 保存结果
if results_rows:
    df = pd.DataFrame(results_rows)
    save_dir = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single')
    csv_path = os.path.join(save_dir, 'forward_results_{}.csv'.format(
        gdir.rgi_id.replace('.', '_')))
    df.to_csv(csv_path, index=False)
    print("\n结果保存到: {}".format(csv_path))

    # 找最佳t0
    best_idx = (df['sim_rgi_ratio'] - 1.0).abs().idxmin()
    best = df.loc[best_idx]
    print("\n最佳t0={}: sim/RGI={:.3f}, area_change={:+.1f}%, best_offset={:+.3f}".format(
        int(best['t0']), best['sim_rgi_ratio'], best['area_change_pct'], best['best_offset']))

In [ ]:
"""
可视化正演时间序列（修正版）
使用真实RGI观测面积而非LIA轮廓面积
"""
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pickle, os
from oggm import cfg

# ========== 加载数据 ==========
pkl_path = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single',
                        'window_offset_single_RGI60-15_00007.pkl')
with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

all_results = data['all_results']
windows = data['windows']
gdir = gdirs_lia[0]

# ===== 关键修正: 获取真实RGI观测面积 =====
uid = int(gdir.rgi_id.split('.')[-1])
main_rgi_id = lia_to_main_rgi[uid]  # LIA UniqueID → RGI ID

# 从gdirs_rgi中找到对应冰川的真实面积
rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
rgi_area = rgi_gdir.rgi_area_m2 / 1e6  # 真实RGI观测面积 (km2)
lia_outline_area = gdir.rgi_area_m2 / 1e6  # LIA轮廓面积 (km2)

print("冰川: {} → MainRGI: {}".format(gdir.rgi_id, main_rgi_id))
print("LIA轮廓面积: {:.2f} km², RGI观测面积: {:.2f} km²".format(lia_outline_area, rgi_area))

# ========== 读取各t0的正演时间序列 ==========
ts_data = {}
for t0 in windows:
    bo = all_results[t0]['best_offset']
    if np.isnan(bo):
        continue
    try:
        ds = xr.open_dataset(gdir.get_filepath('model_diagnostics',
                                                filesuffix='_forward_t0_{}'.format(t0)))
        n = len(ds['area_m2'].values)
        ts_data[t0] = {
            'years': np.arange(t0, t0 + n),
            'area_km2': ds['area_m2'].values / 1e6,
            'volume_km3': ds['volume_m3'].values / 1e9,
            'length_km': ds['length_m'].values / 1e3,
            'best_offset': bo,
        }
        ds.close()
    except Exception as e:
        print("t0={}: {}".format(t0, e))

# ========== 图1: 面积时间序列 ==========
fig, ax = plt.subplots(figsize=(12, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(ts_data)))

for i, t0 in enumerate(sorted(ts_data.keys())):
    d = ts_data[t0]
    ax.plot(d['years'], d['area_km2'], color=cmap[i],
            label='t0={} (Δ={:+.2f}°C)'.format(t0, d['best_offset']), linewidth=1.5)

# RGI真实观测面积
ax.axhline(rgi_area, color='red', linestyle='--', linewidth=2,
           label='RGI observed ({:.2f} km²)'.format(rgi_area))
# LIA轮廓面积（参考）
ax.axhline(lia_outline_area, color='blue', linestyle=':', linewidth=1.5, alpha=0.5,
           label='LIA outline ({:.2f} km²)'.format(lia_outline_area))

ax.set_xlabel('Year')
ax.set_ylabel('Area (km²)')
ax.set_title('{} → {} — Area evolution'.format(gdir.rgi_id, main_rgi_id))
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ========== 图2: sim/RGI ratio vs t0 ==========
fig, ax = plt.subplots(figsize=(10, 4))

t0s = sorted(ts_data.keys())
ratios = [ts_data[t0]['area_km2'][-1] / rgi_area for t0 in t0s]

ax.plot(t0s, ratios, 'ko-', markersize=8)
ax.axhline(1.0, color='red', linestyle='--', label='ratio = 1.0')
ax.fill_between([min(t0s)-25, max(t0s)+25], 0.95, 1.05, alpha=0.15, color='green')

best_i = np.argmin(np.abs(np.array(ratios) - 1.0))
ax.annotate('Best: t0={}, ratio={:.3f}'.format(t0s[best_i], ratios[best_i]),
            xy=(t0s[best_i], ratios[best_i]), fontsize=10,
            xytext=(30, 15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->'),
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

ax.set_xlabel('Starting window (t0)')
ax.set_ylabel('sim_2000 / RGI_observed area ratio')
ax.set_title('{} — sim/RGI ratio (using true RGI area)'.format(gdir.rgi_id))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ========== 汇总表 ==========
print("\n" + "=" * 85)
print("{:<6} {:>8} {:>10} {:>10} {:>12} {:>10}".format(
    "t0", "offset", "sim_LIA", "sim_2000", "RGI_obs", "sim/RGI"))
print("-" * 85)
for i, t0 in enumerate(t0s):
    d = ts_data[t0]
    sim_lia = d['area_km2'][0]
    sim_2000 = d['area_km2'][-1]
    r = sim_2000 / rgi_area
    mark = " ◀ BEST" if t0 == t0s[best_i] else ""
    print("{:<6} {:>+8.3f} {:>10.2f} {:>10.2f} {:>12.2f} {:>10.3f}{}".format(
        t0, d['best_offset'], sim_lia, sim_2000, rgi_area, r, mark))

In [ ]:
"""
batch_offset_search_parallel_test.py
=====================================
Jupyter Notebook 版 - 按 # %% 分cell复制运行
修改点：
  1. TEST_N = 20 只跑前20条（改为 None 全量跑）
  2. 多进程始终开启
  3. execute_entity_task 全部 try-except 防卡死
  4. task_timeout=600s
"""

# %% Cell 1 - 配置

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, workflow, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

# ---- 测试开关 ----
TEST_N = 20  # 设为 None 则全量运行

# ---- 多进程 ----
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['task_timeout'] = 600


class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)


WINDOWS = [1350, 1500, 1650, 1800]
OFFSETS = [round(-i * 0.2, 1) for i in range(11)]
SPINUP_YEARS = 500
HALFSIZE = 30
CLIMATE_SUFFIX = '_ensemble_rec_ref'

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'batch_offset_results')
os.makedirs(SAVE_DIR, exist_ok=True)

print("多进程={}, 进程数={}, timeout={}s".format(
    cfg.PARAMS['use_multiprocessing'], cfg.PARAMS['mp_processes'], cfg.PARAMS['task_timeout']))

# %% Cell 2 - 构建映射 & 筛选

print("[Step 1] 构建LIA→RGI面积映射...")

rgi_area_map = {}
rgi_id_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
        rgi_id_map[gdir.rgi_id] = main_rgi_id
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
        }
    except:
        pass

valid_gdirs = [g for g in gdirs_lia if g.rgi_id in rgi_area_map and g.rgi_id in original_params]
print("有效冰川: {} / {}".format(len(valid_gdirs), len(gdirs_lia)))

if TEST_N is not None:
    valid_gdirs = valid_gdirs[:TEST_N]
    print(">>> 测试模式: 取前 {} 条 <<<".format(len(valid_gdirs)))

print("运行: {} 冰川, {} 窗口 × {} offset = {} 轮".format(
    len(valid_gdirs), len(WINDOWS), len(OFFSETS), len(WINDOWS) * len(OFFSETS)))

# %% Cell 3 - 辅助函数

def batch_set_temp_bias(gdirs, offset):
    for gdir in gdirs:
        orig_tb = original_params[gdir.rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb + offset
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)


def batch_restore_temp_bias(gdirs):
    for gdir in gdirs:
        orig_tb = original_params[gdir.rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)


def extract_results(gdirs, y0, offset):
    suffix = '_forward_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))
    results = []
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            if not os.path.exists(fp):
                continue
            ds = xr.open_dataset(fp)
            rgi_a = rgi_area_map.get(rgi_id, np.nan)
            end_area = float(ds['area_m2'].values[-1]) / 1e6
            lia_area = float(ds['area_m2'].values[0]) / 1e6
            results.append({
                'rgi_id': rgi_id,
                'main_rgi_id': rgi_id_map.get(rgi_id, ''),
                'y0': y0, 'offset': offset,
                'actual_temp_bias': original_params[rgi_id]['temp_bias'] + offset,
                'original_temp_bias': original_params[rgi_id]['temp_bias'],
                'melt_f': original_params[rgi_id]['melt_f'],
                'prcp_fac': original_params[rgi_id]['prcp_fac'],
                'lia_area_km2': lia_area,
                'end_area_km2': end_area,
                'rgi_area_km2': rgi_a,
                'sim_rgi_ratio': end_area / rgi_a if rgi_a > 0 else np.nan,
                'area_change_pct': (end_area - lia_area) / lia_area * 100,
                'lia_volume_km3': float(ds['volume_m3'].values[0]) / 1e9,
                'end_volume_km3': float(ds['volume_m3'].values[-1]) / 1e9,
                'lia_length_km': float(ds['length_m'].values[0]) / 1e3,
                'end_length_km': float(ds['length_m'].values[-1]) / 1e3,
            })
            ds.close()
        except Exception as e:
            print("    [WARN] 提取失败 {}: {}".format(rgi_id, e))
    return results

# %% Cell 4 - 批量运行（主循环）

print("[Step 2] 开始批量运行...")

all_results = []
failed_rounds = []
total_rounds = len(WINDOWS) * len(OFFSETS)
round_count = 0
t_start = time.time()

for y0 in WINDOWS:
    for offset in OFFSETS:
        round_count += 1
        t_round = time.time()

        suffix_spinup = '_spinup_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))
        suffix_forward = '_forward_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))

        print("\n--- [{}/{}] y0={}, offset={:+.1f} ---".format(
            round_count, total_rounds, y0, offset))

        batch_set_temp_bias(valid_gdirs, offset)

        # Spin-up
        print("  Spin-up...")
        try:
            execute_entity_task(
                tasks.run_constant_climate, valid_gdirs,
                nyears=SPINUP_YEARS, y0=y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_spinup,
                store_model_geometry=True,
            )
        except Exception as e:
            print("  [ERROR] Spin-up异常: {}".format(e))
            failed_rounds.append({'y0': y0, 'offset': offset, 'stage': 'spinup', 'error': str(e)})
            batch_restore_temp_bias(valid_gdirs)
            gc.collect()
            continue

        # 筛选成功
        spinup_ok = []
        for gdir in valid_gdirs:
            try:
                fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
                if os.path.exists(fp):
                    spinup_ok.append(gdir)
            except:
                pass
        print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(valid_gdirs)))

        if len(spinup_ok) == 0:
            batch_restore_temp_bias(valid_gdirs)
            gc.collect()
            continue

        # 正演
        print("  正演到2000...")
        try:
            execute_entity_task(
                tasks.run_from_climate_data, spinup_ok,
                min_ys=y0, ye=2000,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                init_model_filesuffix=suffix_spinup,
                output_filesuffix=suffix_forward,
            )
        except Exception as e:
            print("  [ERROR] 正演异常: {}".format(e))
            failed_rounds.append({'y0': y0, 'offset': offset, 'stage': 'forward', 'error': str(e)})

        # 提取
        round_results = extract_results(valid_gdirs, y0, offset)
        all_results.extend(round_results)

        # 恢复
        batch_restore_temp_bias(valid_gdirs)

        # 进度
        elapsed = time.time() - t_round
        remaining = (total_rounds - round_count) * ((time.time() - t_start) / round_count) / 3600
        print("  成功: {}, 耗时: {:.1f}min, 剩余: {:.1f}h".format(
            len(round_results), elapsed / 60, remaining))

        # checkpoint
        pd.DataFrame(all_results).to_csv(
            os.path.join(SAVE_DIR, 'results_checkpoint.csv'), index=False)
        gc.collect()

print("\n完成! 耗时: {:.1f}h, 结果: {}, 失败: {}".format(
    (time.time() - t_start) / 3600, len(all_results), len(failed_rounds)))

if failed_rounds:
    for fr in failed_rounds:
        print("  失败: y0={}, offset={:+.1f}, {}: {}".format(
            fr['y0'], fr['offset'], fr['stage'], fr['error'][:100]))

# %% Cell 5 - 汇总分析

print("[Step 3] 汇总分析...")

if len(all_results) == 0:
    print("无结果，跳过")
else:
    df = pd.DataFrame(all_results)
    df.to_csv(os.path.join(SAVE_DIR, 'all_results.csv'), index=False)

    print("\n{:<6} {:>8} {:>10} {:>10} {:>10} {:>10} {:>8}".format(
        "y0", "offset", "mean_rat", "med_rat", "std_rat", "n_good", "n_total"))
    print("-" * 72)

    summary_rows = []
    for y0 in WINDOWS:
        for offset in OFFSETS:
            sub = df[(df['y0'] == y0) & (df['offset'] == offset)]
            ratios = sub['sim_rgi_ratio'].dropna()
            if len(ratios) == 0:
                continue
            n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
            row = {
                'y0': y0, 'offset': offset,
                'mean_ratio': ratios.mean(), 'median_ratio': ratios.median(),
                'std_ratio': ratios.std(), 'n_good_80_120': int(n_good),
                'n_total': len(ratios),
                'pct_good': n_good / len(ratios) * 100,
            }
            summary_rows.append(row)
            print("{:<6} {:>+8.1f} {:>10.3f} {:>10.3f} {:>10.3f} {:>10} {:>8}".format(
                y0, offset, row['mean_ratio'], row['median_ratio'],
                row['std_ratio'], row['n_good_80_120'], row['n_total']))

    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(os.path.join(SAVE_DIR, 'summary_by_window_offset.csv'), index=False)

    # 每个窗口最优
    print("\n最优offset（每个窗口）:")
    for y0 in WINDOWS:
        sub = df_summary[df_summary['y0'] == y0]
        if len(sub) == 0:
            continue
        best_idx = (sub['median_ratio'] - 1.0).abs().idxmin()
        best = sub.loc[best_idx]
        print("  y0={}: offset={:+.1f}, median={:.3f}, pct_good={:.1f}%".format(
            y0, best['offset'], best['median_ratio'], best['pct_good']))

    # 全局最优
    print("\n全局最优offset:")
    global_best_offset = None
    global_best_score = 999
    for offset in OFFSETS:
        sub = df_summary[df_summary['offset'] == offset]
        if len(sub) == 0:
            continue
        avg_med = sub['median_ratio'].mean()
        avg_pct = sub['pct_good'].mean()
        score = abs(avg_med - 1.0)
        if score < global_best_score:
            global_best_score = score
            global_best_offset = offset
        print("  offset={:+.1f}: avg_median={:.3f}, avg_pct_good={:.1f}%".format(
            offset, avg_med, avg_pct))

    if global_best_offset is not None:
        print("\n>>> 全局最优: offset={:+.1f} <<<".format(global_best_offset))

    # pickle
    pkl_path = os.path.join(SAVE_DIR, 'batch_results.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump({
            'all_results': all_results, 'summary': summary_rows,
            'failed_rounds': failed_rounds, 'windows': WINDOWS,
            'offsets': OFFSETS, 'rgi_area_map': rgi_area_map,
            'rgi_id_map': rgi_id_map, 'valid_glacier_count': len(valid_gdirs),
            'global_best_offset': global_best_offset, 'test_n': TEST_N,
        }, f)
    print("Pickle: {}".format(pkl_path))

print("\n实验完成!")

In [ ]:
"""
melt_f_scale_spinup_test.py
============================
验证 melt_f 缩放方案：
  1. 每条冰川用各自的平衡 scale 缩放 melt_f（使 LIA 几何上 MB≈0）
  2. spin-up 300年（恒定气候）
  3. 正演到 2000年
  4. 对比终点面积 vs RGI 观测面积
"""

# %% Cell 1 - 配置

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---- 参数 ----
TEST_N = 20
Y0 = 1500              # spin-up 用的气候中心年
HALFSIZE = 30
SPINUP_YEARS = 300
CLIMATE_SUFFIX = '_ensemble_rec_ref'
SCALE_FACTORS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

cfg.PARAMS['use_multiprocessing'] = False   # 20条用单进程更稳定
cfg.PARAMS['continue_on_error'] = True

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'meltf_scale_results')
os.makedirs(SAVE_DIR, exist_ok=True)

# %% Cell 2 - 筛选冰川 & 计算每条冰川的平衡 scale

print("[Step 1] 筛选冰川 & 计算平衡 scale...")

# 筛选有 model_flowlines 的冰川
test_gdirs = [g for g in gdirs_lia
              if g.rgi_id in original_params
              and os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))

# 计算每条冰川的平衡 scale（复用之前的诊断逻辑）
eq_scales = {}

for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    params = original_params[rgi_id]
    fls = gdir.read_pickle('model_flowlines')
    original_melt_f = params['melt_f']

    mbs = []
    for scale in SCALE_FACTORS:
        try:
            mb = MyMBModel(
                gdir, filename='gcm_data',
                input_filesuffix=CLIMATE_SUFFIX,
                melt_f=original_melt_f * scale,
                temp_bias=params['temp_bias'],
                prcp_fac=params['prcp_fac'],
            )
            avg = np.mean([mb.get_specific_mb(fls=fls, year=yr)
                           for yr in range(Y0 - HALFSIZE, Y0 + HALFSIZE + 1)])
        except:
            avg = np.nan
        mbs.append(avg)

    # 线性插值找平衡点
    found = False
    for i in range(len(mbs) - 1):
        if not np.isnan(mbs[i]) and not np.isnan(mbs[i+1]):
            if mbs[i] >= 0 >= mbs[i+1]:
                s0, s1 = SCALE_FACTORS[i], SCALE_FACTORS[i+1]
                frac = -mbs[i] / (mbs[i+1] - mbs[i])
                eq_scales[rgi_id] = s0 + frac * (s1 - s0)
                found = True
                break
    if not found:
        eq_scales[rgi_id] = 0.6  # fallback

    uid = rgi_id.split('.')[-1]
    print("  {} : scale={:.3f}, melt_f={:.2f} → {:.2f}".format(
        uid, eq_scales[rgi_id],
        original_melt_f, original_melt_f * eq_scales[rgi_id]))

# %% Cell 3 - 写入缩放后的 melt_f & 运行

print("\n[Step 2] 写入缩放后的 melt_f...")

def batch_set_meltf_scale(gdirs, scales_dict):
    """将每条冰川的 melt_f 缩放写入 mb_calib.json"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        scale = scales_dict[rgi_id]
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['melt_f'] = original_params[rgi_id]['melt_f'] * scale
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def batch_restore_meltf(gdirs):
    """恢复原始 melt_f"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['melt_f'] = original_params[rgi_id]['melt_f']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

# 写入
batch_set_meltf_scale(test_gdirs, eq_scales)

suffix_spinup = '_spinup_meltf_scale'
suffix_forward = '_forward_meltf_scale'

# Spin-up
print("\n[Step 3] Spin-up ({}年, y0={})...".format(SPINUP_YEARS, Y0))
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=suffix_spinup,
        store_model_geometry=True,
    )
except Exception as e:
    print("  Spin-up异常: {}".format(e))
print("  Spin-up耗时: {:.1f}min".format((time.time() - t0) / 60))

# 筛选成功
spinup_ok = []
for gdir in test_gdirs:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
        if os.path.exists(fp):
            spinup_ok.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(test_gdirs)))

# 正演
print("\n[Step 4] 正演 {} → 2000...".format(Y0))
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=suffix_spinup,
        output_filesuffix=suffix_forward,
    )
except Exception as e:
    print("  正演异常: {}".format(e))
print("  正演耗时: {:.1f}min".format((time.time() - t0) / 60))

# 恢复原始参数
batch_restore_meltf(test_gdirs)

# %% Cell 4 - 提取结果 & 分析

print("\n[Step 5] 提取结果...")

results = []
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_forward)
        if not os.path.exists(fp):
            continue
        ds = xr.open_dataset(fp)

        rgi_a = rgi_area_map.get(rgi_id, np.nan)
        end_area = float(ds['area_m2'].values[-1]) / 1e6
        lia_area = float(ds['area_m2'].values[0]) / 1e6
        spinup_area = lia_area  # 正演起点 = spin-up 终点

        # 也读 spin-up 起点面积（LIA 原始面积）
        fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
        ds_sp = xr.open_dataset(fp_sp)
        lia_init_area = float(ds_sp['area_m2'].values[0]) / 1e6
        spinup_end_area = float(ds_sp['area_m2'].values[-1]) / 1e6
        ds_sp.close()

        ratio = end_area / rgi_a if rgi_a > 0 else np.nan
        results.append({
            'uid': uid,
            'rgi_id': rgi_id,
            'scale': eq_scales[rgi_id],
            'orig_melt_f': original_params[rgi_id]['melt_f'],
            'scaled_melt_f': original_params[rgi_id]['melt_f'] * eq_scales[rgi_id],
            'lia_init_area': lia_init_area,
            'spinup_end_area': spinup_end_area,
            'spinup_change_pct': (spinup_end_area - lia_init_area) / lia_init_area * 100,
            'forward_end_area': end_area,
            'rgi_area': rgi_a,
            'sim_rgi_ratio': ratio,
        })
        ds.close()
    except Exception as e:
        print("  提取失败 {}: {}".format(uid, e))

df = pd.DataFrame(results)

# 打印结果
print("\n" + "=" * 110)
print("{:<8} {:>6} {:>8} {:>8} {:>10} {:>10} {:>8} {:>10} {:>8} {:>8}".format(
    "UID", "scale", "orig_mf", "new_mf",
    "LIA_init", "SP_end", "SP_chg%",
    "FW_end", "RGI", "ratio"))
print("-" * 110)

for _, r in df.iterrows():
    print("{:<8} {:>6.2f} {:>8.2f} {:>8.2f} {:>10.2f} {:>10.2f} {:>8.1f} {:>10.2f} {:>8.2f} {:>8.3f}".format(
        r['uid'], r['scale'], r['orig_melt_f'], r['scaled_melt_f'],
        r['lia_init_area'], r['spinup_end_area'], r['spinup_change_pct'],
        r['forward_end_area'], r['rgi_area'], r['sim_rgi_ratio']))

print("-" * 110)
ratios = df['sim_rgi_ratio'].dropna()
n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
print("均值 ratio: {:.3f}  中位数: {:.3f}  std: {:.3f}".format(
    ratios.mean(), ratios.median(), ratios.std()))
print("ratio 在 [0.8, 1.2] 内: {}/{} = {:.1f}%".format(
    n_good, len(ratios), n_good / len(ratios) * 100))
print("spin-up 面积变化均值: {:.1f}%".format(df['spinup_change_pct'].mean()))

df.to_csv(os.path.join(SAVE_DIR, 'meltf_scale_test_results.csv'), index=False)
print("\n结果已保存: {}".format(SAVE_DIR))

In [ ]:
"""
temp_offset_two_strategies_v3.py
=================================
逐冰川 temp_offset，对比两种正演策略：
  策略A: spin-up + 正演 都保留 offset（类似 melt_f 方案）
  策略B: spin-up 加 offset，正演去掉 offset（让真实气候驱动退缩）

流程：
  Step 0: 重建派生变量
  Step 1: 逐冰川扫描offset → 线性插值找 eq_offset（使MB≈0）
  Step 2: 用 eq_offset 做共用 spin-up（300年）
  Step 3A: 策略A 正演（保留offset）
  Step 3B: 策略B 正演（去掉offset）
  Step 4: 对比分析 & 保存

改进：
  - 多进程 + continue_on_error 防卡死
  - 排除极端offset冰川
  - 借鉴单冰川脚本的提取和保存结构
  - 每步打印耗时和成功率
"""

# %% Cell 1 - 配置与导入

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---------- 实验参数 ----------
TEST_N = 20                     # 测试冰川数量
Y0 = 1500                       # spin-up 中心年
HALFSIZE = 30                    # 气候窗口半宽
SPINUP_YEARS = 300               # spin-up 年数
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# offset 扫描档位（用于找 eq_offset）
OFFSETS_SCAN = [0, -0.5, -1.0, -1.5, -2.0, -2.5, -3.0]

# 排除极端 offset 的阈值
MAX_OFFSET_ABS = 2.5  # |offset| > 2.5K 的冰川排除

# ---------- 多进程配置 ----------
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True


SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'offset_two_strategies')
os.makedirs(SAVE_DIR, exist_ok=True)

# %% Cell 2 - 重建派生变量
# 假设 gdirs_lia, gdirs_rgi, lia_to_main_rgi, lia_mb_params_cru 已在环境中

print("=" * 70)
print("逐冰川 temp_offset 两策略对比实验")
print("=" * 70)

rgi_area_map = {}
rgi_id_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
        rgi_id_map[gdir.rgi_id] = main_rgi_id
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f':    params['melt_f'],
            'prcp_fac':  params['prcp_fac'],
        }
    except:
        pass

print("rgi_area_map : {}".format(len(rgi_area_map)))
print("original_params: {}".format(len(original_params)))

# %% Cell 3 - 筛选冰川 & 计算逐冰川平衡 offset

print("\n[Step 1] 筛选冰川 & 计算平衡 offset...")

test_gdirs = [g for g in gdirs_lia
              if g.rgi_id in original_params
              and g.rgi_id in rgi_area_map
              and os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))
print("扫描 offset: {}".format(OFFSETS_SCAN))

t_step1 = time.time()
eq_offsets = {}

for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    params = original_params[rgi_id]
    fls = gdir.read_pickle('model_flowlines')

    mbs = []
    for off in OFFSETS_SCAN:
        try:
            mb = MyMBModel(
                gdir, filename='gcm_data',
                input_filesuffix=CLIMATE_SUFFIX,
                melt_f=params['melt_f'],
                temp_bias=params['temp_bias'] + off,
                prcp_fac=params['prcp_fac'],
            )
            avg = np.mean([mb.get_specific_mb(fls=fls, year=yr)
                           for yr in range(Y0 - HALFSIZE, Y0 + HALFSIZE + 1)])
        except:
            avg = np.nan
        mbs.append(avg)

    # 线性插值找平衡点（MB从正变负或负变正的交叉点）
    found = False
    for i in range(len(mbs) - 1):
        if not np.isnan(mbs[i]) and not np.isnan(mbs[i+1]):
            if mbs[i] <= 0 <= mbs[i+1] or mbs[i] >= 0 >= mbs[i+1]:
                o0, o1 = OFFSETS_SCAN[i], OFFSETS_SCAN[i+1]
                frac = -mbs[i] / (mbs[i+1] - mbs[i])
                eq_offsets[rgi_id] = o0 + frac * (o1 - o0)
                found = True
                break
    if not found:
        eq_offsets[rgi_id] = -1.5  # fallback

    print("  {} : offset={:+.2f}K  MB={}".format(
        uid, eq_offsets[rgi_id],
        '  '.join(['{:+.0f}'.format(m) if not np.isnan(m) else 'nan' for m in mbs])))

print("  Step1 耗时: {:.1f}min".format((time.time() - t_step1) / 60))

# %% Cell 4 - 排除极端 offset 冰川

print("\n[Step 1.5] 排除极端 offset 冰川 (|offset| > {:.1f}K)...".format(MAX_OFFSET_ABS))

test_gdirs_safe = [g for g in test_gdirs
                   if abs(eq_offsets.get(g.rgi_id, -999)) <= MAX_OFFSET_ABS]

excluded = [g.rgi_id.split('.')[-1] for g in test_gdirs
            if g not in test_gdirs_safe]
if excluded:
    print("  排除 {} 条: {}".format(len(excluded), ', '.join(excluded)))
print("  保留 {} 条冰川进入 spin-up".format(len(test_gdirs_safe)))

# %% Cell 5 - 辅助函数

def set_per_glacier_offset(gdirs, offsets_dict):
    """每条冰川写入各自的 temp_bias + offset 到 mb_calib.json"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        off = offsets_dict[rgi_id]
        orig_tb = original_params[rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb + off
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def restore_original_params(gdirs):
    """恢复原始参数到 mb_calib.json"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = original_params[rgi_id]['temp_bias']
        p['melt_f'] = original_params[rgi_id]['melt_f']
        p['prcp_fac'] = original_params[rgi_id]['prcp_fac']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def extract_forward_results(gdirs, suffix_spinup, suffix_forward, strategy_name):
    """提取正演结果，借鉴单冰川脚本的详细字段"""
    results = []
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        try:
            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_forward)
            if not os.path.exists(fp_fw):
                continue
            ds_fw = xr.open_dataset(fp_fw)
            end_area = float(ds_fw['area_m2'].values[-1]) / 1e6
            end_vol = float(ds_fw['volume_m3'].values[-1]) / 1e9
            end_len = float(ds_fw['length_m'].values[-1]) / 1e3
            ds_fw.close()

            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
            ds_sp = xr.open_dataset(fp_sp)
            lia_init_area = float(ds_sp['area_m2'].values[0]) / 1e6
            spinup_end_area = float(ds_sp['area_m2'].values[-1]) / 1e6
            spinup_end_vol = float(ds_sp['volume_m3'].values[-1]) / 1e9
            ds_sp.close()

            rgi_a = rgi_area_map.get(rgi_id, np.nan)
            results.append({
                'uid': uid,
                'rgi_id': rgi_id,
                'strategy': strategy_name,
                'offset': eq_offsets[rgi_id],
                'lia_init_area': lia_init_area,
                'spinup_end_area': spinup_end_area,
                'spinup_chg_pct': (spinup_end_area - lia_init_area) / lia_init_area * 100,
                'spinup_end_vol_km3': spinup_end_vol,
                'forward_end_area': end_area,
                'forward_end_vol_km3': end_vol,
                'forward_end_len_km': end_len,
                'rgi_area': rgi_a,
                'sim_rgi_ratio': end_area / rgi_a if rgi_a > 0 else np.nan,
                'melt_f': original_params[rgi_id]['melt_f'],
                'temp_bias_original': original_params[rgi_id]['temp_bias'],
                'prcp_fac': original_params[rgi_id]['prcp_fac'],
            })
        except Exception as e:
            print("    提取失败 {}: {}".format(uid, e))
    return results

# %% Cell 6 - 共用 Spin-up

SUFFIX_SPINUP = '_spinup_perglacier_offset'

print("\n[Step 2] Spin-up (共用, {}年, y0={}, {}条冰川)...".format(
    SPINUP_YEARS, Y0, len(test_gdirs_safe)))

set_per_glacier_offset(test_gdirs_safe, eq_offsets)

t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs_safe,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX_SPINUP,
        store_model_geometry=True,  # 不存几何，省内存
    )
except Exception as e:
    print("  Spin-up异常: {}".format(e))
print("  Spin-up耗时: {:.1f}min".format((time.time() - t0) / 60))

# 检查哪些成功了
spinup_ok = []
for gdir in test_gdirs_safe:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        if os.path.exists(fp):
            spinup_ok.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(test_gdirs_safe)))

# 打印 spin-up 面积变化
print("\n  {:<8} {:>8} {:>10} {:>10} {:>8}".format(
    "UID", "offset", "init_km2", "end_km2", "chg%"))
print("  " + "-" * 50)
for gdir in spinup_ok:
    uid = gdir.rgi_id.split('.')[-1]
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        ds = xr.open_dataset(fp)
        a0 = float(ds['area_m2'].values[0]) / 1e6
        ae = float(ds['area_m2'].values[-1]) / 1e6
        pct = (ae - a0) / a0 * 100
        ds.close()
        print("  {:<8} {:>+8.2f} {:>10.2f} {:>10.2f} {:>+8.1f}%".format(
            uid, eq_offsets[gdir.rgi_id], a0, ae, pct))
    except:
        pass

# %% Cell 7 - 策略A: 正演保留 offset

print("\n[Step 3A] 策略A: 正演保留 offset ({} 条冰川)...".format(len(spinup_ok)))
SUFFIX_FW_A = '_forward_keep_offset'

# offset 已在 mb_calib.json 中，直接正演
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_SPINUP,
        output_filesuffix=SUFFIX_FW_A,
    )
except Exception as e:
    print("  正演A异常: {}".format(e))
print("  策略A耗时: {:.1f}min".format((time.time() - t0) / 60))

results_a = extract_forward_results(spinup_ok, SUFFIX_SPINUP, SUFFIX_FW_A, 'A_keep_offset')
print("  策略A成功提取: {} 条".format(len(results_a)))

# %% Cell 8 - 策略B: 正演去掉 offset

print("\n[Step 3B] 策略B: 正演去掉 offset ({} 条冰川)...".format(len(spinup_ok)))
SUFFIX_FW_B = '_forward_remove_offset'

# 恢复原始参数（去掉 offset）
restore_original_params(spinup_ok)

t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_SPINUP,
        output_filesuffix=SUFFIX_FW_B,
    )
except Exception as e:
    print("  正演B异常: {}".format(e))
print("  策略B耗时: {:.1f}min".format((time.time() - t0) / 60))

results_b = extract_forward_results(spinup_ok, SUFFIX_SPINUP, SUFFIX_FW_B, 'B_remove_offset')
print("  策略B成功提取: {} 条".format(len(results_b)))

# 恢复参数（安全）
restore_original_params(test_gdirs_safe)

# %% Cell 9 - 对比分析

print("\n" + "=" * 100)
print("[Step 4] 对比分析")
print("=" * 100)

df_a = pd.DataFrame(results_a)
df_b = pd.DataFrame(results_b)

# 逐冰川对比表
print("\n{:<8} {:>8} {:>10} {:>12} {:>12} {:>8} {:>10} {:>10}".format(
    "UID", "offset", "LIA", "SP_end", "A_end", "B_end", "RGI",
    "A_ratio", "B_ratio"))
print("-" * 100)

all_uids = sorted(set(
    (df_a['uid'].tolist() if len(df_a) > 0 else []) +
    (df_b['uid'].tolist() if len(df_b) > 0 else [])))

for uid in all_uids:
    ra = df_a[df_a['uid'] == uid] if len(df_a) > 0 else pd.DataFrame()
    rb = df_b[df_b['uid'] == uid] if len(df_b) > 0 else pd.DataFrame()

    if len(ra) == 0 and len(rb) == 0:
        continue

    ref = ra.iloc[0] if len(ra) > 0 else rb.iloc[0]
    a_end = ra.iloc[0]['forward_end_area'] if len(ra) > 0 else np.nan
    b_end = rb.iloc[0]['forward_end_area'] if len(rb) > 0 else np.nan
    a_rat = ra.iloc[0]['sim_rgi_ratio'] if len(ra) > 0 else np.nan
    b_rat = rb.iloc[0]['sim_rgi_ratio'] if len(rb) > 0 else np.nan

    print("{:<8} {:>+8.2f} {:>8.2f} {:>10.2f} {:>12.2f} {:>12.2f} {:>8.2f} {:>10.3f} {:>10.3f}".format(
        uid, ref['offset'], ref['lia_init_area'], ref['spinup_end_area'],
        a_end if not np.isnan(a_end) else 0,
        b_end if not np.isnan(b_end) else 0,
        ref['rgi_area'],
        a_rat if not np.isnan(a_rat) else 0,
        b_rat if not np.isnan(b_rat) else 0))

print("-" * 100)

# 汇总统计
for label, df_x in [('A (保留offset)', df_a), ('B (去掉offset)', df_b)]:
    if len(df_x) == 0:
        print("\n策略 {}: 无结果".format(label))
        continue
    ratios = df_x['sim_rgi_ratio'].dropna()
    n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
    print("\n策略 {}:".format(label))
    print("  均值ratio={:.3f}  中位数={:.3f}  std={:.3f}".format(
        ratios.mean(), ratios.median(), ratios.std()))
    print("  [0.8-1.2]内: {}/{}={:.1f}%".format(
        n_good, len(ratios), n_good / len(ratios) * 100 if len(ratios) > 0 else 0))
    print("  spin-up面积变化均值: {:.1f}%".format(df_x['spinup_chg_pct'].mean()))

# %% Cell 10 - 保存结果

# CSV
df_all = pd.concat([df_a, df_b])
csv_path = os.path.join(SAVE_DIR, 'two_strategies_results.csv')
df_all.to_csv(csv_path, index=False)
print("\nCSV 已保存: {}".format(csv_path))

# Pickle（包含完整中间信息）
pkl_data = {
    'eq_offsets': eq_offsets,
    'results_a': results_a,
    'results_b': results_b,
    'test_gdirs_ids': [g.rgi_id for g in test_gdirs],
    'safe_gdirs_ids': [g.rgi_id for g in test_gdirs_safe],
    'spinup_ok_ids': [g.rgi_id for g in spinup_ok],
    'config': {
        'TEST_N': TEST_N,
        'Y0': Y0,
        'HALFSIZE': HALFSIZE,
        'SPINUP_YEARS': SPINUP_YEARS,
        'OFFSETS_SCAN': OFFSETS_SCAN,
        'MAX_OFFSET_ABS': MAX_OFFSET_ABS,
    },
}
pkl_path = os.path.join(SAVE_DIR, 'two_strategies_full.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(pkl_data, f)
print("Pickle 已保存: {}".format(pkl_path))

print("\n" + "=" * 70)
print("全部完成！")
print("=" * 70)

In [ ]:
# 补充RGI冰川的反演和flowline初始化
from oggm.workflow import execute_entity_task

cfg.PATHS['working_dir'] = RGI_WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH

execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs_rgi)
execute_entity_task(tasks.prepare_for_inversion, gdirs_rgi)
execute_entity_task(tasks.mass_conservation_inversion, gdirs_rgi)
execute_entity_task(tasks.filter_inversion_output, gdirs_rgi)
execute_entity_task(tasks.init_present_time_glacier, gdirs_rgi)

# 验证
print(os.path.exists(os.path.join(gdirs_rgi[0].dir, 'model_flowlines.pkl')))

In [ ]:
"""
RGI冰川平衡态测试（稳健版）
============================
用RGI冰川 + CRU原始校准参数 + 现代气候，跑constant_climate到稳态
看平衡态面积 vs RGI面积的比值
"""

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

# ============================================================
# 0. 配置
# ============================================================

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

SPINUP_YEARS = 500
Y0 = 1985
HALFSIZE = 15
CLIMATE_FILENAME = 'climate_historical'
SUFFIX = '_equilibrium_test'

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['task_timeout'] = 600

# 测试开关
TEST_N = 20  # 设为 None 全量跑
test_gdirs = gdirs_rgi[:TEST_N] if TEST_N else gdirs_rgi

print("=" * 60)
print("RGI冰川平衡态测试")
print("=" * 60)
print("冰川数量: {}".format(len(test_gdirs)))
print("气候: CRU {}-{}".format(Y0 - HALFSIZE, Y0 + HALFSIZE))
print("spin-up: {}年".format(SPINUP_YEARS))
print("=" * 60)

# ============================================================
# 1. 记录RGI面积
# ============================================================
rgi_areas = {}
for gdir in test_gdirs:
    rgi_areas[gdir.rgi_id] = gdir.rgi_area_m2 / 1e6

print("已记录 {} 条冰川, 面积: {:.2f} - {:.2f} km²".format(
    len(rgi_areas), min(rgi_areas.values()), max(rgi_areas.values())))

# ============================================================
# 2. 运行constant_climate（多进程 + 回退单条）
# ============================================================
print("\n[运行] constant_climate {}年...".format(SPINUP_YEARS))
t0 = time.time()

try:
    execute_entity_task(
        tasks.run_constant_climate,
        test_gdirs,
        nyears=SPINUP_YEARS,
        y0=Y0,
        halfsize=HALFSIZE,
        climate_filename=CLIMATE_FILENAME,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX,
        store_model_geometry=True,
    )
    print("  多进程完成, 耗时 {:.1f}min".format((time.time() - t0) / 60))
except Exception as e:
    print("  [WARN] 多进程异常: {}".format(e))
    print("  [回退] 逐条运行...")
    failed_single = []
    for i, gdir in enumerate(test_gdirs):
        try:
            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS,
                y0=Y0,
                halfsize=HALFSIZE,
                climate_filename=CLIMATE_FILENAME,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=SUFFIX,
                store_model_geometry=True,
            )
        except Exception as e2:
            failed_single.append(gdir.rgi_id)
        if (i + 1) % 10 == 0:
            print("    {}/{}".format(i + 1, len(test_gdirs)))
    print("  单进程完成, 失败: {}".format(len(failed_single)))

print("总耗时: {:.1f}min".format((time.time() - t0) / 60))

# ============================================================
# 3. 提取结果
# ============================================================
print("\n[提取结果]...")

results = []
skip_count = 0
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    rgi_area = rgi_areas[rgi_id]
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX)
        if not os.path.exists(fp):
            skip_count += 1
            continue
        ds = xr.open_dataset(fp)

        area_init = float(ds.area_m2.values[0]) / 1e6
        area_final = float(ds.area_m2.values[-1]) / 1e6
        vol_init = float(ds.volume_m3.values[0]) / 1e9
        vol_final = float(ds.volume_m3.values[-1]) / 1e9

        # 稳态判断
        if len(ds.area_m2) > 50:
            a50 = ds.area_m2.values[-50:] / 1e6
            trend = (a50[-1] - a50[0]) / a50[0] * 100
        else:
            trend = np.nan

        results.append({
            'rgi_id': rgi_id,
            'rgi_area_km2': rgi_area,
            'init_area_km2': area_init,
            'final_area_km2': area_final,
            'ratio': area_final / rgi_area,
            'change_pct': (area_final - area_init) / area_init * 100,
            'last50yr_trend': trend,
            'vol_init_km3': vol_init,
            'vol_final_km3': vol_final,
        })
        ds.close()
    except Exception as e:
        skip_count += 1

df = pd.DataFrame(results)
print("成功: {}, 跳过: {}".format(len(df), skip_count))

# ============================================================
# 4. 结果输出
# ============================================================
if len(df) > 0:
    print("\n" + "=" * 60)
    print("平衡态面积 / RGI面积")
    print("=" * 60)
    print("  均值:   {:.4f}".format(df['ratio'].mean()))
    print("  中位数: {:.4f}".format(df['ratio'].median()))
    print("  标准差: {:.4f}".format(df['ratio'].std()))
    print("  范围:   {:.4f} - {:.4f}".format(df['ratio'].min(), df['ratio'].max()))

    print("\n面积变化 (init→final): 均值 {:.2f}%, 中位数 {:.2f}%".format(
        df['change_pct'].mean(), df['change_pct'].median()))

    print("最后50年趋势: 均值 {:.4f}%, |>1%|的冰川: {}".format(
        df['last50yr_trend'].mean(), (df['last50yr_trend'].abs() > 1).sum()))

    # ratio分布
    print("\n--- ratio分布 ---")
    bins = [0, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 2.0]
    labels = ['<0.7', '0.7-0.8', '0.8-0.85', '0.85-0.9', '0.9-0.95',
              '0.95-1.0', '1.0-1.05', '1.05-1.1', '>1.1']
    df['ratio_bin'] = pd.cut(df['ratio'], bins=bins, labels=labels)
    print(df['ratio_bin'].value_counts().sort_index().to_string())

    # 逐冰川
    print("\n{:<15} {:>9} {:>9} {:>9} {:>8} {:>10}".format(
        'RGI_ID', 'RGI_km2', 'Init_km2', 'End_km2', 'Ratio', 'Trend_%'))
    print("-" * 65)
    for _, r in df.sort_values('ratio').iterrows():
        print("{:<15} {:>9.2f} {:>9.2f} {:>9.2f} {:>8.4f} {:>10.4f}".format(
            r['rgi_id'].split('.')[-1], r['rgi_area_km2'], r['init_area_km2'],
            r['final_area_km2'], r['ratio'], r['last50yr_trend']))

    # 关键对比
    eq = df['ratio'].mean()
    print("\n" + "=" * 60)
    print("本实验 ratio均值: {:.4f}".format(eq))
    print("闭环测试 ratio均值: ~0.86")
    if abs(eq - 0.86) < 0.05:
        print("→ 接近! 0.86主要来自参数瞬态性")
    elif eq > 0.92:
        print("→ 平衡态接近RGI, 0.86中有路径依赖效应")
    else:
        print("→ 平衡态也偏小, 参数瞬态性是主因")

    save_path = os.path.join(cfg.PATHS['working_dir'], 'rgi_equilibrium_results.csv')
    df.to_csv(save_path, index=False)
    print("\n已保存: {}".format(save_path))

print("\n实验完成.")

In [ ]:
import os
gdir = gdirs_lia[0]  # 或 gdirs_rgi[0]
for f in os.listdir(gdir.dir):
    if 'gcm_data' in f or 'climate' in f:
        print(f)

In [ ]:
"""
GCM past1000 数据处理 + 与重建气候振幅对比
=============================================
1. 将6个GCM past1000数据降尺度到冰川点
2. 对比各GCM与ensemble_rec_ref的温度振幅
3. 判断重建气候是否低估了LIA-现代温差
"""

# %% Cell 1 - 处理GCM数据到冰川点

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from oggm import cfg, tasks

# GCM列表
GCM_LIST = ['BCC-CSM1-1', 'CCSM4', 'MPI-ESM-P', 'GISS-E2-R', 'IPSL-CM5A-LR', 'CESM']

# 路径（WSL格式）
base_path_temp = '/mnt/e/2Data/9DEM/oggm_related/6types_GCM/01Combine850-2000/tas'
base_path_precip = '/mnt/e/2Data/9DEM/oggm_related/6types_GCM/01Combine850-2000/pr'

# 只处理一条代表性冰川即可（所有冰川在同一区域，气候差异不大）
# 用RGI冰川，因为有CRU baseline
test_gdirs_gcm = gdirs_rgi[:5]  # 取几条，防止单条失败

print("处理GCM past1000数据...")
print("冰川数: {}".format(len(test_gdirs_gcm)))

for gcm in GCM_LIST:
    fpath_temp = os.path.join(base_path_temp,
        'combined_tas_Amon_{}_past1000_r1i1p1_085001-200512.nc'.format(gcm))
    fpath_precip = os.path.join(base_path_precip,
        'combined_pr_Amon_{}_past1000_r1i1p1_085001-200512.nc'.format(gcm))

    if not os.path.exists(fpath_temp):
        print("  [SKIP] {} 温度文件不存在: {}".format(gcm, fpath_temp))
        continue
    if not os.path.exists(fpath_precip):
        print("  [SKIP] {} 降水文件不存在: {}".format(gcm, fpath_precip))
        continue

    print("  处理 {} ...".format(gcm), end='')
    success = 0
    for gdir in test_gdirs_gcm:
        try:
            tasks.process_cmip_data(
                gdir,
                fpath_temp=fpath_temp,
                fpath_precip=fpath_precip,
                filesuffix='_{}'.format(gcm),
            )
            success += 1
        except Exception as e:
            pass
    print(" 成功 {}/{}".format(success, len(test_gdirs_gcm)))

print("GCM处理完成")

# %% Cell 2 - 读取所有气候数据

print("\n读取气候数据...")

# 找一条成功处理的冰川
use_gdir = None
for gdir in test_gdirs_gcm:
    try:
        fp = gdir.get_filepath('gcm_data', filesuffix='_{}'.format(GCM_LIST[0]))
        if os.path.exists(fp):
            use_gdir = gdir
            break
    except:
        pass

if use_gdir is None:
    print("ERROR: 没有成功处理的冰川!")
else:
    print("使用冰川: {}".format(use_gdir.rgi_id))

# 读取重建气候（从LIA冰川gdir）
gdir_lia = gdirs_lia[0]
fp_rec = gdir_lia.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
ds_rec = xr.open_dataset(fp_rec)
temp_rec = ds_rec['temp'].groupby('time.year').mean()
prcp_rec = ds_rec['prcp'].groupby('time.year').sum()

# 读取各GCM
gcm_temps = {}
gcm_prcps = {}
for gcm in GCM_LIST:
    try:
        fp = use_gdir.get_filepath('gcm_data', filesuffix='_{}'.format(gcm))
        ds = xr.open_dataset(fp)
        gcm_temps[gcm] = ds['temp'].groupby('time.year').mean()
        gcm_prcps[gcm] = ds['prcp'].groupby('time.year').sum()
        ds.close()
        print("  {} : {}-{}".format(gcm,
            int(gcm_temps[gcm].year.values[0]),
            int(gcm_temps[gcm].year.values[-1])))
    except Exception as e:
        print("  {} : 读取失败 - {}".format(gcm, e))

# 读取CRU（如果有）
try:
    fp_cru = use_gdir.get_filepath('climate_historical')
    ds_cru = xr.open_dataset(fp_cru)
    temp_cru = ds_cru['temp'].groupby('time.year').mean()
    prcp_cru = ds_cru['prcp'].groupby('time.year').sum()
    print("  CRU : {}-{}".format(
        int(temp_cru.year.values[0]), int(temp_cru.year.values[-1])))
    ds_cru.close()
    has_cru = True
except:
    has_cru = False
    print("  CRU : 无")

ds_rec.close()
print("数据加载完成, GCM: {}, CRU: {}".format(len(gcm_temps), has_cru))

# %% Cell 3 - 各时段温度对比

print("\n" + "=" * 90)
print("各数据源关键时段温度对比")
print("=" * 90)

periods = {
    'LIA_cold(1550-1700)': (1550, 1700),
    'LIA_peak(1600-1650)': (1600, 1650),
    '1800-1900': (1800, 1900),
    '1900-1950': (1900, 1950),
    '1950-2000': (1950, 2000),
    '1970-2000': (1970, 2000),
}

# 表头
header = "{:<22}".format("时段")
header += "{:>12}".format("Rec_ref")
for gcm in GCM_LIST:
    header += "{:>12}".format(gcm[:8])
if has_cru:
    header += "{:>12}".format("CRU")
print(header)
print("-" * len(header))

period_data = {}  # {period_name: {source: mean_temp}}
for pname, (y1, y2) in periods.items():
    row = "{:<22}".format(pname)

    # 重建气候
    sub = temp_rec.sel(year=slice(y1, y2))
    t_rec = float(sub.mean()) if len(sub) > 0 else np.nan
    row += "{:>12.3f}".format(t_rec)
    period_data.setdefault(pname, {})['Rec_ref'] = t_rec

    # GCM
    for gcm in GCM_LIST:
        if gcm in gcm_temps:
            sub = gcm_temps[gcm].sel(year=slice(y1, y2))
            t = float(sub.mean()) if len(sub) > 0 else np.nan
            row += "{:>12.3f}".format(t)
            period_data[pname][gcm] = t
        else:
            row += "{:>12}".format("N/A")

    # CRU
    if has_cru:
        sub = temp_cru.sel(year=slice(y1, y2))
        t = float(sub.mean()) if len(sub) > 0 else np.nan
        row += "{:>12.3f}".format(t)
        period_data[pname]['CRU'] = t

    print(row)

# %% Cell 4 - 温差对比（核心）

print("\n" + "=" * 90)
print("LIA → 现代 温差对比 (ΔT = T_modern - T_LIA)")
print("=" * 90)

# 定义温差计算
delta_pairs = [
    ('LIA_cold→1970-2000', 'LIA_cold(1550-1700)', '1970-2000'),
    ('LIA_peak→1970-2000', 'LIA_peak(1600-1650)', '1970-2000'),
    ('LIA_cold→1950-2000', 'LIA_cold(1550-1700)', '1950-2000'),
    ('1900-1950→1970-2000', '1900-1950', '1970-2000'),
]

header = "{:<28}".format("温差")
header += "{:>10}".format("Rec_ref")
for gcm in GCM_LIST:
    header += "{:>10}".format(gcm[:8])
if has_cru:
    header += "{:>10}".format("CRU")
print(header)
print("-" * len(header))

dt_results = {}
for label, p_cold, p_warm in delta_pairs:
    row = "{:<28}".format(label)

    # 重建
    if p_cold in period_data and p_warm in period_data:
        dt_rec = period_data[p_warm].get('Rec_ref', np.nan) - period_data[p_cold].get('Rec_ref', np.nan)
        row += "{:>+10.3f}".format(dt_rec)
        dt_results.setdefault(label, {})['Rec_ref'] = dt_rec
    else:
        row += "{:>10}".format("N/A")

    # GCM
    for gcm in GCM_LIST:
        if p_cold in period_data and p_warm in period_data:
            t1 = period_data[p_cold].get(gcm, np.nan)
            t2 = period_data[p_warm].get(gcm, np.nan)
            dt = t2 - t1
            row += "{:>+10.3f}".format(dt)
            dt_results.setdefault(label, {})[gcm] = dt
        else:
            row += "{:>10}".format("N/A")

    # CRU
    if has_cru and p_cold in period_data and p_warm in period_data:
        t1 = period_data[p_cold].get('CRU', np.nan)
        t2 = period_data[p_warm].get('CRU', np.nan)
        if not np.isnan(t1) and not np.isnan(t2):
            dt = t2 - t1
            row += "{:>+10.3f}".format(dt)
            dt_results.setdefault(label, {})['CRU'] = dt
        else:
            row += "{:>10}".format("N/A")

    print(row)

# 汇总
print("\n--- LIA_cold→1970-2000 温差汇总 ---")
key = 'LIA_cold→1970-2000'
if key in dt_results:
    vals = dt_results[key]
    rec_dt = vals.get('Rec_ref', np.nan)
    gcm_dts = [vals[g] for g in GCM_LIST if g in vals and not np.isnan(vals[g])]
    if gcm_dts:
        print("  重建气候 ΔT: {:+.3f}K".format(rec_dt))
        print("  GCM均值 ΔT:  {:+.3f}K".format(np.mean(gcm_dts)))
        print("  GCM范围:     {:+.3f} ~ {:+.3f}K".format(min(gcm_dts), max(gcm_dts)))
        print("  GCM标准差:   {:.3f}K".format(np.std(gcm_dts)))
        if not np.isnan(rec_dt):
            ratio = rec_dt / np.mean(gcm_dts) if np.mean(gcm_dts) != 0 else np.nan
            print("  重建/GCM比值: {:.2f}".format(ratio))
            if ratio < 0.8:
                print("  → 重建气候振幅明显偏小!")
            elif ratio > 1.2:
                print("  → 重建气候振幅偏大")
            else:
                print("  → 重建气候与GCM振幅接近")

# %% Cell 5 - 50年滑动平均振幅对比

print("\n" + "=" * 90)
print("50年滑动平均最大振幅对比")
print("=" * 90)

def get_max_amplitude(temp_annual, window=50):
    """计算滑动平均的最大振幅"""
    smoothed = temp_annual.rolling(year=window, center=True).mean()
    vals = smoothed.values
    valid = vals[~np.isnan(vals)]
    if len(valid) > 0:
        yrs = smoothed.year.values[~np.isnan(vals)]
        min_idx = np.argmin(valid)
        max_idx = np.argmax(valid)
        return {
            'amplitude': float(valid[max_idx] - valid[min_idx]),
            'coldest_year': int(yrs[min_idx]),
            'warmest_year': int(yrs[max_idx]),
            'coldest_temp': float(valid[min_idx]),
            'warmest_temp': float(valid[max_idx]),
        }
    return None

print("{:<15} {:>10} {:>10} {:>12} {:>12}".format(
    "数据源", "振幅(K)", "最冷年", "最暖年", "冷→暖"))
print("-" * 62)

# 重建
amp = get_max_amplitude(temp_rec)
if amp:
    print("{:<15} {:>10.3f} {:>10} {:>12} {:>12}".format(
        "Rec_ref", amp['amplitude'], amp['coldest_year'],
        amp['warmest_year'], "{}→{}".format(amp['coldest_year'], amp['warmest_year'])))

# GCM
gcm_amplitudes = []
for gcm in GCM_LIST:
    if gcm in gcm_temps:
        amp = get_max_amplitude(gcm_temps[gcm])
        if amp:
            gcm_amplitudes.append(amp['amplitude'])
            print("{:<15} {:>10.3f} {:>10} {:>12} {:>12}".format(
                gcm[:14], amp['amplitude'], amp['coldest_year'],
                amp['warmest_year'], "{}→{}".format(amp['coldest_year'], amp['warmest_year'])))

if gcm_amplitudes:
    print("-" * 62)
    print("{:<15} {:>10.3f}".format("GCM均值", np.mean(gcm_amplitudes)))
    print("{:<15} {:>10.3f} ~ {:.3f}".format("GCM范围", min(gcm_amplitudes), max(gcm_amplitudes)))

# %% Cell 6 - 可视化

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# --- 温度时间序列 ---
ax = axes[0]

# 30年滑动平均
window = 30

# 重建气候
rec_smooth = temp_rec.rolling(year=window, center=True).mean()
ax.plot(temp_rec.year.values, rec_smooth.values,
        color='black', linewidth=2.5, label='Rec_ref (ensemble)', zorder=10)

# GCM
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628']
for i, gcm in enumerate(GCM_LIST):
    if gcm in gcm_temps:
        smooth = gcm_temps[gcm].rolling(year=window, center=True).mean()
        ax.plot(gcm_temps[gcm].year.values, smooth.values,
                color=colors[i % len(colors)], linewidth=1.2, alpha=0.7,
                label=gcm)

# CRU
if has_cru:
    cru_smooth = temp_cru.rolling(year=window, center=True).mean()
    ax.plot(temp_cru.year.values, cru_smooth.values,
            color='green', linewidth=2, linestyle='--', label='CRU obs', zorder=9)

ax.set_xlim(1000, 2005)
ax.set_ylabel('Temperature (°C)')
ax.set_title('Temperature: Reconstructed vs GCM past1000 ({}yr smoothed)'.format(window))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# LIA标注
ax.axvspan(1550, 1700, color='lightblue', alpha=0.3, label='LIA cold period')

# --- 温度距平（相对于1500-1600基准期） ---
ax = axes[1]
ref_period = (1500, 1600)

# 重建气候距平
rec_ref_mean = float(temp_rec.sel(year=slice(*ref_period)).mean())
rec_anom = temp_rec - rec_ref_mean
rec_anom_smooth = rec_anom.rolling(year=window, center=True).mean()
ax.plot(temp_rec.year.values, rec_anom_smooth.values,
        color='black', linewidth=2.5, label='Rec_ref', zorder=10)

# GCM距平
for i, gcm in enumerate(GCM_LIST):
    if gcm in gcm_temps:
        gcm_ref = float(gcm_temps[gcm].sel(year=slice(*ref_period)).mean())
        gcm_anom = gcm_temps[gcm] - gcm_ref
        smooth = gcm_anom.rolling(year=window, center=True).mean()
        ax.plot(gcm_temps[gcm].year.values, smooth.values,
                color=colors[i % len(colors)], linewidth=1.2, alpha=0.7,
                label=gcm)

ax.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlim(1000, 2005)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.set_title('Temperature Anomaly relative to {}-{} ({}yr smoothed)'.format(
    ref_period[0], ref_period[1], window))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
ax.axvspan(1550, 1700, color='lightblue', alpha=0.3)

plt.tight_layout()
save_path = os.path.join(cfg.PATHS['working_dir'], 'gcm_vs_reconstruction_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print("图已保存: {}".format(save_path))

# %% Cell 7 - 诊断汇总

print("\n" + "=" * 70)
print("GCM vs 重建气候 振幅对比汇总")
print("=" * 70)

key = 'LIA_cold→1970-2000'
if key in dt_results:
    rec_dt = dt_results[key].get('Rec_ref', np.nan)
    gcm_dts = [dt_results[key][g] for g in GCM_LIST
               if g in dt_results[key] and not np.isnan(dt_results[key][g])]

    print("\nLIA冷期(1550-1700) → 现代(1970-2000) 温差:")
    print("  重建气候:  {:+.3f}K".format(rec_dt))
    if gcm_dts:
        print("  GCM均值:   {:+.3f}K".format(np.mean(gcm_dts)))
        print("  GCM范围:   {:+.3f} ~ {:+.3f}K".format(min(gcm_dts), max(gcm_dts)))
        deficit = np.mean(gcm_dts) - rec_dt
        print("  重建气候比GCM少: {:.3f}K".format(deficit))

    print("\n对冰川模拟的含义:")
    print("  闭环测试: 1.0K降温 → 冰川增长28-73%")
    print("  重建气候ΔT: {:.3f}K → 预期只能驱动约{:.0f}-{:.0f}%的退缩".format(
        abs(rec_dt), abs(rec_dt)/1.0*28, abs(rec_dt)/1.0*73))
    if gcm_dts:
        gcm_mean = np.mean(gcm_dts)
        print("  GCM均值ΔT: {:.3f}K → 预期可驱动约{:.0f}-{:.0f}%的退缩".format(
            abs(gcm_mean), abs(gcm_mean)/1.0*28, abs(gcm_mean)/1.0*73))
        print("\n  如果用GCM振幅替代重建气候，ratio可能从~1.5降到~{:.2f}".format(
            1.5 * abs(rec_dt) / abs(gcm_mean) if abs(gcm_mean) > 0 else np.nan))

print("\n分析完成.")

In [ ]:
"""
GCM验证实验：用Emean past1000驱动LIA→2000正演
================================================
框架A（实验3复刻）：LIA几何上用Emean校准melt_f使MB≈0 → spin-up → 正演
框架B（实验1复刻）：直接用现代CRU参数映射 → spin-up → 正演

对比两框架的ratio，并与ensemble_rec_ref的结果对比
如果ratio显著改善 → 问题在气候数据振幅，不在流程或LIA轮廓
"""

# %% Cell 1 - 配置 & 处理Emean到LIA冰川

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# GCM路径
base_path_temp = '/mnt/e/2Data/9DEM/oggm_related/6types_GCM/01Combine850-2000/tas'
base_path_precip = '/mnt/e/2Data/9DEM/oggm_related/6types_GCM/01Combine850-2000/pr'
fpath_temp = os.path.join(base_path_temp,
    'combined_tas_Amon_Emean_past1000_r1i1p1_085001-200512.nc')
fpath_precip = os.path.join(base_path_precip,
    'combined_pr_Amon_Emean_past1000_r1i1p1_085001-200512.nc')

GCM_SUFFIX = '_Emean'

# 实验参数
TEST_N = 20  # 设为 None 全量跑
SPINUP_YEARS = 500
HALFSIZE = 30
Y0_SPINUP = 1500  # spin-up气候参考期中心

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['task_timeout'] = 600

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'gcm_validation')
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 70)
print("GCM验证实验 (Emean past1000)")
print("=" * 70)

# 处理Emean到LIA冰川
print("\n[Step 0] 处理Emean到LIA冰川...")
if not os.path.exists(fpath_temp):
    print("  ERROR: 温度文件不存在: {}".format(fpath_temp))
elif not os.path.exists(fpath_precip):
    print("  ERROR: 降水文件不存在: {}".format(fpath_precip))
else:
    success, fail = 0, 0
    for gdir in gdirs_lia:
        try:
            tasks.process_cmip_data(
                gdir,
                fpath_temp=fpath_temp,
                fpath_precip=fpath_precip,
                filesuffix=GCM_SUFFIX,
            )
            success += 1
        except Exception as e:
            fail += 1
    print("  完成: 成功={}, 失败={}".format(success, fail))

# %% Cell 2 - 筛选有效冰川 & 准备参数

print("\n[Step 1] 筛选有效冰川...")

# 构建映射
rgi_area_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
        }
    except:
        pass

# 还要有Emean气候文件
valid_gdirs = []
for g in gdirs_lia:
    if g.rgi_id not in rgi_area_map or g.rgi_id not in original_params:
        continue
    try:
        fp = g.get_filepath('gcm_data', filesuffix=GCM_SUFFIX)
        if os.path.exists(fp):
            valid_gdirs.append(g)
    except:
        pass

print("有效冰川: {} / {}".format(len(valid_gdirs), len(gdirs_lia)))

if TEST_N is not None:
    valid_gdirs = valid_gdirs[:TEST_N]
    print(">>> 测试模式: 取前 {} 条 <<<".format(len(valid_gdirs)))

# %% Cell 3 - 框架B：CRU参数映射 + Emean气候 spin-up + 正演
# （先跑B，因为不需要校准，更快）

print("\n" + "=" * 70)
print("框架B：CRU参数映射 + Emean气候")
print("=" * 70)

# 写入CRU参数（和之前实验1一样）
print("[B-1] 写入CRU映射参数...")
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    params = original_params[rgi_id]
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = params['melt_f']
    p['temp_bias'] = params['temp_bias']
    p['prcp_fac'] = params['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)

# Spin-up
SUFFIX_B_SPINUP = '_gcm_B_spinup'
SUFFIX_B_FORWARD = '_gcm_B_forward'

print("[B-2] Spin-up ({}年, y0={})...".format(SPINUP_YEARS, Y0_SPINUP))
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate,
        valid_gdirs,
        nyears=SPINUP_YEARS,
        y0=Y0_SPINUP,
        halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=GCM_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX_B_SPINUP,
        store_model_geometry=True,
    )
except Exception as e:
    print("  [WARN] Spin-up异常: {}".format(e))
print("  耗时: {:.1f}min".format((time.time() - t0) / 60))

# 筛选成功
spinup_ok_B = []
for gdir in valid_gdirs:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_B_SPINUP)
        if os.path.exists(fp):
            spinup_ok_B.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok_B), len(valid_gdirs)))

# 正演
print("[B-3] 正演到2000...")
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data,
        spinup_ok_B,
        min_ys=Y0_SPINUP,
        ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=GCM_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_B_SPINUP,
        output_filesuffix=SUFFIX_B_FORWARD,
    )
except Exception as e:
    print("  [WARN] 正演异常: {}".format(e))
print("  耗时: {:.1f}min".format((time.time() - t0) / 60))

# 提取结果B
results_B = []
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_B_FORWARD)
        if not os.path.exists(fp):
            continue
        ds = xr.open_dataset(fp)
        lia_area = float(ds['area_m2'].values[0]) / 1e6
        end_area = float(ds['area_m2'].values[-1]) / 1e6
        rgi_area = rgi_area_map[rgi_id]
        results_B.append({
            'rgi_id': rgi_id,
            'framework': 'B_CRU_params',
            'lia_area_km2': lia_area,
            'end_area_km2': end_area,
            'rgi_area_km2': rgi_area,
            'ratio': end_area / rgi_area,
        })
        ds.close()
    except:
        pass

df_B = pd.DataFrame(results_B)
print("\n框架B结果: {}条".format(len(df_B)))
if len(df_B) > 0:
    print("  ratio均值:   {:.4f}".format(df_B['ratio'].mean()))
    print("  ratio中位数: {:.4f}".format(df_B['ratio'].median()))

# %% Cell 4 - 框架A：LIA几何上用Emean校准melt_f使MB≈0 → spin-up → 正演

print("\n" + "=" * 70)
print("框架A：MB≈0校准melt_f + Emean气候")
print("=" * 70)

# 校准melt_f使得LIA几何上MB≈0
print("[A-1] 在LIA几何上用Emean校准melt_f...")

SCALE_RANGE = np.arange(0.1, 1.51, 0.05)  # melt_f缩放因子搜索范围
Y0_CALIB = Y0_SPINUP

calibrated_params = {}
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    params = original_params[rgi_id]
    original_melt_f = params['melt_f']

    try:
        fls = gdir.read_pickle('model_flowlines')
    except:
        continue

    # 搜索使MB≈0的melt_f缩放因子
    best_scale = None
    best_mb_abs = 999999

    for scale in SCALE_RANGE:
        new_melt_f = original_melt_f * scale
        try:
            mb = MyMBModel(
                gdir,
                filename='gcm_data',
                input_filesuffix=GCM_SUFFIX,
                melt_f=new_melt_f,
                temp_bias=params['temp_bias'],
                prcp_fac=params['prcp_fac'],
            )
            mb_vals = [mb.get_specific_mb(fls=fls, year=yr)
                       for yr in range(Y0_CALIB - HALFSIZE, Y0_CALIB + HALFSIZE + 1)]
            avg_mb = np.mean(mb_vals)

            if abs(avg_mb) < best_mb_abs:
                best_mb_abs = abs(avg_mb)
                best_scale = scale
                best_mb = avg_mb
        except:
            pass

    if best_scale is not None:
        calibrated_params[rgi_id] = {
            'melt_f': original_melt_f * best_scale,
            'scale': best_scale,
            'mb_residual': best_mb,
            'temp_bias': params['temp_bias'],
            'prcp_fac': params['prcp_fac'],
        }

print("  校准成功: {}/{}".format(len(calibrated_params), len(valid_gdirs)))

# 统计缩放因子
scales = [v['scale'] for v in calibrated_params.values()]
if scales:
    print("  melt_f缩放因子: 均值={:.3f}, 中位数={:.3f}, std={:.3f}".format(
        np.mean(scales), np.median(scales), np.std(scales)))
    print("  MB残差(绝对值): 均值={:.1f} kg/m²/yr".format(
        np.mean([abs(v['mb_residual']) for v in calibrated_params.values()])))

# 写入校准参数
print("[A-2] 写入校准参数...")
gdirs_A = []
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    if rgi_id not in calibrated_params:
        continue
    cp = calibrated_params[rgi_id]
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = cp['melt_f']
    p['temp_bias'] = cp['temp_bias']
    p['prcp_fac'] = cp['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)
    gdirs_A.append(gdir)

# Spin-up
SUFFIX_A_SPINUP = '_gcm_A_spinup'
SUFFIX_A_FORWARD = '_gcm_A_forward'

print("[A-3] Spin-up ({}年, y0={})...".format(SPINUP_YEARS, Y0_SPINUP))
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate,
        gdirs_A,
        nyears=SPINUP_YEARS,
        y0=Y0_SPINUP,
        halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=GCM_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX_A_SPINUP,
        store_model_geometry=True,
    )
except Exception as e:
    print("  [WARN] Spin-up异常: {}".format(e))
print("  耗时: {:.1f}min".format((time.time() - t0) / 60))

spinup_ok_A = []
for gdir in gdirs_A:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_A_SPINUP)
        if os.path.exists(fp):
            spinup_ok_A.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok_A), len(gdirs_A)))

# 正演
print("[A-4] 正演到2000...")
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data,
        spinup_ok_A,
        min_ys=Y0_SPINUP,
        ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=GCM_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_A_SPINUP,
        output_filesuffix=SUFFIX_A_FORWARD,
    )
except Exception as e:
    print("  [WARN] 正演异常: {}".format(e))
print("  耗时: {:.1f}min".format((time.time() - t0) / 60))

# 提取结果A
results_A = []
for gdir in gdirs_A:
    rgi_id = gdir.rgi_id
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_A_FORWARD)
        if not os.path.exists(fp):
            continue
        ds = xr.open_dataset(fp)
        lia_area = float(ds['area_m2'].values[0]) / 1e6
        end_area = float(ds['area_m2'].values[-1]) / 1e6
        rgi_area = rgi_area_map[rgi_id]
        results_A.append({
            'rgi_id': rgi_id,
            'framework': 'A_MB0_calib',
            'lia_area_km2': lia_area,
            'end_area_km2': end_area,
            'rgi_area_km2': rgi_area,
            'ratio': end_area / rgi_area,
            'melt_f_scale': calibrated_params[rgi_id]['scale'],
        })
        ds.close()
    except:
        pass

df_A = pd.DataFrame(results_A)
print("\n框架A结果: {}条".format(len(df_A)))
if len(df_A) > 0:
    print("  ratio均值:   {:.4f}".format(df_A['ratio'].mean()))
    print("  ratio中位数: {:.4f}".format(df_A['ratio'].median()))

# %% Cell 5 - 恢复原始参数

print("\n[恢复] 写回CRU原始参数...")
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    if rgi_id not in original_params:
        continue
    params = original_params[rgi_id]
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = params['melt_f']
    p['temp_bias'] = params['temp_bias']
    p['prcp_fac'] = params['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)
print("  已恢复")

# %% Cell 6 - 对比汇总

print("\n" + "=" * 90)
print("三套气候数据 × 两个框架 ratio对比")
print("=" * 90)

# 汇总表
print("\n{:<35} {:>10} {:>10} {:>8}".format(
    "实验", "ratio均值", "ratio中位", "N"))
print("-" * 68)

# 之前的ensemble_rec_ref结果（手动填入或从保存的结果读取）
print("{:<35} {:>10} {:>10} {:>8}".format(
    "Rec_ref + 框架A(MB≈0)", "~1.508", "~1.358", "20"))
print("{:<35} {:>10} {:>10} {:>8}".format(
    "Rec_ref + 框架B(CRU映射)", "~1.60", "~N/A", "20"))

if len(df_A) > 0:
    print("{:<35} {:>10.4f} {:>10.4f} {:>8}".format(
        "Emean + 框架A(MB≈0)",
        df_A['ratio'].mean(), df_A['ratio'].median(), len(df_A)))

if len(df_B) > 0:
    print("{:<35} {:>10.4f} {:>10.4f} {:>8}".format(
        "Emean + 框架B(CRU映射)",
        df_B['ratio'].mean(), df_B['ratio'].median(), len(df_B)))

# 改善量
print("\n--- 改善分析 ---")
rec_ratio_A = 1.508  # 之前实验3的结果
rec_ratio_B = 1.60   # 之前实验1的结果

if len(df_A) > 0:
    improve_A = rec_ratio_A - df_A['ratio'].mean()
    print("框架A: Rec_ref ratio={:.3f} → Emean ratio={:.4f}, 改善={:+.3f}".format(
        rec_ratio_A, df_A['ratio'].mean(), -improve_A))

if len(df_B) > 0:
    improve_B = rec_ratio_B - df_B['ratio'].mean()
    print("框架B: Rec_ref ratio={:.2f} → Emean ratio={:.4f}, 改善={:+.3f}".format(
        rec_ratio_B, df_B['ratio'].mean(), -improve_B))

# 逐冰川对比
if len(df_A) > 0:
    print("\n--- 框架A 逐冰川结果 ---")
    print("{:<15} {:>9} {:>9} {:>9} {:>8} {:>8}".format(
        'RGI_ID', 'LIA_km2', 'End_km2', 'RGI_km2', 'Ratio', 'Scale'))
    print("-" * 62)
    for _, r in df_A.sort_values('ratio').iterrows():
        print("{:<15} {:>9.2f} {:>9.2f} {:>9.2f} {:>8.4f} {:>8.3f}".format(
            r['rgi_id'].split('.')[-1], r['lia_area_km2'],
            r['end_area_km2'], r['rgi_area_km2'], r['ratio'],
            r.get('melt_f_scale', np.nan)))

if len(df_B) > 0:
    print("\n--- 框架B 逐冰川结果 ---")
    print("{:<15} {:>9} {:>9} {:>9} {:>8}".format(
        'RGI_ID', 'LIA_km2', 'End_km2', 'RGI_km2', 'Ratio'))
    print("-" * 52)
    for _, r in df_B.sort_values('ratio').iterrows():
        print("{:<15} {:>9.2f} {:>9.2f} {:>9.2f} {:>8.4f}".format(
            r['rgi_id'].split('.')[-1], r['lia_area_km2'],
            r['end_area_km2'], r['rgi_area_km2'], r['ratio']))

# 保存
all_results = pd.concat([df_A, df_B], ignore_index=True)
all_results.to_csv(os.path.join(SAVE_DIR, 'gcm_validation_results.csv'), index=False)
print("\n结果已保存: {}".format(os.path.join(SAVE_DIR, 'gcm_validation_results.csv')))

print("\n" + "=" * 70)
print("GCM验证实验完成!")
print("=" * 70)

In [ ]:
"""
GCM冰川格点温度距平分析
==========================
计算各GCM在LIA期间相对于1961-1990的温度距平
同时计算重建气候的距平作为对比
使用已降尺度到冰川点的数据
"""

# %% Cell 1 - 加载数据

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import os

GCM_LIST = ['BCC-CSM1-1', 'CCSM4', 'MPI-ESM-P', 'GISS-E2-R', 'IPSL-CM5A-LR', 'CESM']
REF_PERIOD = (1961, 1990)  # 基准期

# 多条冰川取平均，更稳健
use_gdirs_rgi = gdirs_rgi[:5]  # GCM数据在rgi冰川上
gdir_lia = gdirs_lia[0]        # 重建气候在lia冰川上

print("=" * 70)
print("GCM冰川格点温度距平 (相对于{}-{})".format(*REF_PERIOD))
print("=" * 70)

# 读取各GCM年均温（多冰川平均）
gcm_annual = {}
for gcm in GCM_LIST:
    glacier_temps = []
    for gdir in use_gdirs_rgi:
        try:
            fp = gdir.get_filepath('gcm_data', filesuffix='_{}'.format(gcm))
            ds = xr.open_dataset(fp)
            t = ds['temp'].groupby('time.year').mean()
            glacier_temps.append(t)
            ds.close()
        except:
            pass
    if glacier_temps:
        # 对齐年份后取平均
        min_yr = max(int(t.year.values[0]) for t in glacier_temps)
        max_yr = min(int(t.year.values[-1]) for t in glacier_temps)
        aligned = [t.sel(year=slice(min_yr, max_yr)) for t in glacier_temps]
        gcm_annual[gcm] = sum(aligned) / len(aligned)
        print("  {} : {}-{}, {}条冰川平均".format(
            gcm, min_yr, max_yr, len(glacier_temps)))

# 读取重建气候
fp_rec = gdir_lia.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
ds_rec = xr.open_dataset(fp_rec)
rec_annual = ds_rec['temp'].groupby('time.year').mean()
ds_rec.close()
print("  Rec_ref : {}-{}".format(
    int(rec_annual.year.values[0]), int(rec_annual.year.values[-1])))

# 读取CRU
try:
    fp_cru = use_gdirs_rgi[0].get_filepath('climate_historical')
    ds_cru = xr.open_dataset(fp_cru)
    cru_annual = ds_cru['temp'].groupby('time.year').mean()
    ds_cru.close()
    has_cru = True
    print("  CRU : {}-{}".format(
        int(cru_annual.year.values[0]), int(cru_annual.year.values[-1])))
except:
    has_cru = False

# %% Cell 2 - 计算距平

print("\n[计算距平]...")

# 各数据源的1961-1990基准值
ref_temps = {}

# GCM基准
gcm_anomalies = {}
for gcm in GCM_LIST:
    if gcm not in gcm_annual:
        continue
    ref = float(gcm_annual[gcm].sel(year=slice(*REF_PERIOD)).mean())
    ref_temps[gcm] = ref
    gcm_anomalies[gcm] = gcm_annual[gcm] - ref
    print("  {} 基准温度 ({}-{}): {:.3f}°C".format(gcm, *REF_PERIOD, ref))

# 重建气候基准
ref_rec = float(rec_annual.sel(year=slice(*REF_PERIOD)).mean())
ref_temps['Rec_ref'] = ref_rec
rec_anomaly = rec_annual - ref_rec
print("  Rec_ref 基准温度: {:.3f}°C".format(ref_rec))

# CRU基准
if has_cru:
    ref_cru = float(cru_annual.sel(year=slice(*REF_PERIOD)).mean())
    ref_temps['CRU'] = ref_cru
    cru_anomaly = cru_annual - ref_cru
    print("  CRU 基准温度: {:.3f}°C".format(ref_cru))

# %% Cell 3 - 各时段距平统计

print("\n" + "=" * 90)
print("各时段温度距平 (相对于{}-{})".format(*REF_PERIOD))
print("=" * 90)

periods = {
    '1000-1200': (1000, 1200),
    '1200-1400': (1200, 1400),
    '1400-1550': (1400, 1550),
    'LIA_cold(1550-1700)': (1550, 1700),
    'LIA_peak(1600-1650)': (1600, 1650),
    '1700-1800': (1700, 1800),
    '1800-1900': (1800, 1900),
    '1900-1950': (1900, 1950),
    '1961-1990(ref=0)': (1961, 1990),
    '1970-2000': (1970, 2000),
}

# 表头
header = "{:<22}".format("时段")
header += "{:>10}".format("Rec_ref")
for gcm in GCM_LIST:
    if gcm in gcm_anomalies:
        header += "{:>10}".format(gcm[:8])
if has_cru:
    header += "{:>10}".format("CRU")
header += "{:>10}".format("GCM均值")
print(header)
print("-" * len(header))

period_anomalies = {}
for pname, (y1, y2) in periods.items():
    row = "{:<22}".format(pname)

    # 重建
    sub = rec_anomaly.sel(year=slice(y1, y2))
    a_rec = float(sub.mean()) if len(sub) > 0 else np.nan
    row += "{:>+10.3f}".format(a_rec)
    period_anomalies.setdefault(pname, {})['Rec_ref'] = a_rec

    # GCM
    gcm_vals = []
    for gcm in GCM_LIST:
        if gcm in gcm_anomalies:
            sub = gcm_anomalies[gcm].sel(year=slice(y1, y2))
            a = float(sub.mean()) if len(sub) > 0 else np.nan
            row += "{:>+10.3f}".format(a)
            period_anomalies[pname][gcm] = a
            if not np.isnan(a):
                gcm_vals.append(a)

    # CRU
    if has_cru:
        sub = cru_anomaly.sel(year=slice(y1, y2))
        a = float(sub.mean()) if len(sub) > 0 else np.nan
        row += "{:>+10.3f}".format(a) if not np.isnan(a) else "{:>10}".format("N/A")

    # GCM均值
    if gcm_vals:
        row += "{:>+10.3f}".format(np.mean(gcm_vals))

    print(row)

# %% Cell 4 - LIA距平的详细对比

print("\n" + "=" * 70)
print("LIA冷期(1550-1700) 距平详细对比")
print("=" * 70)

pname = 'LIA_cold(1550-1700)'
if pname in period_anomalies:
    pa = period_anomalies[pname]

    print("\n{:<20} {:>12} {:>12}".format("数据源", "距平(°C)", "说明"))
    print("-" * 48)

    print("{:<20} {:>+12.3f} {:>12}".format("Rec_ref", pa['Rec_ref'], "重建气候"))

    gcm_vals = []
    for gcm in GCM_LIST:
        if gcm in pa and not np.isnan(pa[gcm]):
            gcm_vals.append(pa[gcm])
            print("{:<20} {:>+12.3f}".format(gcm, pa[gcm]))

    if gcm_vals:
        print("-" * 48)
        print("{:<20} {:>+12.3f}".format("GCM均值", np.mean(gcm_vals)))
        print("{:<20} {:>+12.3f}".format("GCM中位数", np.median(gcm_vals)))
        print("{:<20} {:>12}".format("GCM范围",
            "{:+.3f} ~ {:+.3f}".format(min(gcm_vals), max(gcm_vals))))

        # 排除CESM
        gcm_no_cesm = [pa[g] for g in GCM_LIST
                       if g in pa and g != 'CESM' and not np.isnan(pa[g])]
        if gcm_no_cesm:
            print("{:<20} {:>+12.3f} {:>12}".format(
                "GCM均值(无CESM)", np.mean(gcm_no_cesm), "排除异常值"))

        print("\n重建 vs GCM:")
        print("  重建距平:     {:+.3f}°C".format(pa['Rec_ref']))
        print("  GCM均值距平:  {:+.3f}°C".format(np.mean(gcm_vals)))
        print("  重建/GCM比值: {:.2f}".format(
            pa['Rec_ref'] / np.mean(gcm_vals) if np.mean(gcm_vals) != 0 else np.nan))
        if gcm_no_cesm:
            print("  重建/GCM比值(无CESM): {:.2f}".format(
                pa['Rec_ref'] / np.mean(gcm_no_cesm) if np.mean(gcm_no_cesm) != 0 else np.nan))

# %% Cell 5 - LIA peak距平

print("\n" + "=" * 70)
print("LIA最冷期(1600-1650) 距平详细对比")
print("=" * 70)

pname = 'LIA_peak(1600-1650)'
if pname in period_anomalies:
    pa = period_anomalies[pname]

    print("\n{:<20} {:>12}".format("数据源", "距平(°C)"))
    print("-" * 35)
    print("{:<20} {:>+12.3f}".format("Rec_ref", pa['Rec_ref']))

    gcm_vals = []
    for gcm in GCM_LIST:
        if gcm in pa and not np.isnan(pa[gcm]):
            gcm_vals.append(pa[gcm])
            print("{:<20} {:>+12.3f}".format(gcm, pa[gcm]))

    if gcm_vals:
        print("-" * 35)
        print("{:<20} {:>+12.3f}".format("GCM均值", np.mean(gcm_vals)))

# %% Cell 6 - 可视化

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
window = 30

# --- 距平时间序列 ---
ax = axes[0]

rec_smooth = rec_anomaly.rolling(year=window, center=True).mean()
ax.plot(rec_annual.year.values, rec_smooth.values,
        color='black', linewidth=2.5, label='Rec_ref', zorder=10)

colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628']
for i, gcm in enumerate(GCM_LIST):
    if gcm in gcm_anomalies:
        smooth = gcm_anomalies[gcm].rolling(year=window, center=True).mean()
        ax.plot(gcm_anomalies[gcm].year.values, smooth.values,
                color=colors[i % len(colors)], linewidth=1.2, alpha=0.7,
                label=gcm)

if has_cru:
    cru_smooth = cru_anomaly.rolling(year=window, center=True).mean()
    ax.plot(cru_annual.year.values, cru_smooth.values,
            color='green', linewidth=2, linestyle='--', label='CRU', zorder=9)

ax.axhline(0, color='gray', linewidth=0.5)
ax.axvspan(1550, 1700, color='lightblue', alpha=0.3, label='LIA cold')
ax.set_xlim(1000, 2005)
ax.set_ylabel('Temperature Anomaly (°C)')
ax.set_title('Temperature Anomaly relative to {}-{} ({}yr smoothed)'.format(
    *REF_PERIOD, window))
ax.legend(loc='lower right', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# --- LIA冷期距平柱状图 ---
ax = axes[1]

sources = ['Rec_ref']
vals = [period_anomalies['LIA_cold(1550-1700)']['Rec_ref']]
bar_colors = ['black']

for i, gcm in enumerate(GCM_LIST):
    if gcm in period_anomalies['LIA_cold(1550-1700)']:
        sources.append(gcm)
        vals.append(period_anomalies['LIA_cold(1550-1700)'][gcm])
        bar_colors.append(colors[i % len(colors)])

x = np.arange(len(sources))
bars = ax.bar(x, vals, color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5)

# GCM均值线
gcm_only = [v for s, v in zip(sources, vals) if s != 'Rec_ref']
if gcm_only:
    ax.axhline(np.mean(gcm_only), color='red', linestyle='--', linewidth=1.5,
               label='GCM mean = {:+.3f}°C'.format(np.mean(gcm_only)))

ax.set_xticks(x)
ax.set_xticklabels(sources, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Temperature Anomaly (°C)')
ax.set_title('LIA Cold Period (1550-1700) Anomaly relative to {}-{}'.format(*REF_PERIOD))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='gray', linewidth=0.5)

# 数值标注
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val - 0.05,
            '{:+.2f}'.format(val), ha='center', va='top', fontsize=9, fontweight='bold')

plt.tight_layout()
save_path = os.path.join(cfg.PATHS['working_dir'], 'gcm_lia_anomaly_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print("图已保存: {}".format(save_path))

# %% Cell 7 - 汇总

print("\n" + "=" * 70)
print("距平分析汇总")
print("=" * 70)

lia_cold = period_anomalies.get('LIA_cold(1550-1700)', {})
rec_a = lia_cold.get('Rec_ref', np.nan)
gcm_vals = [lia_cold[g] for g in GCM_LIST if g in lia_cold and not np.isnan(lia_cold[g])]
gcm_no_cesm = [lia_cold[g] for g in GCM_LIST
               if g in lia_cold and g != 'CESM' and not np.isnan(lia_cold[g])]

print("\nLIA冷期相对于1961-1990的温度距平:")
print("  重建气候:       {:+.3f}°C".format(rec_a))
if gcm_vals:
    print("  GCM均值(6个):   {:+.3f}°C".format(np.mean(gcm_vals)))
if gcm_no_cesm:
    print("  GCM均值(无CESM): {:+.3f}°C".format(np.mean(gcm_no_cesm)))

print("\n含义:")
print("  重建气候认为LIA比1961-1990冷 {:.2f}°C".format(abs(rec_a)))
if gcm_vals:
    print("  GCM认为LIA比1961-1990冷 {:.2f}°C".format(abs(np.mean(gcm_vals))))
    print("  重建气候的LIA变冷幅度是GCM的 {:.0f}%".format(
        abs(rec_a) / abs(np.mean(gcm_vals)) * 100))

print("\n对冰川模拟的直接影响:")
print("  框架B验证: Emean(ΔT≈1.15K) → ratio≈1.06")
print("  框架B验证: Rec_ref(ΔT≈0.67K) → ratio≈1.60")
print("  结论: 气候振幅是ratio偏大的主因")

print("\n分析完成.")

In [ ]:
"""
Offset扫描实验：spinup + 正演，全程保持同一offset
==================================================
对20条冰川，offset从0到-2扫描，20条并行
检验点1: spinup结束面积 vs LIA轮廓
检验点2: 正演结束面积 vs RGI观测
"""

# %% Cell 1 - 配置

import json, os, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---------- 实验参数 ----------
TEST_N = 20
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_END = 2000
CLIMATE_SUFFIX = '_ensemble_rec_ref'
OFFSETS = [0, -0.5, -1.0, -1.5, -2.0]

cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 12
cfg.PARAMS['continue_on_error'] = True

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'offset_sweep_test')
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 70)
print("Offset扫描: spinup + 正演, 全程保持同一offset")
print("Y0={}, spinup={}yr, 正演→{}, offsets={}".format(
    Y0, SPINUP_YEARS, FORWARD_END, OFFSETS))
print("=" * 70)

# %% Cell 2 - 构建面积映射

rgi_area_map = {}
rgi_id_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
        rgi_id_map[gdir.rgi_id] = main_rgi_id
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f':    params['melt_f'],
            'prcp_fac':  params['prcp_fac'],
        }
    except:
        pass

print("rgi_area_map : {}".format(len(rgi_area_map)))
print("original_params: {}".format(len(original_params)))

# %% Cell 3 - 筛选冰川

test_gdirs = [g for g in gdirs_lia
              if g.rgi_id in original_params
              and g.rgi_id in rgi_area_map
              and os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))
print("\n{:<8} {:>10} {:>10} {:>10}".format("UID", "LIA_km2", "RGI_km2", "LIA/RGI"))
print("-" * 42)
for gdir in test_gdirs:
    rid = gdir.rgi_id
    uid = rid.split('.')[-1]
    lia_a = gdir.rgi_area_m2 / 1e6
    rgi_a = rgi_area_map[rid]
    print("{:<8} {:>10.2f} {:>10.2f} {:>10.2f}".format(uid, lia_a, rgi_a, lia_a / rgi_a))

# %% Cell 4 - 按offset分组，20条冰川并行

print("\n[实验主体] 按offset分组并行...")
t_total = time.time()

for offset in OFFSETS:
    print("\n=== offset = {:+.1f} ===".format(offset))
    t0 = time.time()

    suffix_sp = '_sweep_sp_off{}'.format(str(offset).replace('.', 'p').replace('-', 'n'))
    suffix_fw = '_sweep_fw_off{}'.format(str(offset).replace('.', 'p').replace('-', 'n'))

    # 批量写入 temp_bias + offset
    for gdir in test_gdirs:
        orig = original_params[gdir.rgi_id]
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig['temp_bias'] + offset
        p['melt_f'] = orig['melt_f']
        p['prcp_fac'] = orig['prcp_fac']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

    # Spinup - 20条并行
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=suffix_sp,
        store_model_geometry=True,
    )

    # 正演 - 20条并行（保持同一offset）
    execute_entity_task(
        tasks.run_from_climate_data, test_gdirs,
        min_ys=Y0, ye=FORWARD_END,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=suffix_sp,
        output_filesuffix=suffix_fw,
    )

    # 批量恢复原始参数
    for gdir in test_gdirs:
        orig = original_params[gdir.rgi_id]
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig['temp_bias']
        p['melt_f'] = orig['melt_f']
        p['prcp_fac'] = orig['prcp_fac']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

    print("  offset={:+.1f} 完成, 耗时: {:.1f}min".format(offset, (time.time() - t0) / 60))

print("\n实验总耗时: {:.1f}min".format((time.time() - t_total) / 60))

# %% Cell 5 - 提取全部结果

print("\n[提取结果]")
all_results = {}

for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    lia_area = gdir.rgi_area_m2 / 1e6
    rgi_area = rgi_area_map[rgi_id]

    glacier_results = {}
    for offset in OFFSETS:
        suffix_sp = '_sweep_sp_off{}'.format(str(offset).replace('.', 'p').replace('-', 'n'))
        suffix_fw = '_sweep_fw_off{}'.format(str(offset).replace('.', 'p').replace('-', 'n'))

        try:
            ds_sp = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp))
            sp_start = float(ds_sp['area_m2'].values[0]) / 1e6
            sp_end = float(ds_sp['area_m2'].values[-1]) / 1e6
            ds_sp.close()

            ds_fw = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw))
            fw_end = float(ds_fw['area_m2'].values[-1]) / 1e6
            ds_fw.close()

            glacier_results[offset] = {
                'sp_start': sp_start,
                'sp_end': sp_end,
                'fw_end': fw_end,
                'sp_lia_ratio': sp_end / lia_area,
                'fw_rgi_ratio': fw_end / rgi_area,
                'status': 'ok',
            }
        except:
            glacier_results[offset] = {'status': 'failed'}

    all_results[rgi_id] = {
        'uid': uid,
        'lia_area': lia_area,
        'rgi_area': rgi_area,
        'offsets': glacier_results,
    }

print("提取完成: {} 条冰川".format(len(all_results)))

# %% Cell 6 - 汇总表

print("\n" + "=" * 130)
print("汇总: spinup结束/LIA  和  正演结束/RGI")
print("=" * 130)

# 表头
header = "{:<8} {:>8} {:>8}".format("UID", "LIA", "RGI")
for off in OFFSETS:
    header += " | off={:+.1f} sp/LIA fw/RGI".format(off)
print(header)
print("-" * 130)

for rgi_id, r in all_results.items():
    line = "{:<8} {:>8.2f} {:>8.2f}".format(r['uid'], r['lia_area'], r['rgi_area'])
    for off in OFFSETS:
        gr = r['offsets'].get(off, {})
        if gr.get('status') == 'ok':
            line += " |       {:>6.3f} {:>6.3f}".format(gr['sp_lia_ratio'], gr['fw_rgi_ratio'])
        else:
            line += " |        FAIL   FAIL"
    print(line)

# 各offset统计
print("\n--- 各offset统计 ---")
print("{:<8} {:>12} {:>12} {:>12} {:>12} {:>8}".format(
    "offset", "sp/LIA_med", "sp/LIA_std", "fw/RGI_med", "fw/RGI_std", "N_ok"))
print("-" * 70)

for off in OFFSETS:
    sp_ratios = []
    fw_ratios = []
    for r in all_results.values():
        gr = r['offsets'].get(off, {})
        if gr.get('status') == 'ok':
            sp_ratios.append(gr['sp_lia_ratio'])
            fw_ratios.append(gr['fw_rgi_ratio'])
    if sp_ratios:
        print("{:<8} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f} {:>8}".format(
            off, np.median(sp_ratios), np.std(sp_ratios),
            np.median(fw_ratios), np.std(fw_ratios), len(sp_ratios)))

# %% Cell 7 - 可视化

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 收集统计
off_arr = []
sp_med = []
fw_med = []
sp_q25, sp_q75 = [], []
fw_q25, fw_q75 = [], []

for off in OFFSETS:
    sp_r = [r['offsets'][off]['sp_lia_ratio']
            for r in all_results.values()
            if r['offsets'].get(off, {}).get('status') == 'ok']
    fw_r = [r['offsets'][off]['fw_rgi_ratio']
            for r in all_results.values()
            if r['offsets'].get(off, {}).get('status') == 'ok']
    if sp_r and fw_r:
        off_arr.append(off)
        sp_med.append(np.median(sp_r))
        fw_med.append(np.median(fw_r))
        sp_q25.append(np.percentile(sp_r, 25))
        sp_q75.append(np.percentile(sp_r, 75))
        fw_q25.append(np.percentile(fw_r, 25))
        fw_q75.append(np.percentile(fw_r, 75))

off_arr = np.array(off_arr)

# 左图: sp/LIA vs offset
axes[0].plot(off_arr, sp_med, 'o-', color='blue', linewidth=2, markersize=8, label='median')
axes[0].fill_between(off_arr, sp_q25, sp_q75, alpha=0.2, color='blue', label='25-75%')
axes[0].axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='target=1.0')
axes[0].set_xlabel('Temperature offset (K)')
axes[0].set_ylabel('Spinup end area / LIA area')
axes[0].set_title('Spinup稳态 vs LIA轮廓')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图: fw/RGI vs offset
axes[1].plot(off_arr, fw_med, 's-', color='green', linewidth=2, markersize=8, label='median')
axes[1].fill_between(off_arr, fw_q25, fw_q75, alpha=0.2, color='green', label='25-75%')
axes[1].axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='target=1.0')
axes[1].set_xlabel('Temperature offset (K)')
axes[1].set_ylabel('Forward end area / RGI area')
axes[1].set_title('正演2000年 vs RGI观测')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Offset sweep: Y0={}, spinup={}yr, rec_ref climate'.format(Y0, SPINUP_YEARS),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 图2: 逐冰川scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = plt.cm.viridis(np.linspace(0, 1, len(OFFSETS)))
for i, off in enumerate(OFFSETS):
    sp_r = []
    fw_r = []
    for r in all_results.values():
        gr = r['offsets'].get(off, {})
        if gr.get('status') == 'ok':
            sp_r.append(gr['sp_lia_ratio'])
            fw_r.append(gr['fw_rgi_ratio'])
    axes[0].scatter([off] * len(sp_r), sp_r, color=colors[i], alpha=0.6, s=40)
    axes[1].scatter([off] * len(fw_r), fw_r, color=colors[i], alpha=0.6, s=40)

axes[0].axhline(1.0, color='red', linestyle='--')
axes[0].set_xlabel('Offset (K)')
axes[0].set_ylabel('sp_end / LIA')
axes[0].set_title('Spinup: 每条冰川')
axes[0].grid(True, alpha=0.3)

axes[1].axhline(1.0, color='red', linestyle='--')
axes[1].set_xlabel('Offset (K)')
axes[1].set_ylabel('fw_end / RGI')
axes[1].set_title('Forward: 每条冰川')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# %% Cell 8 - 保存

pkl_path = os.path.join(SAVE_DIR, 'offset_sweep_results.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'all_results': all_results,
        'offsets': OFFSETS,
        'config': {
            'Y0': Y0, 'HALFSIZE': HALFSIZE, 'SPINUP_YEARS': SPINUP_YEARS,
            'FORWARD_END': FORWARD_END, 'CLIMATE_SUFFIX': CLIMATE_SUFFIX,
            'TEST_N': TEST_N,
        },
    }, f)
print("Pickle: {}".format(pkl_path))

csv_rows = []
for rgi_id, r in all_results.items():
    for off in OFFSETS:
        gr = r['offsets'].get(off, {})
        if gr.get('status') == 'ok':
            csv_rows.append({
                'uid': r['uid'], 'rgi_id': rgi_id,
                'lia_area': r['lia_area'], 'rgi_area': r['rgi_area'],
                'offset': off,
                'sp_start': gr['sp_start'], 'sp_end': gr['sp_end'],
                'fw_end': gr['fw_end'],
                'sp_lia_ratio': gr['sp_lia_ratio'],
                'fw_rgi_ratio': gr['fw_rgi_ratio'],
            })

csv_path = os.path.join(SAVE_DIR, 'offset_sweep_results.csv')
pd.DataFrame(csv_rows).to_csv(csv_path, index=False)
print("CSV: {}".format(csv_path))

# 恢复所有冰川原始参数（最终双保险）
print("\n恢复所有原始参数...")
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    if rgi_id not in original_params:
        continue
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    orig = original_params[rgi_id]
    p['temp_bias'] = orig['temp_bias']
    p['melt_f'] = orig['melt_f']
    p['prcp_fac'] = orig['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)
print("恢复完成")

print("\n" + "=" * 70)
print("全部完成！")
print("=" * 70)

In [ ]:
"""
GCM Emean驱动: spinup + 正演, 20条冰川
防卡死改进版：
  1. task_timeout=600（非0）
  2. mp_processes 降到10
  3. execute_entity_task 外层 try-except + 单进程回退
  4. 文件存在性筛选，失败跳过
  5. xr.open_dataset 全部用 try-finally 确保关闭
"""

import json, os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ============================================================
# 配置
# ============================================================
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_END = 2000

# 防卡死关键配置
cfg.PARAMS['task_timeout'] = 600        # 单任务超时600秒（非0！）
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15         # 从15降到10，减少内存压力
cfg.PARAMS['continue_on_error'] = True

# ============================================================
# 检查GCM气候文件
# ============================================================
print("检查GCM气候文件...")
sample_dir = test_gdirs[0].dir
gcm_files = [f for f in sorted(os.listdir(sample_dir)) if f.startswith('gcm_data')]
print("可用气候文件:")
for f in gcm_files:
    print("  {}".format(f))

gcm_suffix = '_Emean'
n_exist = sum(1 for g in test_gdirs
              if os.path.exists(os.path.join(g.dir, 'gcm_data{}.nc'.format(gcm_suffix))))
print("找到 {}/{} 条冰川有气候文件 (suffix='{}')".format(n_exist, len(test_gdirs), gcm_suffix))

if n_exist == 0:
    raise FileNotFoundError("没有找到气候文件，请检查suffix")

# ============================================================
# 恢复原始参数
# ============================================================
for gdir in test_gdirs:
    orig = original_params[gdir.rgi_id]
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = orig['melt_f']
    p['temp_bias'] = orig['temp_bias']
    p['prcp_fac'] = orig['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)

print("参数已恢复为原始值")

# ============================================================
# 主实验
# ============================================================
print("=" * 70)
print("GCM驱动: spinup {}yr (Y0={}) + 正演→{}".format(SPINUP_YEARS, Y0, FORWARD_END))
print("suffix = '{}', timeout={}s, processes={}".format(
    gcm_suffix, cfg.PARAMS['task_timeout'], cfg.PARAMS['mp_processes']))
print("=" * 70)

suffix_sp = '_gcm_emean_sp'
suffix_fw = '_gcm_emean_fw'

# ---------- Spinup ----------
t0 = time.time()
print("\n[Spinup] run_constant_climate ({} 条冰川)...".format(len(test_gdirs)))

spinup_success = False
try:
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=gcm_suffix,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=suffix_sp,
        store_model_geometry=True,
    )
    spinup_success = True
    print("  多进程Spinup完成: {:.1f}min".format((time.time() - t0) / 60))
except Exception as e:
    print("  多进程Spinup出错: {}".format(str(e)[:100]))

# 多进程失败 → 回退单进程逐条跑
if not spinup_success:
    print("  [回退] 关闭多进程，逐条运行spinup...")
    cfg.PARAMS['use_multiprocessing'] = False
    t_single = time.time()
    for i, gdir in enumerate(test_gdirs):
        uid = gdir.rgi_id.split('.')[-1]
        try:
            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=gcm_suffix,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_sp,
                store_model_geometry=True,
            )
            print("    [{}/{}] {} OK".format(i+1, len(test_gdirs), uid))
        except Exception as e2:
            print("    [{}/{}] {} FAIL: {}".format(
                i+1, len(test_gdirs), uid, str(e2)[:60]))
    cfg.PARAMS['use_multiprocessing'] = True  # 恢复
    print("  单进程spinup完成: {:.1f}min".format((time.time() - t_single) / 60))

# 用文件存在性筛选成功的冰川
valid_sp = []
for gdir in test_gdirs:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
        if os.path.exists(fp):
            valid_sp.append(gdir)
    except:
        pass
print("\nSpinup成功: {}/{}".format(len(valid_sp), len(test_gdirs)))

if len(valid_sp) == 0:
    print("ERROR: 没有冰川spinup成功，终止")
else:
    # ---------- 正演 ----------
    t1 = time.time()
    print("\n[正演] run_from_climate_data ({} 条冰川)...".format(len(valid_sp)))

    forward_success = False
    try:
        execute_entity_task(
            tasks.run_from_climate_data, valid_sp,
            min_ys=Y0, ye=FORWARD_END,
            climate_filename='gcm_data',
            climate_input_filesuffix=gcm_suffix,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=suffix_sp,
            output_filesuffix=suffix_fw,
        )
        forward_success = True
        print("  多进程正演完成: {:.1f}min".format((time.time() - t1) / 60))
    except Exception as e:
        print("  多进程正演出错: {}".format(str(e)[:100]))

    # 多进程失败 → 回退单进程
    if not forward_success:
        print("  [回退] 逐条运行正演...")
        cfg.PARAMS['use_multiprocessing'] = False
        t_single = time.time()
        for i, gdir in enumerate(valid_sp):
            uid = gdir.rgi_id.split('.')[-1]
            try:
                tasks.run_from_climate_data(
                    gdir,
                    min_ys=Y0, ye=FORWARD_END,
                    climate_filename='gcm_data',
                    climate_input_filesuffix=gcm_suffix,
                    mb_model_class=MyMBModel,
                    evolution_model=FluxBasedModel,
                    init_model_filesuffix=suffix_sp,
                    output_filesuffix=suffix_fw,
                )
                print("    [{}/{}] {} OK".format(i+1, len(valid_sp), uid))
            except Exception as e2:
                print("    [{}/{}] {} FAIL: {}".format(
                    i+1, len(valid_sp), uid, str(e2)[:60]))
        cfg.PARAMS['use_multiprocessing'] = True
        print("  单进程正演完成: {:.1f}min".format((time.time() - t_single) / 60))

    # ---------- 提取结果 ----------
    print("\n[提取结果]")
    results = {}
    for gdir in test_gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        lia_area = gdir.rgi_area_m2 / 1e6
        rgi_area = rgi_area_map[rgi_id]

        ds_sp = None
        ds_fw = None
        try:
            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw)
            if not os.path.exists(fp_sp) or not os.path.exists(fp_fw):
                results[rgi_id] = {'uid': uid, 'status': 'missing_file'}
                continue

            ds_sp = xr.open_dataset(fp_sp)
            sp_end = float(ds_sp['area_m2'].values[-1]) / 1e6

            ds_fw = xr.open_dataset(fp_fw)
            fw_start = float(ds_fw['area_m2'].values[0]) / 1e6
            fw_end = float(ds_fw['area_m2'].values[-1]) / 1e6

            results[rgi_id] = {
                'uid': uid, 'lia_area': lia_area, 'rgi_area': rgi_area,
                'sp_end': sp_end, 'sp_lia_ratio': sp_end / lia_area,
                'fw_start': fw_start, 'fw_end': fw_end,
                'fw_rgi_ratio': fw_end / rgi_area,
                'jump_pct': (fw_start - sp_end) / sp_end * 100 if sp_end > 0 else np.nan,
                'status': 'ok',
            }
        except Exception as e:
            results[rgi_id] = {'uid': uid, 'status': 'error: {}'.format(str(e)[:60])}
        finally:
            if ds_sp is not None:
                ds_sp.close()
            if ds_fw is not None:
                ds_fw.close()

    # ---------- 汇总 ----------
    print("\n" + "=" * 100)
    print("GCM Emean驱动结果汇总")
    print("=" * 100)
    print("{:<8} {:>8} {:>8} {:>8} {:>8} {:>8} {:>8} {:>10} {:>8}".format(
        "UID", "LIA", "RGI", "sp_end", "sp/LIA", "fw_end", "fw/RGI", "跳变%", "状态"))
    print("-" * 100)

    sp_ratios = []
    fw_ratios = []

    for rgi_id in sorted(results.keys()):
        r = results[rgi_id]
        if r['status'] == 'ok':
            print("{:<8} {:>8.2f} {:>8.2f} {:>8.2f} {:>8.3f} {:>8.2f} {:>8.3f} {:>+10.1f}% {:>8}".format(
                r['uid'], r['lia_area'], r['rgi_area'],
                r['sp_end'], r['sp_lia_ratio'],
                r['fw_end'], r['fw_rgi_ratio'],
                r['jump_pct'], 'ok'))
            sp_ratios.append(r['sp_lia_ratio'])
            fw_ratios.append(r['fw_rgi_ratio'])
        else:
            print("{:<8} {:>88} {:>8}".format(r['uid'], '', r['status'][:20]))

    if sp_ratios:
        print("\n统计 (N={}):".format(len(sp_ratios)))
        print("  sp/LIA: median={:.3f}, std={:.3f}".format(
            np.median(sp_ratios), np.std(sp_ratios)))
        print("  fw/RGI: median={:.3f}, std={:.3f}".format(
            np.median(fw_ratios), np.std(fw_ratios)))

    # ---------- 与Rec_ref offset=0 对比 ----------
    if 'all_results' in dir():
        print("\n" + "=" * 70)
        print("对比: Rec_ref (offset=0) vs GCM Emean")
        print("=" * 70)
        print("{:<8} {:>12} {:>12} {:>12} {:>12}".format(
            "UID", "Rec_sp/LIA", "Rec_fw/RGI", "GCM_sp/LIA", "GCM_fw/RGI"))
        print("-" * 60)

        for rgi_id in sorted(results.keys()):
            r_gcm = results[rgi_id]
            r_rec = all_results.get(rgi_id, {})
            rec_off0 = r_rec.get('offsets', {}).get(0, {})
            uid = rgi_id.split('.')[-1]
            if r_gcm.get('status') == 'ok' and rec_off0.get('status') == 'ok':
                print("{:<8} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f}".format(
                    uid,
                    rec_off0['sp_lia_ratio'], rec_off0['fw_rgi_ratio'],
                    r_gcm['sp_lia_ratio'], r_gcm['fw_rgi_ratio']))

print("\n总耗时: {:.1f}min".format((time.time() - t0) / 60))
print("完成!")

In [ ]:
"""
绘制各冰川 spinup + 正演 面积时间序列
======================================
每条冰川一个子图，显示：
  - 灰色: spin-up阶段面积演化
  - 蓝色: 正演阶段面积演化
  - 绿色虚线: LIA轮廓面积
  - 橙色虚线: RGI观测面积
  - 垂直线: spin-up与正演分界点

用法: 直接粘贴到你的notebook中运行
前置变量: test_gdirs, rgi_area_map, original_params, cfg
需要已经跑完 spinup 和 forward（文件已存在）
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ============================================================
# 配置 - 根据你的实验修改这些suffix
# ============================================================
# 方案1: GCM Emean实验
SUFFIX_SP = '_gcm_emean_sp'
SUFFIX_FW = '_gcm_emean_fw'
TITLE_PREFIX = 'GCM Emean'

# 方案2: Rec_ref offset sweep（取消注释切换）
# SUFFIX_SP = '_sweep_sp_off0p0'
# SUFFIX_FW = '_sweep_fw_off0p0'
# TITLE_PREFIX = 'Rec_ref offset=0'

# 方案3: MB≈0校准实验
# SUFFIX_SP = '_spinup_mb0calib'
# SUFFIX_FW = '_forward_mb0calib'
# TITLE_PREFIX = 'MB≈0 Calibration'

Y0 = 1500
SPINUP_YEARS = 300

# ============================================================
# 收集数据
# ============================================================
print("收集时间序列数据...")

ts_data = []
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    lia_area = gdir.rgi_area_m2 / 1e6
    rgi_area = rgi_area_map.get(rgi_id, np.nan)

    ds_sp = None
    ds_fw = None
    try:
        fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SP)
        fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_FW)

        if not os.path.exists(fp_sp):
            print("  {} : spinup文件不存在，跳过".format(uid))
            continue

        # 读取 spinup
        ds_sp = xr.open_dataset(fp_sp)
        area_sp = ds_sp['area_m2'].values / 1e6
        vol_sp = ds_sp['volume_m3'].values / 1e9
        n_sp = len(area_sp)
        # spinup年份: 从 Y0-SPINUP_YEARS 到 Y0
        years_sp = np.linspace(Y0 - SPINUP_YEARS, Y0, n_sp)

        entry = {
            'uid': uid, 'rgi_id': rgi_id,
            'lia_area': lia_area, 'rgi_area': rgi_area,
            'years_sp': years_sp, 'area_sp': area_sp, 'vol_sp': vol_sp,
            'sp_end_area': area_sp[-1],
            'has_forward': False,
        }

        # 读取 forward（可能不存在）
        if os.path.exists(fp_fw):
            ds_fw = xr.open_dataset(fp_fw)
            area_fw = ds_fw['area_m2'].values / 1e6
            vol_fw = ds_fw['volume_m3'].values / 1e9
            n_fw = len(area_fw)
            years_fw = np.arange(Y0, Y0 + n_fw)

            entry.update({
                'years_fw': years_fw, 'area_fw': area_fw, 'vol_fw': vol_fw,
                'fw_end_area': area_fw[-1],
                'fw_rgi_ratio': area_fw[-1] / rgi_area if rgi_area > 0 else np.nan,
                'has_forward': True,
            })

        ts_data.append(entry)
        print("  {} : sp={:.2f}→{:.2f} km²{}".format(
            uid, area_sp[0], area_sp[-1],
            "  fw→{:.2f} km²  ratio={:.3f}".format(
                entry['fw_end_area'], entry['fw_rgi_ratio'])
            if entry['has_forward'] else "  (无正演)"))

    except Exception as e:
        print("  {} : 读取失败 - {}".format(uid, str(e)[:60]))
    finally:
        if ds_sp is not None:
            ds_sp.close()
        if ds_fw is not None:
            ds_fw.close()

print("\n成功读取: {} 条冰川".format(len(ts_data)))

# ============================================================
# 图1: 每条冰川独立子图
# ============================================================
n = len(ts_data)
if n == 0:
    print("无数据，跳过绑图")
else:
    ncols = min(5, n)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows))
    if n == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, d in enumerate(ts_data):
        ax = axes[idx]

        # spinup 灰色
        ax.plot(d['years_sp'], d['area_sp'],
                color='#666666', linewidth=1.5, alpha=0.7, label='Spin-up')

        # forward 蓝色
        if d['has_forward']:
            ax.plot(d['years_fw'], d['area_fw'],
                    color='#2166ac', linewidth=2.0, label='Forward')

        # 分界线
        ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        # LIA轮廓面积 (绿色)
        ax.axhline(d['lia_area'], color='#1b7837', linestyle=':',
                    linewidth=1.5, alpha=0.8, label='LIA outline')

        # RGI观测面积 (橙色)
        ax.axhline(d['rgi_area'], color='#e66101', linestyle='--',
                    linewidth=1.8, alpha=0.9, label='RGI observed')

        # 标题
        title = 'UID {}'.format(d['uid'])
        if d['has_forward']:
            title += ' | ratio={:.3f}'.format(d['fw_rgi_ratio'])
        ax.set_title(title, fontsize=9, fontweight='bold')

        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.2)

        # 只第一个子图显示图例
        if idx == 0:
            ax.legend(fontsize=6, loc='best')

    # 隐藏多余子图
    for i in range(n, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle('{} | Spinup ({}yr) + Forward → {}\n'
                 'Green: LIA outline | Orange: RGI observed'.format(
                     TITLE_PREFIX, SPINUP_YEARS, Y0 + 500),
                 fontsize=13, fontweight='bold')
    fig.supxlabel('Year', fontsize=11)
    fig.supylabel('Area (km²)', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.PATHS['working_dir'],
                'ts_individual_{}.png'.format(SUFFIX_SP.strip('_'))),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("图1已保存")

    # ============================================================
    # 图2: 所有冰川归一化叠加
    # ============================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 左图: 面积 / LIA面积
    ax = axes[0]
    for d in ts_data:
        norm_sp = d['area_sp'] / d['lia_area']
        color = '#2166ac' if d.get('fw_rgi_ratio', 2) < 1.2 else '#b2182b'
        ax.plot(d['years_sp'], norm_sp, color=color, linewidth=0.8, alpha=0.4)
        if d['has_forward']:
            norm_fw = d['area_fw'] / d['lia_area']
            ax.plot(d['years_fw'], norm_fw, color=color, linewidth=1.2, alpha=0.6)

    ax.axhline(1.0, color='#1b7837', linestyle=':', linewidth=2, label='LIA = 1.0')
    ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Year')
    ax.set_ylabel('Area / LIA area')
    ax.set_title('Normalized by LIA outline area')
    ax.legend()
    ax.grid(True, alpha=0.2)

    # 右图: 面积 / RGI面积
    ax = axes[1]
    for d in ts_data:
        if np.isnan(d['rgi_area']) or d['rgi_area'] <= 0:
            continue
        norm_sp = d['area_sp'] / d['rgi_area']
        color = '#2166ac' if d.get('fw_rgi_ratio', 2) < 1.2 else '#b2182b'
        ax.plot(d['years_sp'], norm_sp, color=color, linewidth=0.8, alpha=0.4)
        if d['has_forward']:
            norm_fw = d['area_fw'] / d['rgi_area']
            ax.plot(d['years_fw'], norm_fw, color=color, linewidth=1.2, alpha=0.6)

    ax.axhline(1.0, color='#e66101', linestyle='--', linewidth=2, label='RGI = 1.0')
    ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Year')
    ax.set_ylabel('Area / RGI area')
    ax.set_title('Normalized by RGI observed area\n(blue: ratio<1.2, red: ratio≥1.2)')
    ax.legend()
    ax.grid(True, alpha=0.2)

    fig.suptitle('{} | All glaciers overlay'.format(TITLE_PREFIX),
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.PATHS['working_dir'],
                'ts_overlay_{}.png'.format(SUFFIX_SP.strip('_'))),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("图2已保存")

    # ============================================================
    # 图3: ratio柱状图
    # ============================================================
    fw_data = [d for d in ts_data if d['has_forward'] and not np.isnan(d.get('fw_rgi_ratio', np.nan))]
    if fw_data:
        fw_data_sorted = sorted(fw_data, key=lambda x: x['fw_rgi_ratio'])
        uids = [d['uid'] for d in fw_data_sorted]
        ratios = [d['fw_rgi_ratio'] for d in fw_data_sorted]
        colors = ['#2166ac' if 0.8 <= r <= 1.2 else '#b2182b' for r in ratios]

        fig, ax = plt.subplots(figsize=(max(10, len(uids) * 0.6), 5))
        bars = ax.bar(range(len(uids)), ratios, color=colors,
                      edgecolor='black', linewidth=0.5, alpha=0.8)
        ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5, label='ratio=1.0')
        ax.fill_between([-0.5, len(uids)-0.5], 0.8, 1.2,
                        alpha=0.1, color='green', label='[0.8, 1.2]')

        ax.set_xticks(range(len(uids)))
        ax.set_xticklabels(uids, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('sim_2000 / RGI area')
        ax.set_xlabel('Glacier UID')
        ax.set_title('{} | Forward end area / RGI observed\n'
                     'median={:.3f}, [0.8-1.2]: {}/{}'.format(
                         TITLE_PREFIX,
                         np.median(ratios),
                         sum(1 for r in ratios if 0.8 <= r <= 1.2),
                         len(ratios)))
        ax.legend()
        ax.grid(True, alpha=0.2, axis='y')

        # 数值标注
        for bar, r in zip(bars, ratios):
            ax.text(bar.get_x() + bar.get_width()/2,
                    r + 0.02 * (1 if r >= 0 else -1),
                    '{:.2f}'.format(r), ha='center', va='bottom', fontsize=7)

        plt.tight_layout()
        plt.savefig(os.path.join(cfg.PATHS['working_dir'],
                    'ratio_bar_{}.png'.format(SUFFIX_SP.strip('_'))),
                    dpi=150, bbox_inches='tight')
        plt.show()
        print("图3已保存")

    # ============================================================
    # 汇总表
    # ============================================================
    print("\n" + "=" * 90)
    print("汇总")
    print("=" * 90)
    print("{:<8} {:>8} {:>8} {:>10} {:>10} {:>10} {:>8}".format(
        "UID", "LIA", "RGI", "SP_end", "FW_end", "FW/RGI", "状态"))
    print("-" * 70)
    for d in sorted(ts_data, key=lambda x: x.get('fw_rgi_ratio', 999)):
        if d['has_forward']:
            print("{:<8} {:>8.2f} {:>8.2f} {:>10.2f} {:>10.2f} {:>10.3f} {:>8}".format(
                d['uid'], d['lia_area'], d['rgi_area'],
                d['sp_end_area'], d['fw_end_area'], d['fw_rgi_ratio'], 'ok'))
        else:
            print("{:<8} {:>8.2f} {:>8.2f} {:>10.2f} {:>10} {:>10} {:>8}".format(
                d['uid'], d['lia_area'], d['rgi_area'],
                d['sp_end_area'], '-', '-', 'no_fw'))

    if fw_data:
        ratios_all = [d['fw_rgi_ratio'] for d in fw_data]
        n_good = sum(1 for r in ratios_all if 0.8 <= r <= 1.2)
        print("-" * 70)
        print("ratio: mean={:.3f}  median={:.3f}  std={:.3f}".format(
            np.mean(ratios_all), np.median(ratios_all), np.std(ratios_all)))
        print("[0.8-1.2]: {}/{} = {:.1f}%".format(
            n_good, len(ratios_all),
            n_good / len(ratios_all) * 100))

print("\n绑图完成!")

In [ ]:
"""
重建气候(ensemble_rec_ref)驱动: spinup + 正演, 20条冰川
防卡死改进版：
  1. task_timeout=600
  2. execute_entity_task 外层 try-except + 单进程回退
  3. 文件存在性筛选，失败跳过
  4. xr.open_dataset 全部用 try-finally 确保关闭
"""

import json, os, time
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ============================================================
# 配置
# ============================================================
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_END = 2000

# 重建气候suffix（替代GCM Emean的 '_Emean'）
climate_suffix = '_ensemble_rec_ref'

# 防卡死关键配置
cfg.PARAMS['task_timeout'] = 600
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True

# ============================================================
# 检查重建气候文件
# ============================================================
print("检查重建气候文件...")
sample_dir = test_gdirs[0].dir
gcm_files = [f for f in sorted(os.listdir(sample_dir)) if f.startswith('gcm_data')]
print("可用气候文件:")
for f in gcm_files:
    print("  {}".format(f))

n_exist = sum(1 for g in test_gdirs
              if os.path.exists(os.path.join(g.dir, 'gcm_data{}.nc'.format(climate_suffix))))
print("找到 {}/{} 条冰川有气候文件 (suffix='{}')".format(n_exist, len(test_gdirs), climate_suffix))

if n_exist == 0:
    raise FileNotFoundError("没有找到气候文件，请检查suffix")

# ============================================================
# 恢复原始参数
# ============================================================
for gdir in test_gdirs:
    orig = original_params[gdir.rgi_id]
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = orig['melt_f']
    p['temp_bias'] = orig['temp_bias']
    p['prcp_fac'] = orig['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)

print("参数已恢复为原始值")

# ============================================================
# 主实验
# ============================================================
print("=" * 70)
print("重建气候驱动: spinup {}yr (Y0={}) + 正演→{}".format(SPINUP_YEARS, Y0, FORWARD_END))
print("suffix = '{}', timeout={}s, processes={}".format(
    climate_suffix, cfg.PARAMS['task_timeout'], cfg.PARAMS['mp_processes']))
print("=" * 70)

suffix_sp = '_rec_ref_sp'
suffix_fw = '_rec_ref_fw'

# ---------- Spinup ----------
t0 = time.time()
print("\n[Spinup] run_constant_climate ({} 条冰川)...".format(len(test_gdirs)))

spinup_success = False
try:
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=climate_suffix,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=suffix_sp,
        store_model_geometry=True,
    )
    spinup_success = True
    print("  多进程Spinup完成: {:.1f}min".format((time.time() - t0) / 60))
except Exception as e:
    print("  多进程Spinup出错: {}".format(str(e)[:100]))

# 多进程失败 → 回退单进程逐条跑
if not spinup_success:
    print("  [回退] 关闭多进程，逐条运行spinup...")
    cfg.PARAMS['use_multiprocessing'] = False
    t_single = time.time()
    for i, gdir in enumerate(test_gdirs):
        uid = gdir.rgi_id.split('.')[-1]
        try:
            tasks.run_constant_climate(
                gdir,
                nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=climate_suffix,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_sp,
                store_model_geometry=True,
            )
            print("    [{}/{}] {} OK".format(i+1, len(test_gdirs), uid))
        except Exception as e2:
            print("    [{}/{}] {} FAIL: {}".format(
                i+1, len(test_gdirs), uid, str(e2)[:60]))
    cfg.PARAMS['use_multiprocessing'] = True
    print("  单进程spinup完成: {:.1f}min".format((time.time() - t_single) / 60))

# 用文件存在性筛选成功的冰川
valid_sp = []
for gdir in test_gdirs:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
        if os.path.exists(fp):
            valid_sp.append(gdir)
    except:
        pass
print("\nSpinup成功: {}/{}".format(len(valid_sp), len(test_gdirs)))

if len(valid_sp) == 0:
    print("ERROR: 没有冰川spinup成功，终止")
else:
    # ---------- 正演 ----------
    t1 = time.time()
    print("\n[正演] run_from_climate_data ({} 条冰川)...".format(len(valid_sp)))

    forward_success = False
    try:
        execute_entity_task(
            tasks.run_from_climate_data, valid_sp,
            min_ys=Y0, ye=FORWARD_END,
            climate_filename='gcm_data',
            climate_input_filesuffix=climate_suffix,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=suffix_sp,
            output_filesuffix=suffix_fw,
        )
        forward_success = True
        print("  多进程正演完成: {:.1f}min".format((time.time() - t1) / 60))
    except Exception as e:
        print("  多进程正演出错: {}".format(str(e)[:100]))

    # 多进程失败 → 回退单进程
    if not forward_success:
        print("  [回退] 逐条运行正演...")
        cfg.PARAMS['use_multiprocessing'] = False
        t_single = time.time()
        for i, gdir in enumerate(valid_sp):
            uid = gdir.rgi_id.split('.')[-1]
            try:
                tasks.run_from_climate_data(
                    gdir,
                    min_ys=Y0, ye=FORWARD_END,
                    climate_filename='gcm_data',
                    climate_input_filesuffix=climate_suffix,
                    mb_model_class=MyMBModel,
                    evolution_model=FluxBasedModel,
                    init_model_filesuffix=suffix_sp,
                    output_filesuffix=suffix_fw,
                )
                print("    [{}/{}] {} OK".format(i+1, len(valid_sp), uid))
            except Exception as e2:
                print("    [{}/{}] {} FAIL: {}".format(
                    i+1, len(valid_sp), uid, str(e2)[:60]))
        cfg.PARAMS['use_multiprocessing'] = True
        print("  单进程正演完成: {:.1f}min".format((time.time() - t_single) / 60))

    # ---------- 提取结果 ----------
    print("\n[提取结果]")
    results_rec = {}
    for gdir in test_gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        lia_area = gdir.rgi_area_m2 / 1e6
        rgi_area = rgi_area_map[rgi_id]

        ds_sp = None
        ds_fw = None
        try:
            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw)
            if not os.path.exists(fp_sp) or not os.path.exists(fp_fw):
                results_rec[rgi_id] = {'uid': uid, 'status': 'missing_file'}
                continue

            ds_sp = xr.open_dataset(fp_sp)
            sp_end = float(ds_sp['area_m2'].values[-1]) / 1e6

            ds_fw = xr.open_dataset(fp_fw)
            fw_start = float(ds_fw['area_m2'].values[0]) / 1e6
            fw_end = float(ds_fw['area_m2'].values[-1]) / 1e6

            results_rec[rgi_id] = {
                'uid': uid, 'lia_area': lia_area, 'rgi_area': rgi_area,
                'sp_end': sp_end, 'sp_lia_ratio': sp_end / lia_area,
                'fw_start': fw_start, 'fw_end': fw_end,
                'fw_rgi_ratio': fw_end / rgi_area,
                'jump_pct': (fw_start - sp_end) / sp_end * 100 if sp_end > 0 else np.nan,
                'status': 'ok',
            }
        except Exception as e:
            results_rec[rgi_id] = {'uid': uid, 'status': 'error: {}'.format(str(e)[:60])}
        finally:
            if ds_sp is not None:
                ds_sp.close()
            if ds_fw is not None:
                ds_fw.close()

    # ---------- 汇总 ----------
    print("\n" + "=" * 100)
    print("重建气候(ensemble_rec_ref)驱动结果汇总")
    print("=" * 100)
    print("{:<8} {:>8} {:>8} {:>8} {:>8} {:>8} {:>8} {:>10} {:>8}".format(
        "UID", "LIA", "RGI", "sp_end", "sp/LIA", "fw_end", "fw/RGI", "跳变%", "状态"))
    print("-" * 100)

    sp_ratios = []
    fw_ratios = []

    for rgi_id in sorted(results_rec.keys()):
        r = results_rec[rgi_id]
        if r['status'] == 'ok':
            print("{:<8} {:>8.2f} {:>8.2f} {:>8.2f} {:>8.3f} {:>8.2f} {:>8.3f} {:>+10.1f}% {:>8}".format(
                r['uid'], r['lia_area'], r['rgi_area'],
                r['sp_end'], r['sp_lia_ratio'],
                r['fw_end'], r['fw_rgi_ratio'],
                r['jump_pct'], 'ok'))
            sp_ratios.append(r['sp_lia_ratio'])
            fw_ratios.append(r['fw_rgi_ratio'])
        else:
            print("{:<8} {:>88} {:>8}".format(r['uid'], '', r['status'][:20]))

    if sp_ratios:
        print("\n统计 (N={}):".format(len(sp_ratios)))
        print("  sp/LIA: median={:.3f}, mean={:.3f}, std={:.3f}".format(
            np.median(sp_ratios), np.mean(sp_ratios), np.std(sp_ratios)))
        print("  fw/RGI: median={:.3f}, mean={:.3f}, std={:.3f}".format(
            np.median(fw_ratios), np.mean(fw_ratios), np.std(fw_ratios)))
        n_good = sum(1 for r in fw_ratios if 0.8 <= r <= 1.2)
        print("  fw/RGI在[0.8-1.2]: {}/{} = {:.1f}%".format(
            n_good, len(fw_ratios), n_good / len(fw_ratios) * 100))

    # ---------- 与GCM Emean对比（如果results变量存在）----------
    if 'results' in dir() and isinstance(results, dict):
        print("\n" + "=" * 70)
        print("对比: 重建气候(Rec_ref) vs GCM Emean")
        print("=" * 70)
        print("{:<8} {:>12} {:>12} {:>12} {:>12}".format(
            "UID", "Rec_sp/LIA", "Rec_fw/RGI", "GCM_sp/LIA", "GCM_fw/RGI"))
        print("-" * 60)

        rec_fw_all = []
        gcm_fw_all = []
        for rgi_id in sorted(results_rec.keys()):
            r_rec = results_rec[rgi_id]
            r_gcm = results.get(rgi_id, {})
            uid = rgi_id.split('.')[-1]
            if r_rec.get('status') == 'ok' and r_gcm.get('status') == 'ok':
                print("{:<8} {:>12.3f} {:>12.3f} {:>12.3f} {:>12.3f}".format(
                    uid,
                    r_rec['sp_lia_ratio'], r_rec['fw_rgi_ratio'],
                    r_gcm['sp_lia_ratio'], r_gcm['fw_rgi_ratio']))
                rec_fw_all.append(r_rec['fw_rgi_ratio'])
                gcm_fw_all.append(r_gcm['fw_rgi_ratio'])

        if rec_fw_all and gcm_fw_all:
            print("-" * 60)
            print("{:<8} {:>12} {:>12.3f} {:>12} {:>12.3f}".format(
                "median", "", np.median(rec_fw_all), "", np.median(gcm_fw_all)))
            print("{:<8} {:>12} {:>12.3f} {:>12} {:>12.3f}".format(
                "mean", "", np.mean(rec_fw_all), "", np.mean(gcm_fw_all)))
            print("\n结论:")
            if np.median(rec_fw_all) > np.median(gcm_fw_all):
                print("  Rec_ref fw/RGI ({:.3f}) > GCM ({:.3f})".format(
                    np.median(rec_fw_all), np.median(gcm_fw_all)))
                print("  → 重建气候振幅不足，冰川退缩不够，ratio偏大")
            else:
                print("  Rec_ref fw/RGI ({:.3f}) <= GCM ({:.3f})".format(
                    np.median(rec_fw_all), np.median(gcm_fw_all)))
                print("  → 重建气候振幅足够或更大")

print("\n总耗时: {:.1f}min".format((time.time() - t0) / 60))
print("完成!")

In [ ]:
"""
绘制 重建气候 vs GCM Emean 双气候源 spinup+正演 时间序列
==========================================================
图1: 重建气候 - 每条冰川子图
图2: GCM Emean - 每条冰川子图
图3: 双源对比 - 每条冰川同一子图叠加两条线
图4: ratio柱状图对比
图5: 归一化叠加

前置变量: test_gdirs, rgi_area_map, original_params, cfg
需要已跑完两套实验的spinup和forward文件
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ============================================================
# 配置
# ============================================================
Y0 = 1500
SPINUP_YEARS = 300

# 两套实验的suffix
EXPERIMENTS = {
    'Rec_ref': {
        'suffix_sp': '_rec_ref_sp',
        'suffix_fw': '_rec_ref_fw',
        'color_sp': '#888888',
        'color_fw': '#2166ac',
        'label': 'Rec_ref',
    },
    'GCM_Emean': {
        'suffix_sp': '_gcm_emean_sp',
        'suffix_fw': '_gcm_emean_fw',
        'color_sp': '#bbbbbb',
        'color_fw': '#b2182b',
        'label': 'GCM Emean',
    },
}

# ============================================================
# 收集数据
# ============================================================
print("收集时间序列数据...")

all_ts = {}  # {exp_name: {rgi_id: {...}}}

for exp_name, exp_cfg in EXPERIMENTS.items():
    all_ts[exp_name] = {}
    n_ok = 0

    for gdir in test_gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        lia_area = gdir.rgi_area_m2 / 1e6
        rgi_area = rgi_area_map.get(rgi_id, np.nan)

        ds_sp = None
        ds_fw = None
        try:
            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=exp_cfg['suffix_sp'])
            if not os.path.exists(fp_sp):
                continue

            ds_sp = xr.open_dataset(fp_sp)
            area_sp = ds_sp['area_m2'].values / 1e6
            n_sp = len(area_sp)
            years_sp = np.linspace(Y0 - SPINUP_YEARS, Y0, n_sp)

            entry = {
                'uid': uid, 'rgi_id': rgi_id,
                'lia_area': lia_area, 'rgi_area': rgi_area,
                'years_sp': years_sp, 'area_sp': area_sp,
                'sp_end': area_sp[-1],
                'has_forward': False,
            }

            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=exp_cfg['suffix_fw'])
            if os.path.exists(fp_fw):
                ds_fw = xr.open_dataset(fp_fw)
                area_fw = ds_fw['area_m2'].values / 1e6
                n_fw = len(area_fw)
                years_fw = np.arange(Y0, Y0 + n_fw)
                entry.update({
                    'years_fw': years_fw, 'area_fw': area_fw,
                    'fw_end': area_fw[-1],
                    'fw_rgi_ratio': area_fw[-1] / rgi_area if rgi_area > 0 else np.nan,
                    'has_forward': True,
                })

            all_ts[exp_name][rgi_id] = entry
            n_ok += 1

        except Exception as e:
            pass
        finally:
            if ds_sp is not None:
                ds_sp.close()
            if ds_fw is not None:
                ds_fw.close()

    print("  {}: {} 条冰川".format(exp_name, n_ok))

# 找两套实验都成功的冰川
common_ids = sorted(
    set(all_ts['Rec_ref'].keys()) & set(all_ts['GCM_Emean'].keys()))
print("两套实验都成功: {} 条".format(len(common_ids)))

if len(common_ids) == 0:
    print("无共同冰川数据，检查实验是否完成")
else:

    # ============================================================
    # 图1 & 图2: 每套实验独立子图
    # ============================================================
    for exp_name, exp_cfg in EXPERIMENTS.items():
        ts = all_ts[exp_name]
        glaciers = [ts[rid] for rid in common_ids if rid in ts]
        n = len(glaciers)
        if n == 0:
            continue

        ncols = min(5, n)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4 * nrows))
        if nrows * ncols == 1:
            axes = np.array([axes])
        axes = axes.flatten()

        for idx, d in enumerate(glaciers):
            ax = axes[idx]
            ax.plot(d['years_sp'], d['area_sp'],
                    color=exp_cfg['color_sp'], linewidth=1.5, alpha=0.7, label='Spin-up')
            if d['has_forward']:
                ax.plot(d['years_fw'], d['area_fw'],
                        color=exp_cfg['color_fw'], linewidth=2.0, label='Forward')
            ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
            ax.axhline(d['lia_area'], color='#1b7837', linestyle=':', linewidth=1.5, alpha=0.8)
            ax.axhline(d['rgi_area'], color='#e66101', linestyle='--', linewidth=1.8, alpha=0.9)

            title = '{}'.format(d['uid'])
            if d['has_forward']:
                title += ' | r={:.3f}'.format(d['fw_rgi_ratio'])
            ax.set_title(title, fontsize=9, fontweight='bold')
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.2)
            if idx == 0:
                ax.legend(fontsize=6, loc='best')

        for i in range(n, len(axes)):
            axes[i].set_visible(False)

        fig.suptitle('{} | Spinup {}yr + Forward → 2000\n'
                     'Green: LIA | Orange: RGI'.format(exp_name, SPINUP_YEARS),
                     fontsize=13, fontweight='bold')
        fig.supxlabel('Year', fontsize=11)
        fig.supylabel('Area (km²)', fontsize=11)
        plt.tight_layout()
        save_path = os.path.join(cfg.PATHS['working_dir'],
                                 'ts_individual_{}.png'.format(exp_name))
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print("图已保存: {}".format(save_path))

    # ============================================================
    # 图3: 双源对比 - 每条冰川同一子图叠两条线
    # ============================================================
    n = len(common_ids)
    ncols = min(5, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
    if nrows * ncols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, rgi_id in enumerate(common_ids):
        ax = axes[idx]
        d_rec = all_ts['Rec_ref'][rgi_id]
        d_gcm = all_ts['GCM_Emean'][rgi_id]
        uid = d_rec['uid']

        # Rec_ref
        ax.plot(d_rec['years_sp'], d_rec['area_sp'],
                color='#2166ac', linewidth=1.0, alpha=0.4)
        if d_rec['has_forward']:
            ax.plot(d_rec['years_fw'], d_rec['area_fw'],
                    color='#2166ac', linewidth=2.0, label='Rec_ref')

        # GCM Emean
        ax.plot(d_gcm['years_sp'], d_gcm['area_sp'],
                color='#b2182b', linewidth=1.0, alpha=0.4)
        if d_gcm['has_forward']:
            ax.plot(d_gcm['years_fw'], d_gcm['area_fw'],
                    color='#b2182b', linewidth=2.0, label='GCM Emean')

        # 参考线
        ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
        ax.axhline(d_rec['lia_area'], color='#1b7837', linestyle=':',
                    linewidth=1.5, alpha=0.7)
        ax.axhline(d_rec['rgi_area'], color='#e66101', linestyle='--',
                    linewidth=1.8, alpha=0.9)

        # 标题
        r_rec = d_rec.get('fw_rgi_ratio', np.nan)
        r_gcm = d_gcm.get('fw_rgi_ratio', np.nan)
        title_parts = [uid]
        if not np.isnan(r_rec):
            title_parts.append('Rec={:.2f}'.format(r_rec))
        if not np.isnan(r_gcm):
            title_parts.append('GCM={:.2f}'.format(r_gcm))
        ax.set_title(' | '.join(title_parts), fontsize=8, fontweight='bold')

        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.2)
        if idx == 0:
            ax.legend(fontsize=6, loc='best')

    for i in range(n, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle('Rec_ref vs GCM Emean | 双源对比\n'
                 'Blue=Rec_ref  Red=GCM  Green:LIA  Orange:RGI',
                 fontsize=13, fontweight='bold')
    fig.supxlabel('Year', fontsize=11)
    fig.supylabel('Area (km²)', fontsize=11)
    plt.tight_layout()
    save_path = os.path.join(cfg.PATHS['working_dir'], 'ts_dual_comparison.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("双源对比图已保存: {}".format(save_path))

    # ============================================================
    # 图4: ratio柱状图对比（分组柱状图）
    # ============================================================
    fw_ids = [rid for rid in common_ids
              if all_ts['Rec_ref'][rid]['has_forward']
              and all_ts['GCM_Emean'][rid]['has_forward']]

    if fw_ids:
        uids = [all_ts['Rec_ref'][rid]['uid'] for rid in fw_ids]
        ratios_rec = [all_ts['Rec_ref'][rid]['fw_rgi_ratio'] for rid in fw_ids]
        ratios_gcm = [all_ts['GCM_Emean'][rid]['fw_rgi_ratio'] for rid in fw_ids]

        # 按Rec_ref ratio排序
        order = np.argsort(ratios_rec)
        uids = [uids[i] for i in order]
        ratios_rec = [ratios_rec[i] for i in order]
        ratios_gcm = [ratios_gcm[i] for i in order]

        x = np.arange(len(uids))
        width = 0.35

        fig, ax = plt.subplots(figsize=(max(12, len(uids) * 0.8), 6))
        bars1 = ax.bar(x - width/2, ratios_rec, width,
                        color='#2166ac', alpha=0.8, edgecolor='black',
                        linewidth=0.5, label='Rec_ref')
        bars2 = ax.bar(x + width/2, ratios_gcm, width,
                        color='#b2182b', alpha=0.8, edgecolor='black',
                        linewidth=0.5, label='GCM Emean')

        ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5)
        ax.fill_between([-0.5, len(uids)-0.5], 0.8, 1.2,
                        alpha=0.1, color='green', label='[0.8, 1.2]')

        ax.set_xticks(x)
        ax.set_xticklabels(uids, rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('sim_2000 / RGI area', fontsize=11)
        ax.set_xlabel('Glacier UID', fontsize=11)

        n_good_rec = sum(1 for r in ratios_rec if 0.8 <= r <= 1.2)
        n_good_gcm = sum(1 for r in ratios_gcm if 0.8 <= r <= 1.2)
        ax.set_title('fw/RGI ratio对比\n'
                      'Rec_ref: med={:.3f}, [0.8-1.2]={}/{} | '
                      'GCM: med={:.3f}, [0.8-1.2]={}/{}'.format(
                          np.median(ratios_rec), n_good_rec, len(ratios_rec),
                          np.median(ratios_gcm), n_good_gcm, len(ratios_gcm)),
                      fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.2, axis='y')

        # 数值标注
        for bar, r in zip(bars1, ratios_rec):
            ax.text(bar.get_x() + bar.get_width()/2, r + 0.02,
                    '{:.2f}'.format(r), ha='center', va='bottom', fontsize=6,
                    color='#2166ac')
        for bar, r in zip(bars2, ratios_gcm):
            ax.text(bar.get_x() + bar.get_width()/2, r + 0.02,
                    '{:.2f}'.format(r), ha='center', va='bottom', fontsize=6,
                    color='#b2182b')

        plt.tight_layout()
        save_path = os.path.join(cfg.PATHS['working_dir'], 'ratio_bar_dual.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print("ratio柱状图已保存: {}".format(save_path))

    # ============================================================
    # 图5: 归一化叠加（面积/RGI）
    # ============================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax_idx, (exp_name, exp_cfg) in enumerate(EXPERIMENTS.items()):
        ax = axes[ax_idx]
        ts = all_ts[exp_name]

        for rgi_id in common_ids:
            if rgi_id not in ts:
                continue
            d = ts[rgi_id]
            rgi_a = d['rgi_area']
            if np.isnan(rgi_a) or rgi_a <= 0:
                continue

            ratio_val = d.get('fw_rgi_ratio', 2.0)
            color = exp_cfg['color_fw'] if ratio_val < 1.2 else '#999999'

            ax.plot(d['years_sp'], d['area_sp'] / rgi_a,
                    color=color, linewidth=0.8, alpha=0.3)
            if d['has_forward']:
                ax.plot(d['years_fw'], d['area_fw'] / rgi_a,
                        color=color, linewidth=1.2, alpha=0.6)

        ax.axhline(1.0, color='#e66101', linestyle='--', linewidth=2,
                    label='RGI = 1.0')
        ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.4)
        ax.set_xlabel('Year')
        ax.set_ylabel('Area / RGI area')
        ax.set_title('{}'.format(exp_name), fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.2)

    fig.suptitle('Normalized area (/ RGI) | Rec_ref vs GCM Emean',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_path = os.path.join(cfg.PATHS['working_dir'], 'ts_overlay_dual.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("归一化叠加图已保存: {}".format(save_path))

    # ============================================================
    # 汇总表
    # ============================================================
    print("\n" + "=" * 100)
    print("双源对比汇总表")
    print("=" * 100)
    print("{:<8} {:>8} {:>8} {:>10} {:>10} {:>10} {:>10}".format(
        "UID", "LIA", "RGI", "Rec_fw", "Rec_ratio", "GCM_fw", "GCM_ratio"))
    print("-" * 70)

    for rgi_id in common_ids:
        d_rec = all_ts['Rec_ref'].get(rgi_id, {})
        d_gcm = all_ts['GCM_Emean'].get(rgi_id, {})
        uid = d_rec.get('uid', '?')

        rec_fw = d_rec.get('fw_end', np.nan)
        rec_r = d_rec.get('fw_rgi_ratio', np.nan)
        gcm_fw = d_gcm.get('fw_end', np.nan)
        gcm_r = d_gcm.get('fw_rgi_ratio', np.nan)

        print("{:<8} {:>8.2f} {:>8.2f} {:>10.2f} {:>10.3f} {:>10.2f} {:>10.3f}".format(
            uid, d_rec.get('lia_area', np.nan), d_rec.get('rgi_area', np.nan),
            rec_fw if not np.isnan(rec_fw) else 0,
            rec_r if not np.isnan(rec_r) else 0,
            gcm_fw if not np.isnan(gcm_fw) else 0,
            gcm_r if not np.isnan(gcm_r) else 0))

    # 汇总统计
    rec_ratios = [all_ts['Rec_ref'][rid]['fw_rgi_ratio']
                  for rid in fw_ids if not np.isnan(all_ts['Rec_ref'][rid].get('fw_rgi_ratio', np.nan))]
    gcm_ratios = [all_ts['GCM_Emean'][rid]['fw_rgi_ratio']
                  for rid in fw_ids if not np.isnan(all_ts['GCM_Emean'][rid].get('fw_rgi_ratio', np.nan))]

    print("-" * 70)
    if rec_ratios:
        print("Rec_ref:  mean={:.3f}  median={:.3f}  std={:.3f}  [0.8-1.2]={}/{}".format(
            np.mean(rec_ratios), np.median(rec_ratios), np.std(rec_ratios),
            sum(1 for r in rec_ratios if 0.8 <= r <= 1.2), len(rec_ratios)))
    if gcm_ratios:
        print("GCM:      mean={:.3f}  median={:.3f}  std={:.3f}  [0.8-1.2]={}/{}".format(
            np.mean(gcm_ratios), np.median(gcm_ratios), np.std(gcm_ratios),
            sum(1 for r in gcm_ratios if 0.8 <= r <= 1.2), len(gcm_ratios)))

    if rec_ratios and gcm_ratios:
        print("\n结论:")
        diff = np.median(rec_ratios) - np.median(gcm_ratios)
        if diff > 0.05:
            print("  GCM Emean比Rec_ref更接近RGI (中位数差={:+.3f})".format(-diff))
            print("  → 重建气候振幅偏小是ratio偏大的重要原因")
        elif diff < -0.05:
            print("  Rec_ref比GCM Emean更接近RGI (中位数差={:+.3f})".format(-diff))
        else:
            print("  两者差异不大 (中位数差={:+.3f})".format(-diff))

print("\n绘图完成!")

In [ ]:
"""
批量运行所有可用气候数据: spinup + 正演
==========================================
对每套气候数据跑同样的流程:
  spinup 300yr (Y0=1500±30) + 正演→2000
  提取结果并汇总对比

已跑过的: _ensemble_rec_ref, _Emean
待跑: _ensemble_6, _ensemble_9, _ensemble_all_methods,
      _ensemble_my_methods, _ensemble_rec_result, _rec_test
"""

import json, os, time, gc
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ============================================================
# 配置
# ============================================================
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
FORWARD_END = 2000

cfg.PARAMS['task_timeout'] = 600
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 12
cfg.PARAMS['continue_on_error'] = True

# 要跑的气候数据suffix列表（排除已跑过的）
ALREADY_RUN = ['_ensemble_rec_ref', '_Emean']

# 自动检测所有可用的气候文件
sample_dir = test_gdirs[0].dir
all_gcm_files = sorted([f for f in os.listdir(sample_dir)
                         if f.startswith('gcm_data') and f.endswith('.nc')])

ALL_SUFFIXES = []
for f in all_gcm_files:
    suffix = '_' + f.replace('gcm_data_', '').replace('.nc', '')
    if suffix not in ALREADY_RUN:
        ALL_SUFFIXES.append(suffix)

print("=" * 70)
print("批量运行所有气候数据")
print("=" * 70)
print("已跑过: {}".format(ALREADY_RUN))
print("待运行: {}".format(ALL_SUFFIXES))
print("冰川数: {}, spinup={}yr, Y0={}, 正演→{}".format(
    len(test_gdirs), SPINUP_YEARS, Y0, FORWARD_END))
print("=" * 70)

# ============================================================
# 保存目录
# ============================================================
SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'multi_climate_results')
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# 核心函数
# ============================================================

def restore_original_params(gdirs):
    """恢复所有冰川原始参数"""
    for gdir in gdirs:
        orig = original_params[gdir.rgi_id]
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['melt_f'] = orig['melt_f']
        p['temp_bias'] = orig['temp_bias']
        p['prcp_fac'] = orig['prcp_fac']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)


def run_one_climate(gdirs, climate_suffix):
    """
    对一套气候数据运行 spinup + 正演，返回结果字典
    包含防卡死: try-except + 单进程回退
    """
    # suffix命名: 去掉开头的下划线，简化
    tag = climate_suffix.strip('_')
    suffix_sp = '_sp_{}'.format(tag)
    suffix_fw = '_fw_{}'.format(tag)

    print("\n" + "=" * 60)
    print("[{}] 开始运行".format(tag))
    print("  climate_suffix='{}', suffix_sp='{}', suffix_fw='{}'".format(
        climate_suffix, suffix_sp, suffix_fw))
    print("=" * 60)

    # 检查气候文件存在性
    valid_gdirs = []
    for g in gdirs:
        fp = os.path.join(g.dir, 'gcm_data{}.nc'.format(climate_suffix))
        if os.path.exists(fp):
            valid_gdirs.append(g)
    print("  有气候文件的冰川: {}/{}".format(len(valid_gdirs), len(gdirs)))

    if len(valid_gdirs) == 0:
        print("  跳过: 无气候文件")
        return None

    # 恢复原始参数
    restore_original_params(valid_gdirs)

    # ---------- Spinup ----------
    t0 = time.time()
    print("  [Spinup] {} 条冰川...".format(len(valid_gdirs)))

    spinup_success = False
    try:
        execute_entity_task(
            tasks.run_constant_climate, valid_gdirs,
            nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
            climate_filename='gcm_data',
            climate_input_filesuffix=climate_suffix,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            output_filesuffix=suffix_sp,
            store_model_geometry=True,
        )
        spinup_success = True
        print("    多进程完成: {:.1f}min".format((time.time() - t0) / 60))
    except Exception as e:
        print("    多进程出错: {}".format(str(e)[:80]))

    # 回退单进程
    if not spinup_success:
        print("    [回退] 逐条spinup...")
        cfg.PARAMS['use_multiprocessing'] = False
        for i, gdir in enumerate(valid_gdirs):
            uid = gdir.rgi_id.split('.')[-1]
            try:
                tasks.run_constant_climate(
                    gdir,
                    nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
                    climate_filename='gcm_data',
                    climate_input_filesuffix=climate_suffix,
                    mb_model_class=MyMBModel,
                    evolution_model=FluxBasedModel,
                    output_filesuffix=suffix_sp,
                    store_model_geometry=True,
                )
            except Exception as e2:
                if i < 3:
                    print("      {} FAIL: {}".format(uid, str(e2)[:50]))
        cfg.PARAMS['use_multiprocessing'] = True
        print("    单进程完成: {:.1f}min".format((time.time() - t0) / 60))

    # 筛选spinup成功的
    sp_ok = []
    for gdir in valid_gdirs:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            if os.path.exists(fp):
                sp_ok.append(gdir)
        except:
            pass
    print("  Spinup成功: {}/{}".format(len(sp_ok), len(valid_gdirs)))

    if len(sp_ok) == 0:
        print("  跳过正演: 无spinup成功")
        return None

    # ---------- 正演 ----------
    t1 = time.time()
    print("  [正演] {} 条冰川...".format(len(sp_ok)))

    forward_success = False
    try:
        execute_entity_task(
            tasks.run_from_climate_data, sp_ok,
            min_ys=Y0, ye=FORWARD_END,
            climate_filename='gcm_data',
            climate_input_filesuffix=climate_suffix,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=suffix_sp,
            output_filesuffix=suffix_fw,
        )
        forward_success = True
        print("    多进程完成: {:.1f}min".format((time.time() - t1) / 60))
    except Exception as e:
        print("    多进程出错: {}".format(str(e)[:80]))

    if not forward_success:
        print("    [回退] 逐条正演...")
        cfg.PARAMS['use_multiprocessing'] = False
        for i, gdir in enumerate(sp_ok):
            uid = gdir.rgi_id.split('.')[-1]
            try:
                tasks.run_from_climate_data(
                    gdir,
                    min_ys=Y0, ye=FORWARD_END,
                    climate_filename='gcm_data',
                    climate_input_filesuffix=climate_suffix,
                    mb_model_class=MyMBModel,
                    evolution_model=FluxBasedModel,
                    init_model_filesuffix=suffix_sp,
                    output_filesuffix=suffix_fw,
                )
            except Exception as e2:
                if i < 3:
                    print("      {} FAIL: {}".format(uid, str(e2)[:50]))
        cfg.PARAMS['use_multiprocessing'] = True
        print("    单进程完成: {:.1f}min".format((time.time() - t1) / 60))

    # ---------- 提取结果 ----------
    results = {}
    for gdir in valid_gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        lia_area = gdir.rgi_area_m2 / 1e6
        rgi_area = rgi_area_map.get(rgi_id, np.nan)

        ds_sp = None
        ds_fw = None
        try:
            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_sp)
            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_fw)
            if not os.path.exists(fp_sp) or not os.path.exists(fp_fw):
                continue

            ds_sp = xr.open_dataset(fp_sp)
            sp_end = float(ds_sp['area_m2'].values[-1]) / 1e6

            ds_fw = xr.open_dataset(fp_fw)
            fw_end = float(ds_fw['area_m2'].values[-1]) / 1e6

            results[rgi_id] = {
                'uid': uid,
                'lia_area': lia_area,
                'rgi_area': rgi_area,
                'sp_end': sp_end,
                'sp_lia_ratio': sp_end / lia_area,
                'fw_end': fw_end,
                'fw_rgi_ratio': fw_end / rgi_area if rgi_area > 0 else np.nan,
                'status': 'ok',
            }
        except:
            pass
        finally:
            if ds_sp is not None:
                ds_sp.close()
            if ds_fw is not None:
                ds_fw.close()

    total_time = (time.time() - t0) / 60
    n_ok = sum(1 for r in results.values() if r['status'] == 'ok')
    print("  结果: {}/{}, 总耗时: {:.1f}min".format(n_ok, len(valid_gdirs), total_time))

    # 恢复参数
    restore_original_params(valid_gdirs)
    gc.collect()

    return results


# ============================================================
# 主循环: 逐气候数据运行
# ============================================================
all_climate_results = {}
timing = {}

t_total = time.time()

for ci, climate_suffix in enumerate(ALL_SUFFIXES):
    tag = climate_suffix.strip('_')
    print("\n\n{'='*70}")
    print("[{}/{}] 气候数据: {}".format(ci + 1, len(ALL_SUFFIXES), tag))
    print("{'='*70}")

    t_this = time.time()
    results = run_one_climate(test_gdirs, climate_suffix)

    if results is not None:
        all_climate_results[tag] = results
        timing[tag] = (time.time() - t_this) / 60

        # checkpoint保存
        checkpoint = {
            'all_climate_results': all_climate_results,
            'timing': timing,
        }
        with open(os.path.join(SAVE_DIR, 'checkpoint.pkl'), 'wb') as f:
            pickle.dump(checkpoint, f)

    print("\n[{}/{}] {} 完成, 累计耗时: {:.1f}min".format(
        ci + 1, len(ALL_SUFFIXES), tag, (time.time() - t_total) / 60))

print("\n\n" + "=" * 70)
print("全部运行完成! 总耗时: {:.1f}min".format((time.time() - t_total) / 60))
print("=" * 70)

# ============================================================
# 汇总对比表
# ============================================================
print("\n" + "=" * 100)
print("各气候数据 fw/RGI ratio 汇总")
print("=" * 100)

# 加入已跑过的结果（如果在内存中）
if 'results_rec' in dir() and isinstance(results_rec, dict):
    all_climate_results['ensemble_rec_ref'] = results_rec
if 'results' in dir() and isinstance(results, dict):
    all_climate_results['Emean'] = results

# 表头
all_tags = sorted(all_climate_results.keys())
header = "{:<8}".format("UID")
for tag in all_tags:
    header += " {:>12}".format(tag[:12])
print(header)
print("-" * (8 + 13 * len(all_tags)))

# 逐冰川
all_rgi_ids = set()
for tag, res in all_climate_results.items():
    all_rgi_ids.update(res.keys())

for rgi_id in sorted(all_rgi_ids):
    uid = rgi_id.split('.')[-1]
    row = "{:<8}".format(uid)
    for tag in all_tags:
        res = all_climate_results.get(tag, {})
        r = res.get(rgi_id, {})
        ratio = r.get('fw_rgi_ratio', np.nan)
        if np.isnan(ratio):
            row += " {:>12}".format("-")
        else:
            row += " {:>12.3f}".format(ratio)
    print(row)

# 统计行
print("-" * (8 + 13 * len(all_tags)))
for stat_name, stat_func in [("mean", np.mean), ("median", np.median), ("std", np.std)]:
    row = "{:<8}".format(stat_name)
    for tag in all_tags:
        res = all_climate_results.get(tag, {})
        ratios = [r['fw_rgi_ratio'] for r in res.values()
                  if r.get('status') == 'ok' and not np.isnan(r.get('fw_rgi_ratio', np.nan))]
        if ratios:
            row += " {:>12.3f}".format(stat_func(ratios))
        else:
            row += " {:>12}".format("-")
    print(row)

# [0.8-1.2]比例
row = "{:<8}".format("[.8-1.2]")
for tag in all_tags:
    res = all_climate_results.get(tag, {})
    ratios = [r['fw_rgi_ratio'] for r in res.values()
              if r.get('status') == 'ok' and not np.isnan(r.get('fw_rgi_ratio', np.nan))]
    if ratios:
        n_good = sum(1 for r in ratios if 0.8 <= r <= 1.2)
        row += " {:>10}{:>2}".format("{}/{}".format(n_good, len(ratios)), "")
    else:
        row += " {:>12}".format("-")
print(row)

# ============================================================
# spinup衰减对比
# ============================================================
print("\n" + "=" * 100)
print("各气候数据 sp/LIA ratio 汇总 (spinup衰减程度)")
print("=" * 100)

header = "{:<8}".format("UID")
for tag in all_tags:
    header += " {:>12}".format(tag[:12])
print(header)
print("-" * (8 + 13 * len(all_tags)))

for rgi_id in sorted(all_rgi_ids):
    uid = rgi_id.split('.')[-1]
    row = "{:<8}".format(uid)
    for tag in all_tags:
        res = all_climate_results.get(tag, {})
        r = res.get(rgi_id, {})
        ratio = r.get('sp_lia_ratio', np.nan)
        if np.isnan(ratio):
            row += " {:>12}".format("-")
        else:
            row += " {:>12.3f}".format(ratio)
    print(row)

print("-" * (8 + 13 * len(all_tags)))
for stat_name, stat_func in [("mean", np.mean), ("median", np.median)]:
    row = "{:<8}".format(stat_name)
    for tag in all_tags:
        res = all_climate_results.get(tag, {})
        ratios = [r['sp_lia_ratio'] for r in res.values()
                  if r.get('status') == 'ok' and not np.isnan(r.get('sp_lia_ratio', np.nan))]
        if ratios:
            row += " {:>12.3f}".format(stat_func(ratios))
        else:
            row += " {:>12}".format("-")
    print(row)

# ============================================================
# 保存最终结果
# ============================================================
final_pkl = os.path.join(SAVE_DIR, 'all_climate_comparison.pkl')
with open(final_pkl, 'wb') as f:
    pickle.dump({
        'all_climate_results': all_climate_results,
        'timing': timing,
        'all_tags': all_tags,
        'config': {
            'Y0': Y0, 'HALFSIZE': HALFSIZE, 'SPINUP_YEARS': SPINUP_YEARS,
            'FORWARD_END': FORWARD_END, 'ALL_SUFFIXES': ALL_SUFFIXES,
        },
    }, f)
print("\nPickle: {}".format(final_pkl))

# CSV: 长格式
csv_rows = []
for tag, res in all_climate_results.items():
    for rgi_id, r in res.items():
        if r.get('status') != 'ok':
            continue
        csv_rows.append({
            'climate': tag,
            'rgi_id': rgi_id,
            'uid': r['uid'],
            'lia_area': r['lia_area'],
            'rgi_area': r['rgi_area'],
            'sp_end': r['sp_end'],
            'sp_lia_ratio': r['sp_lia_ratio'],
            'fw_end': r['fw_end'],
            'fw_rgi_ratio': r['fw_rgi_ratio'],
        })
csv_path = os.path.join(SAVE_DIR, 'all_climate_comparison.csv')
pd.DataFrame(csv_rows).to_csv(csv_path, index=False)
print("CSV: {}".format(csv_path))

# 耗时统计
print("\n各气候数据耗时:")
for tag, t in sorted(timing.items()):
    print("  {:<30}: {:.1f}min".format(tag, t))

print("\n全部完成!")

In [ ]:
"""
绘制所有气候数据的 spinup + 正演时间序列对比
=============================================
图1: 每条冰川子图，8套气候线叠加
图2: fw/RGI ratio 分组柱状图
图3: sp/LIA ratio 热力图（spinup衰减程度）
图4: fw/RGI ratio 热力图
图5: 归一化叠加（面积/RGI）

前置变量: test_gdirs, rgi_area_map, original_params, cfg
需要 all_climate_results 字典（从batch脚本或checkpoint.pkl加载）
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import pickle

# ============================================================
# 配置
# ============================================================
Y0 = 1500
SPINUP_YEARS = 300

# 所有气候数据的suffix和显示标签
# 根据你的实际结果调整
CLIMATE_CONFIGS = {
    'ensemble_rec_ref':    {'suffix_sp': '_sp_ensemble_rec_ref',    'suffix_fw': '_fw_ensemble_rec_ref',    'label': 'Rec_ref',     'color': '#2166ac'},
    'Emean':               {'suffix_sp': '_gcm_emean_sp',           'suffix_fw': '_gcm_emean_fw',           'label': 'GCM Emean',   'color': '#b2182b'},
    'ensemble_6':          {'suffix_sp': '_sp_ensemble_6',          'suffix_fw': '_fw_ensemble_6',          'label': 'Ens_6',       'color': '#1b7837'},
    'ensemble_9':          {'suffix_sp': '_sp_ensemble_9',          'suffix_fw': '_fw_ensemble_9',          'label': 'Ens_9',       'color': '#762a83'},
    'ensemble_all_methods':{'suffix_sp': '_sp_ensemble_all_methods','suffix_fw': '_fw_ensemble_all_methods','label': 'Ens_all',     'color': '#e66101'},
    'ensemble_my_methods': {'suffix_sp': '_sp_ensemble_my_methods', 'suffix_fw': '_fw_ensemble_my_methods', 'label': 'Ens_my',      'color': '#5e3c99'},
    'ensemble_rec_result': {'suffix_sp': '_sp_ensemble_rec_result', 'suffix_fw': '_fw_ensemble_rec_result', 'label': 'Rec_result',  'color': '#d95f02'},
    'rec_test':            {'suffix_sp': '_sp_rec_test',            'suffix_fw': '_fw_rec_test',            'label': 'Rec_test',    'color': '#66a61e'},
}

# 如果之前的 rec_ref 和 Emean 用的是不同的suffix命名，在这里修正
# 例如之前rec_ref可能是 '_rec_ref_sp' / '_rec_ref_fw'
# Emean可能是 '_gcm_emean_sp' / '_gcm_emean_fw'
# 请根据实际文件名调整上面的suffix

# ============================================================
# 收集数据
# ============================================================
print("收集所有气候数据的时间序列...")

all_ts = {}  # {climate_tag: {rgi_id: {...}}}

for tag, cfg_item in CLIMATE_CONFIGS.items():
    all_ts[tag] = {}
    n_ok = 0

    for gdir in test_gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        lia_area = gdir.rgi_area_m2 / 1e6
        rgi_area = rgi_area_map.get(rgi_id, np.nan)

        ds_sp = None
        ds_fw = None
        try:
            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=cfg_item['suffix_sp'])
            if not os.path.exists(fp_sp):
                continue

            ds_sp = xr.open_dataset(fp_sp)
            area_sp = ds_sp['area_m2'].values / 1e6
            n_sp = len(area_sp)
            years_sp = np.linspace(Y0 - SPINUP_YEARS, Y0, n_sp)

            entry = {
                'uid': uid, 'rgi_id': rgi_id,
                'lia_area': lia_area, 'rgi_area': rgi_area,
                'years_sp': years_sp, 'area_sp': area_sp,
                'sp_end': area_sp[-1],
                'sp_lia_ratio': area_sp[-1] / lia_area,
                'has_forward': False,
            }

            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=cfg_item['suffix_fw'])
            if os.path.exists(fp_fw):
                ds_fw = xr.open_dataset(fp_fw)
                area_fw = ds_fw['area_m2'].values / 1e6
                n_fw = len(area_fw)
                years_fw = np.arange(Y0, Y0 + n_fw)
                entry.update({
                    'years_fw': years_fw, 'area_fw': area_fw,
                    'fw_end': area_fw[-1],
                    'fw_rgi_ratio': area_fw[-1] / rgi_area if rgi_area > 0 else np.nan,
                    'has_forward': True,
                })

            all_ts[tag][rgi_id] = entry
            n_ok += 1

        except:
            pass
        finally:
            if ds_sp is not None:
                ds_sp.close()
            if ds_fw is not None:
                ds_fw.close()

    print("  {:<25}: {} 条冰川".format(tag, n_ok))

# 找所有冰川ID（至少在一套数据中有结果）
all_rgi_ids = set()
for tag, ts in all_ts.items():
    all_rgi_ids.update(ts.keys())
all_rgi_ids = sorted(all_rgi_ids)
print("总冰川数: {}".format(len(all_rgi_ids)))

# 有效的气候标签（至少有1条冰川）
valid_tags = [tag for tag in CLIMATE_CONFIGS if len(all_ts.get(tag, {})) > 0]
print("有效气候数据: {}".format(valid_tags))

# ============================================================
# 图1: 每条冰川子图，所有气候线叠加
# ============================================================
print("\n绑制图1: 每条冰川多气候叠加...")

n = len(all_rgi_ids)
ncols = min(5, n)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
if nrows * ncols == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, rgi_id in enumerate(all_rgi_ids):
    ax = axes[idx]

    # 从任一数据集取LIA和RGI面积
    ref_entry = None
    for tag in valid_tags:
        if rgi_id in all_ts[tag]:
            ref_entry = all_ts[tag][rgi_id]
            break
    if ref_entry is None:
        continue

    uid = ref_entry['uid']
    lia_area = ref_entry['lia_area']
    rgi_area = ref_entry['rgi_area']

    # 绘制每套气候
    ratio_texts = []
    for tag in valid_tags:
        if rgi_id not in all_ts[tag]:
            continue
        d = all_ts[tag][rgi_id]
        color = CLIMATE_CONFIGS[tag]['color']
        label = CLIMATE_CONFIGS[tag]['label']

        # spinup (淡色)
        ax.plot(d['years_sp'], d['area_sp'],
                color=color, linewidth=0.8, alpha=0.3)
        # forward (实色)
        if d['has_forward']:
            ax.plot(d['years_fw'], d['area_fw'],
                    color=color, linewidth=1.5, alpha=0.8, label=label)
            ratio_texts.append('{:.2f}'.format(d['fw_rgi_ratio']))

    # 参考线
    ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.6, alpha=0.3)
    ax.axhline(lia_area, color='#1b7837', linestyle=':', linewidth=1.2, alpha=0.6)
    ax.axhline(rgi_area, color='#e66101', linestyle='--', linewidth=1.5, alpha=0.8)

    ax.set_title(uid, fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.15)

    if idx == 0:
        ax.legend(fontsize=5, loc='best', ncol=2)

for i in range(n, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('All Climate Datasets | Spinup {}yr + Forward → {}\n'
             'Green: LIA outline | Orange: RGI observed'.format(SPINUP_YEARS, Y0 + 500),
             fontsize=13, fontweight='bold')
fig.supxlabel('Year', fontsize=11)
fig.supylabel('Area (km²)', fontsize=11)
plt.tight_layout()
save_path = os.path.join(cfg.PATHS['working_dir'], 'multi_climate_results',
                         'ts_all_climates_individual.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print("图1已保存: {}".format(save_path))

# ============================================================
# 图2: fw/RGI ratio 分组柱状图
# ============================================================
print("\n绑制图2: fw/RGI ratio柱状图...")

# 收集有forward结果的冰川
fw_uids = []
fw_data = {}  # {tag: [ratios in uid order]}

for rgi_id in all_rgi_ids:
    uid = rgi_id.split('.')[-1]
    has_any = False
    for tag in valid_tags:
        d = all_ts.get(tag, {}).get(rgi_id, {})
        if d.get('has_forward', False):
            has_any = True
            break
    if has_any:
        fw_uids.append(uid)
        for tag in valid_tags:
            d = all_ts.get(tag, {}).get(rgi_id, {})
            fw_data.setdefault(tag, []).append(
                d.get('fw_rgi_ratio', np.nan) if d.get('has_forward', False) else np.nan)

if fw_uids:
    n_tags = len(valid_tags)
    x = np.arange(len(fw_uids))
    width = 0.8 / n_tags

    fig, ax = plt.subplots(figsize=(max(14, len(fw_uids) * 1.0), 7))

    for ti, tag in enumerate(valid_tags):
        ratios = fw_data.get(tag, [np.nan] * len(fw_uids))
        color = CLIMATE_CONFIGS[tag]['color']
        label = CLIMATE_CONFIGS[tag]['label']
        offset = (ti - n_tags / 2 + 0.5) * width
        bars = ax.bar(x + offset, ratios, width * 0.9,
                       color=color, alpha=0.75, edgecolor='black',
                       linewidth=0.3, label=label)

    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5)
    ax.fill_between([-0.5, len(fw_uids) - 0.5], 0.8, 1.2,
                    alpha=0.08, color='green')

    ax.set_xticks(x)
    ax.set_xticklabels(fw_uids, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('sim_2000 / RGI area', fontsize=11)
    ax.set_xlabel('Glacier UID', fontsize=11)
    ax.set_title('fw/RGI ratio: All Climate Datasets', fontsize=13, fontweight='bold')
    ax.legend(fontsize=7, ncol=4, loc='upper left')
    ax.grid(True, alpha=0.15, axis='y')

    plt.tight_layout()
    save_path = os.path.join(cfg.PATHS['working_dir'], 'multi_climate_results',
                             'ratio_bar_all_climates.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("图2已保存: {}".format(save_path))

# ============================================================
# 图3: 热力图 - fw/RGI ratio
# ============================================================
print("\n绑制图3: fw/RGI ratio热力图...")

uids_for_heatmap = []
ratio_matrix = []

for rgi_id in all_rgi_ids:
    uid = rgi_id.split('.')[-1]
    row = []
    has_data = False
    for tag in valid_tags:
        d = all_ts.get(tag, {}).get(rgi_id, {})
        r = d.get('fw_rgi_ratio', np.nan) if d.get('has_forward', False) else np.nan
        row.append(r)
        if not np.isnan(r):
            has_data = True
    if has_data:
        uids_for_heatmap.append(uid)
        ratio_matrix.append(row)

if ratio_matrix:
    ratio_arr = np.array(ratio_matrix)
    labels_col = [CLIMATE_CONFIGS[t]['label'] for t in valid_tags]

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(uids_for_heatmap) * 0.4)),
                              gridspec_kw={'width_ratios': [3, 3]})

    # 左: fw/RGI
    ax = axes[0]
    im = ax.imshow(ratio_arr, aspect='auto', cmap='RdYlBu_r',
                    vmin=0.3, vmax=2.0)
    ax.set_xticks(range(len(labels_col)))
    ax.set_xticklabels(labels_col, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(uids_for_heatmap)))
    ax.set_yticklabels(uids_for_heatmap, fontsize=8)
    ax.set_title('fw/RGI ratio (closer to 1.0 = better)', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # 数值标注
    for i in range(len(uids_for_heatmap)):
        for j in range(len(valid_tags)):
            val = ratio_arr[i, j]
            if not np.isnan(val):
                color = 'white' if val > 1.5 or val < 0.5 else 'black'
                ax.text(j, i, '{:.2f}'.format(val), ha='center', va='center',
                        fontsize=6, color=color)

    # 右: sp/LIA
    sp_matrix = []
    for rgi_id in all_rgi_ids:
        uid = rgi_id.split('.')[-1]
        if uid not in uids_for_heatmap:
            continue
        row = []
        for tag in valid_tags:
            d = all_ts.get(tag, {}).get(rgi_id, {})
            r = d.get('sp_lia_ratio', np.nan)
            row.append(r)
        sp_matrix.append(row)

    sp_arr = np.array(sp_matrix)
    ax = axes[1]
    im2 = ax.imshow(sp_arr, aspect='auto', cmap='RdYlBu_r',
                     vmin=0.2, vmax=1.0)
    ax.set_xticks(range(len(labels_col)))
    ax.set_xticklabels(labels_col, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(uids_for_heatmap)))
    ax.set_yticklabels(uids_for_heatmap, fontsize=8)
    ax.set_title('sp/LIA ratio (spinup decay, closer to 1.0 = less decay)', fontsize=11)
    plt.colorbar(im2, ax=ax, shrink=0.8)

    for i in range(len(uids_for_heatmap)):
        for j in range(len(valid_tags)):
            val = sp_arr[i, j]
            if not np.isnan(val):
                color = 'white' if val < 0.4 else 'black'
                ax.text(j, i, '{:.2f}'.format(val), ha='center', va='center',
                        fontsize=6, color=color)

    fig.suptitle('All Climate Datasets Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_path = os.path.join(cfg.PATHS['working_dir'], 'multi_climate_results',
                             'heatmap_all_climates.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("图3已保存: {}".format(save_path))

# ============================================================
# 图4: 各气候数据的median ratio对比
# ============================================================
print("\n绑制图4: 各气候数据汇总统计...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: fw/RGI median + IQR
ax = axes[0]
tags_plot = []
medians_fw = []
q25_fw = []
q75_fw = []

for tag in valid_tags:
    ratios = [all_ts[tag][rid]['fw_rgi_ratio']
              for rid in all_rgi_ids
              if rid in all_ts[tag] and all_ts[tag][rid].get('has_forward', False)
              and not np.isnan(all_ts[tag][rid].get('fw_rgi_ratio', np.nan))]
    if ratios:
        tags_plot.append(CLIMATE_CONFIGS[tag]['label'])
        medians_fw.append(np.median(ratios))
        q25_fw.append(np.percentile(ratios, 25))
        q75_fw.append(np.percentile(ratios, 75))

x = np.arange(len(tags_plot))
colors = [CLIMATE_CONFIGS[t]['color'] for t in valid_tags if CLIMATE_CONFIGS[t]['label'] in tags_plot]

ax.bar(x, medians_fw, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.errorbar(x, medians_fw,
            yerr=[np.array(medians_fw) - np.array(q25_fw),
                  np.array(q75_fw) - np.array(medians_fw)],
            fmt='none', color='black', capsize=4)
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(tags_plot, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('fw/RGI ratio')
ax.set_title('Forward end / RGI (median ± IQR)')
ax.grid(True, alpha=0.2, axis='y')

for i, (m, t) in enumerate(zip(medians_fw, tags_plot)):
    ax.text(i, m + 0.03, '{:.3f}'.format(m), ha='center', va='bottom', fontsize=8)

# 右: sp/LIA median + IQR
ax = axes[1]
tags_plot2 = []
medians_sp = []
q25_sp = []
q75_sp = []

for tag in valid_tags:
    ratios = [all_ts[tag][rid]['sp_lia_ratio']
              for rid in all_rgi_ids
              if rid in all_ts[tag]
              and not np.isnan(all_ts[tag][rid].get('sp_lia_ratio', np.nan))]
    if ratios:
        tags_plot2.append(CLIMATE_CONFIGS[tag]['label'])
        medians_sp.append(np.median(ratios))
        q25_sp.append(np.percentile(ratios, 25))
        q75_sp.append(np.percentile(ratios, 75))

x2 = np.arange(len(tags_plot2))
colors2 = [CLIMATE_CONFIGS[t]['color'] for t in valid_tags if CLIMATE_CONFIGS[t]['label'] in tags_plot2]

ax.bar(x2, medians_sp, color=colors2, alpha=0.7, edgecolor='black', linewidth=0.5)
ax.errorbar(x2, medians_sp,
            yerr=[np.array(medians_sp) - np.array(q25_sp),
                  np.array(q75_sp) - np.array(medians_sp)],
            fmt='none', color='black', capsize=4)
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5)
ax.set_xticks(x2)
ax.set_xticklabels(tags_plot2, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('sp/LIA ratio')
ax.set_title('Spinup end / LIA (median ± IQR)')
ax.grid(True, alpha=0.2, axis='y')

for i, (m, t) in enumerate(zip(medians_sp, tags_plot2)):
    ax.text(i, m + 0.02, '{:.3f}'.format(m), ha='center', va='bottom', fontsize=8)

fig.suptitle('Climate Dataset Comparison Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(cfg.PATHS['working_dir'], 'multi_climate_results',
                         'summary_stats_all_climates.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print("图4已保存: {}".format(save_path))

# ============================================================
# 汇总表
# ============================================================
print("\n" + "=" * 100)
print("汇总统计")
print("=" * 100)
print("{:<15} {:>10} {:>10} {:>10} {:>10} {:>10} {:>8}".format(
    "Climate", "fw_mean", "fw_med", "fw_std", "sp_med", "[.8-1.2]", "N"))
print("-" * 75)

for tag in valid_tags:
    label = CLIMATE_CONFIGS[tag]['label']
    fw_ratios = [all_ts[tag][rid]['fw_rgi_ratio']
                 for rid in all_rgi_ids
                 if rid in all_ts[tag] and all_ts[tag][rid].get('has_forward', False)
                 and not np.isnan(all_ts[tag][rid].get('fw_rgi_ratio', np.nan))]
    sp_ratios = [all_ts[tag][rid]['sp_lia_ratio']
                 for rid in all_rgi_ids
                 if rid in all_ts[tag]
                 and not np.isnan(all_ts[tag][rid].get('sp_lia_ratio', np.nan))]

    if fw_ratios:
        n_good = sum(1 for r in fw_ratios if 0.8 <= r <= 1.2)
        print("{:<15} {:>10.3f} {:>10.3f} {:>10.3f} {:>10.3f} {:>8}/{:<2} {:>6}".format(
            label,
            np.mean(fw_ratios), np.median(fw_ratios), np.std(fw_ratios),
            np.median(sp_ratios) if sp_ratios else np.nan,
            n_good, len(fw_ratios), len(fw_ratios)))
    else:
        print("{:<15} {:>60}".format(label, "无数据"))

print("\n绘图完成!")

In [ ]:
"""
逆演化 + 正演验证 完整框架
===============================
从现代RGI冰川出发，施加温度偏差spinup让冰川增长到LIA范围，
然后用重建气候数据正演到2000年，验证能否回到RGI面积。

框架结构：
  Step 0: 环境准备 + 数据加载
  Step 1: 现代冰川基线（gdirs_rgi已校准，直接加载）
  Step 2: 处理重建气候到RGI冰川的gdir中
  Step 3: 逆演化Spinup — 多档位温度偏差，从现代增长到LIA
  Step 4: 约束1检查 — spinup面积 vs 遥感LIA面积，找best_bias
  Step 5: 正演 — 从best_bias的spinup状态出发，用重建气候驱动到2000年
  Step 6: 约束2检查 — 正演终点面积 vs RGI面积
  Step 7: 综合分析与可视化

运行环境：Jupyter Notebook，逐Cell执行
前提条件：
  - gdirs_rgi 已完成标准OGGM流程（预处理+CRU校准+冰厚反演）
  - 重建气候数据已生成（gcm_data_ensemble_rec_ref.nc 等）
  - lia_to_main_rgi 映射已建立
"""

# %% Cell 0 — 导入与配置
# ============================================================

import json, os, gc, time, warnings
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

from oggm import cfg, workflow, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

warnings.filterwarnings('ignore')

class MyMBModel(MonthlyTIModel):
    """跳过校准参数检查的MB模型"""
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ============================================================
# ★★★ 用户配置区 ★★★  — 根据你的实际路径修改
# ============================================================

# --- 路径配置 ---
RGI_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"

# 重建气候数据所在目录（之前已经为LIA gdirs生成过的）
# 需要包含 gcm_data_ensemble_rec_ref.nc 等文件
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"

# 输出目录
OUTPUT_DIR = "/mnt/f/oggm_related/Rec_climate/inverse_evolution_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 实验参数 ---
# 温度偏差扫描范围（相对于现代CRU气候态的降温值）
TEMP_BIASES = [-1.5, -1.25, -1.0, -0.75, -0.5, -0.25, 0.0, 0.25]

# Spinup年数（需要足够长让冰川达到稳态）
SPINUP_YEARS = 600

# Spinup气候基准（CRU参考期中心年和半宽）
# 使用现代CRU气候态，y0=1985, halfsize=15 → 1970-2000平均气候
SPINUP_Y0 = 1985
SPINUP_HALFSIZE = 15

# 正演起始年（应匹配重建气候数据的覆盖范围）
FORWARD_START_YEAR = 1400
FORWARD_END_YEAR = 2000

# 重建气候文件后缀
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# LIA面积匹配容差（spinup面积/LIA面积在此范围内视为匹配）
LIA_MATCH_TOL = (0.85, 1.15)

# 正演ratio容差
RGI_MATCH_TOL = (0.80, 1.20)

# 测试冰川数量（None=全量）
TEST_N = 20

# --- 多进程配置 ---
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True

print("=" * 70)
print("逆演化 + 正演验证框架")
print("=" * 70)
print("温度偏差范围: {} 到 {} K ({} 档)".format(
    TEMP_BIASES[0], TEMP_BIASES[-1], len(TEMP_BIASES)))
print("Spinup: {} 年, 气候基准 y0={} ±{}".format(
    SPINUP_YEARS, SPINUP_Y0, SPINUP_HALFSIZE))
print("正演: {} → {} 年".format(FORWARD_START_YEAR, FORWARD_END_YEAR))
print("输出目录: {}".format(OUTPUT_DIR))
print("=" * 70)


# %% Cell 1 — 加载已校准的RGI冰川
# ============================================================
# 前提: gdirs_rgi 已经在之前的notebook中完成了:
#   - workflow.init_glacier_directories
#   - tasks.process_cru_data
#   - mb_calibration_from_scalar_mb (Hugonnet 2000-2020)
#   - mass_conservation_inversion
#   - init_present_time_glacier
#
# 这里直接加载，不重新跑
# ============================================================

print("\n[Step 1] 加载已校准的RGI冰川...")

cfg.PATHS['working_dir'] = RGI_WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 80
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['baseline_climate'] = 'CRU'

# ★ 需要 rgi_to_calibrate（已在前置notebook定义）
# 如果不在环境中，取消下面注释重建:
# import geopandas as gpd
# rgi = gpd.read_file("你的RGI路径")
# lia = gpd.read_file("你的LIA路径")
# ... (筛选逻辑)

gdirs_rgi = workflow.init_glacier_directories(rgi_to_calibrate, reset=False)
print("  加载完成: {} 个RGI冰川".format(len(gdirs_rgi)))

# 验证: 检查是否已有 model_flowlines
n_ready = 0
for gdir in gdirs_rgi:
    try:
        fls = gdir.read_pickle('model_flowlines')
        n_ready += 1
    except:
        pass
print("  已有model_flowlines: {}/{}".format(n_ready, len(gdirs_rgi)))

if n_ready < len(gdirs_rgi) * 0.8:
    print("  ⚠️ 警告: 大量冰川缺少model_flowlines，请先完成标准OGGM流程")


# %% Cell 2 — 构建映射关系 + 筛选有效冰川
# ============================================================

print("\n[Step 2] 构建映射关系...")

# --- 2.1 RGI面积映射: MainRGI ID → RGI面积(km2) ---
rgi_area_by_id = {}
for gdir in gdirs_rgi:
    rgi_area_by_id[gdir.rgi_id] = gdir.rgi_area_m2 / 1e6

# --- 2.2 LIA面积映射: MainRGI ID → LIA面积(km2) ---
# 从lia_matched获取（需要在环境中）
lia_area_by_mainrgi = {}
for _, row in lia_matched.iterrows():
    uid = int(row['UniqueID'])
    main_rgi = row['MainRGI']
    lia_area = row['Area']  # LIA轮廓面积 (km2)
    lia_area_by_mainrgi[main_rgi] = lia_area

# --- 2.3 反向映射: MainRGI → UniqueID ---
mainrgi_to_uid = {v: k for k, v in lia_to_main_rgi.items()}

# --- 2.4 筛选有效冰川 ---
# 条件: (1) 有model_flowlines (2) 有LIA面积映射 (3) 有mb_calib
valid_gdirs = []
for gdir in gdirs_rgi:
    rgi_id = gdir.rgi_id
    try:
        fls = gdir.read_pickle('model_flowlines')
        calib = gdir.read_json('mb_calib')
        if rgi_id in lia_area_by_mainrgi:
            valid_gdirs.append(gdir)
    except:
        pass

if TEST_N is not None:
    valid_gdirs = valid_gdirs[:TEST_N]
    print("  >>> 测试模式: 取前 {} 条 <<<".format(TEST_N))

print("  有效冰川: {} 条".format(len(valid_gdirs)))

# 打印一些统计
lia_areas = [lia_area_by_mainrgi[g.rgi_id] for g in valid_gdirs]
rgi_areas = [rgi_area_by_id[g.rgi_id] for g in valid_gdirs]
ratios = [l / r for l, r in zip(lia_areas, rgi_areas)]
print("  LIA/RGI面积比: 均值={:.2f}, 范围=[{:.2f}, {:.2f}]".format(
    np.mean(ratios), np.min(ratios), np.max(ratios)))


# %% Cell 3 — 将重建气候数据复制到RGI冰川的gdir中
# ============================================================
# 关键步骤: RGI的gdir中只有CRU气候，需要把重建气候也放进去
# 重建气候之前是为LIA gdirs生成的，现在需要为RGI gdirs生成
#
# 两种策略:
#   A) 从LIA gdir中复制gcm_data文件（如果RGI和LIA共享同一网格点）
#   B) 重新为RGI gdirs生成重建气候（更稳妥）
#
# 这里用策略A: 从对应的LIA gdir复制
# ============================================================

print("\n[Step 3] 将重建气候数据复制到RGI gdirs...")

n_copied = 0
n_failed = 0

for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    uid = mainrgi_to_uid.get(rgi_id)
    if uid is None:
        n_failed += 1
        continue

    # LIA gdir的路径
    lia_rgi_id = "RGI60-15.{:05d}".format(uid)
    lia_gdir_path = os.path.join(LIA_WORKING_DIR, lia_rgi_id[:8], lia_rgi_id[:11], lia_rgi_id)

    # 源文件: LIA gdir中的重建气候
    src_file = os.path.join(lia_gdir_path, "gcm_data{}.nc".format(CLIMATE_SUFFIX))

    if not os.path.exists(src_file):
        n_failed += 1
        continue

    # 目标文件: RGI gdir中
    dst_file = gdir.get_filepath('gcm_data', filesuffix=CLIMATE_SUFFIX)

    try:
        import shutil
        shutil.copy2(src_file, dst_file)
        n_copied += 1
    except Exception as e:
        print("  复制失败 {}: {}".format(rgi_id, str(e)[:50]))
        n_failed += 1

print("  成功复制: {}, 失败: {}".format(n_copied, n_failed))

# 更新valid_gdirs: 只保留有重建气候数据的
valid_gdirs_final = []
for gdir in valid_gdirs:
    fp = gdir.get_filepath('gcm_data', filesuffix=CLIMATE_SUFFIX)
    if os.path.exists(fp):
        valid_gdirs_final.append(gdir)

valid_gdirs = valid_gdirs_final
print("  最终有效冰川: {} 条".format(len(valid_gdirs)))


# %% Cell 4 — 逆演化 Spinup: 多档位温度偏差
# ============================================================
# 从现代RGI冰川出发，施加不同温度偏差（降温），
# 用run_constant_climate做600年spinup，让冰川增长
#
# 注意: 这里用的是CRU现代气候 + temp_bias偏移
# 物理含义: "LIA时期比1970-2000平均气候冷X度"
# ============================================================

print("\n[Step 4] 逆演化 Spinup ({} 年, {} 档温度偏差)...".format(
    SPINUP_YEARS, len(TEMP_BIASES)))

# 保存原始参数（用于恢复）
original_params = {}
for gdir in valid_gdirs:
    try:
        params = gdir.read_json('mb_calib')
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
        }
    except:
        pass

def batch_set_temp_bias(gdirs, offset):
    """给所有冰川的temp_bias加上offset"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        if rgi_id not in original_params:
            continue
        try:
            params = gdir.read_json('mb_calib')
            params['temp_bias'] = original_params[rgi_id]['temp_bias'] + offset
            gdir.write_json(params, 'mb_calib')
        except:
            pass

def batch_restore_params(gdirs):
    """恢复所有冰川的原始参数"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        if rgi_id not in original_params:
            continue
        try:
            params = gdir.read_json('mb_calib')
            params['temp_bias'] = original_params[rgi_id]['temp_bias']
            params['melt_f'] = original_params[rgi_id]['melt_f']
            params['prcp_fac'] = original_params[rgi_id]['prcp_fac']
            gdir.write_json(params, 'mb_calib')
        except:
            pass

# --- 逐档位运行spinup ---
spinup_results = {}  # {bias_value: {rgi_id: {area, volume, length}}}

for bias in TEMP_BIASES:
    suffix = '_inv_spinup_bias{}'.format(
        str(bias).replace('.', 'p').replace('-', 'n'))

    print("\n  --- Temp bias = {} K ---".format(bias))
    t0 = time.time()

    # 设置偏移后的temp_bias
    batch_set_temp_bias(valid_gdirs, bias)

    # 运行spinup
    try:
        execute_entity_task(
            tasks.run_constant_climate,
            valid_gdirs,
            nyears=SPINUP_YEARS,
            y0=SPINUP_Y0,
            halfsize=SPINUP_HALFSIZE,
            # 使用CRU气候（默认climate_historical），不是重建气候
            output_filesuffix=suffix,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
        )
    except Exception as e:
        print("    批量异常: {}".format(str(e)[:100]))

    # 恢复原始参数
    batch_restore_params(valid_gdirs)

    # 提取结果
    bias_results = {}
    n_ok = 0
    for gdir in valid_gdirs:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            with xr.open_dataset(fp) as ds:
                # 取最后10年平均（检查是否稳态）
                area_end = float(ds.area_m2.isel(time=slice(-10, None)).mean()) / 1e6
                vol_end = float(ds.volume_m3.isel(time=slice(-10, None)).mean()) / 1e9
                length_end = float(ds.length_m.isel(time=slice(-10, None)).mean()) / 1e3
                # 取最后50年的面积变化率（判断稳态）
                if len(ds.time) > 50:
                    a_late = float(ds.area_m2.isel(time=slice(-10, None)).mean())
                    a_mid = float(ds.area_m2.isel(time=slice(-60, -50)).mean())
                    area_change_pct = (a_late - a_mid) / a_mid * 100
                else:
                    area_change_pct = np.nan

            bias_results[gdir.rgi_id] = {
                'area_km2': area_end,
                'volume_km3': vol_end,
                'length_km': length_end,
                'area_change_pct': area_change_pct,
            }
            n_ok += 1
        except:
            pass

    spinup_results[bias] = bias_results
    dt = time.time() - t0
    print("    成功: {}/{}, 耗时: {:.1f} min".format(n_ok, len(valid_gdirs), dt / 60))

print("\n  所有spinup完成!")

# 保存中间结果
import pickle
with open(os.path.join(OUTPUT_DIR, 'spinup_results.pkl'), 'wb') as f:
    pickle.dump(spinup_results, f)
print("  结果已保存: spinup_results.pkl")


# %% Cell 5 — 约束1: Spinup面积 vs LIA遥感面积，找best_bias
# ============================================================

print("\n[Step 5] 约束1分析: Spinup面积 vs LIA遥感面积")
print("=" * 90)

# 构建分析表
rows = []
for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id
    rgi_area = rgi_area_by_id.get(rgi_id, np.nan)
    lia_area = lia_area_by_mainrgi.get(rgi_id, np.nan)

    row = {
        'rgi_id': rgi_id,
        'rgi_area_km2': rgi_area,
        'lia_area_km2': lia_area,
        'lia_rgi_ratio': lia_area / rgi_area if rgi_area > 0 else np.nan,
    }

    # 每个bias档位的spinup面积
    best_bias = None
    best_ratio_diff = 999

    for bias in TEMP_BIASES:
        res = spinup_results.get(bias, {}).get(rgi_id, None)
        if res is None:
            row['sp_area_bias{}'.format(bias)] = np.nan
            row['sp_lia_ratio_bias{}'.format(bias)] = np.nan
            continue

        sp_area = res['area_km2']
        sp_lia_ratio = sp_area / lia_area if lia_area > 0 else np.nan
        row['sp_area_bias{}'.format(bias)] = sp_area
        row['sp_lia_ratio_bias{}'.format(bias)] = sp_lia_ratio

        # 找最接近LIA面积的bias
        if not np.isnan(sp_lia_ratio):
            diff = abs(sp_lia_ratio - 1.0)
            if diff < best_ratio_diff:
                best_ratio_diff = diff
                best_bias = bias

    row['best_bias'] = best_bias
    row['best_sp_area'] = spinup_results.get(best_bias, {}).get(rgi_id, {}).get('area_km2', np.nan) if best_bias is not None else np.nan
    row['best_sp_lia_ratio'] = row['best_sp_area'] / lia_area if best_bias is not None and lia_area > 0 else np.nan

    rows.append(row)

df_constraint1 = pd.DataFrame(rows)

# 打印总结
print("\n逐冰川最优温度偏差:")
print("-" * 90)
print("{:18s} {:>8s} {:>8s} {:>8s} {:>10s} {:>10s} {:>8s}".format(
    'RGI_ID', 'RGI_km2', 'LIA_km2', 'LIA/RGI', 'best_bias', 'SP_km2', 'SP/LIA'))
print("-" * 90)

for _, r in df_constraint1.iterrows():
    if r['best_bias'] is not None:
        print("{:18s} {:8.2f} {:8.2f} {:8.2f} {:10.2f} {:10.2f} {:8.3f}".format(
            r['rgi_id'], r['rgi_area_km2'], r['lia_area_km2'], r['lia_rgi_ratio'],
            r['best_bias'], r['best_sp_area'], r['best_sp_lia_ratio']))

# 统计
valid_best = df_constraint1.dropna(subset=['best_bias'])
print("\n" + "=" * 90)
print("约束1统计:")
print("  有效冰川: {} / {}".format(len(valid_best), len(df_constraint1)))
print("  best_bias 分布: 均值={:.2f}, 中位数={:.2f}, std={:.2f}".format(
    valid_best['best_bias'].mean(),
    valid_best['best_bias'].median(),
    valid_best['best_bias'].std()))
print("  best_bias 范围: [{:.2f}, {:.2f}]".format(
    valid_best['best_bias'].min(), valid_best['best_bias'].max()))

# SP/LIA ratio
in_tol = valid_best[
    (valid_best['best_sp_lia_ratio'] >= LIA_MATCH_TOL[0]) &
    (valid_best['best_sp_lia_ratio'] <= LIA_MATCH_TOL[1])
]
print("  SP/LIA ratio 在 {} 内: {}/{} = {:.1f}%".format(
    LIA_MATCH_TOL, len(in_tol), len(valid_best),
    100 * len(in_tol) / max(len(valid_best), 1)))

# 保存
df_constraint1.to_csv(os.path.join(OUTPUT_DIR, 'constraint1_analysis.csv'), index=False)
print("  已保存: constraint1_analysis.csv")


# %% Cell 6 — 确定正演起始年（温度对接）
# ============================================================
# 关键: spinup用的是现代气候-X度 (恒定)
# 正演切换到重建气候时，需要温度连续
# → 找重建气候中温度恰好比现代低X度的年份作为正演起始年
# ============================================================

print("\n[Step 6] 确定正演起始年（温度对接）...")

# 读取一个冰川的重建气候，分析温度时间序列
sample_gdir = valid_gdirs[0]
fp_gcm = sample_gdir.get_filepath('gcm_data', filesuffix=CLIMATE_SUFFIX)
with xr.open_dataset(fp_gcm) as ds_gcm:
    temp_annual = ds_gcm.temp.groupby('time.year').mean()
    years = temp_annual.year.values
    temps = temp_annual.values

# 读取CRU现代参考温度（spinup基准）
fp_cru = sample_gdir.get_filepath('climate_historical')
with xr.open_dataset(fp_cru) as ds_cru:
    # 1970-2000平均温度（匹配spinup的y0=1985, halfsize=15）
    cru_ref_temp = float(ds_cru.temp.sel(
        time=slice('{}-01'.format(SPINUP_Y0 - SPINUP_HALFSIZE),
                   '{}-12'.format(SPINUP_Y0 + SPINUP_HALFSIZE))
    ).mean())

print("  CRU参考温度 ({}-{}): {:.2f} K".format(
    SPINUP_Y0 - SPINUP_HALFSIZE, SPINUP_Y0 + SPINUP_HALFSIZE, cru_ref_temp))

# 对每个best_bias，找重建气候中对应的起始年
# 温度对接: 重建气候中30年滑动平均温度 ≈ CRU参考温度 + bias
temp_30yr = pd.Series(temps, index=years).rolling(30, center=True).mean()

print("\n  各bias对应的温度对接点:")
print("  {:>8s} {:>12s} {:>12s} {:>12s}".format(
    'bias(K)', 'target_T(K)', 'match_year', 'actual_T(K)'))
print("  " + "-" * 50)

bias_start_year = {}
for bias in TEMP_BIASES:
    target_temp = cru_ref_temp + bias  # spinup等效温度
    # 找重建气候中最接近target_temp的年份（排除首尾NaN）
    valid_mask = temp_30yr.notna()
    if valid_mask.sum() == 0:
        continue
    diffs = abs(temp_30yr[valid_mask] - target_temp)
    match_year = int(diffs.idxmin())
    actual_temp = float(temp_30yr[match_year])
    bias_start_year[bias] = match_year
    print("  {:8.2f} {:12.2f} {:12d} {:12.2f}".format(
        bias, target_temp, match_year, actual_temp))

print("\n  ⚠️ 注意: 如果target_T和actual_T差异大，说明重建气候无法完全匹配spinup气候")
print("         这种情况下正演起始会有温度跳变，但会比之前的方案小得多")


# %% Cell 7 — 正演: 从spinup状态出发，用重建气候驱动到2000年
# ============================================================
# 对每条冰川:
#   1. 使用其best_bias对应的spinup终态作为初始条件
#   2. 从温度对接的起始年开始正演
#   3. 用重建气候驱动到2000年
#
# 正演阶段: 恢复原始参数，不加任何人为偏差
# ============================================================

print("\n[Step 7] 正演: 重建气候驱动到 {} 年...".format(FORWARD_END_YEAR))

# 确保参数已恢复
batch_restore_params(valid_gdirs)

forward_results = {}  # {rgi_id: {area_end, volume_end, ratio, ...}}
n_ok = 0
n_fail = 0

for gdir in valid_gdirs:
    rgi_id = gdir.rgi_id

    # 获取该冰川的best_bias
    row = df_constraint1[df_constraint1['rgi_id'] == rgi_id]
    if len(row) == 0 or pd.isna(row.iloc[0]['best_bias']):
        n_fail += 1
        continue
    best_bias = float(row.iloc[0]['best_bias'])

    # spinup后缀
    sp_suffix = '_inv_spinup_bias{}'.format(
        str(best_bias).replace('.', 'p').replace('-', 'n'))

    # 正演起始年（温度对接）
    start_year = bias_start_year.get(best_bias, FORWARD_START_YEAR)

    # 正演后缀
    fw_suffix = '_inv_forward'

    try:
        tasks.run_from_climate_data(
            gdir,
            min_ys=start_year,
            ye=FORWARD_END_YEAR,
            climate_filename='gcm_data',
            climate_input_filesuffix=CLIMATE_SUFFIX,
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=sp_suffix,
            output_filesuffix=fw_suffix,
        )

        # 提取结果
        fp = gdir.get_filepath('model_diagnostics', filesuffix=fw_suffix)
        with xr.open_dataset(fp) as ds:
            area_end = float(ds.area_m2.isel(time=-1)) / 1e6
            vol_end = float(ds.volume_m3.isel(time=-1)) / 1e9
            length_end = float(ds.length_m.isel(time=-1)) / 1e3

        rgi_area = rgi_area_by_id[rgi_id]
        lia_area = lia_area_by_mainrgi[rgi_id]
        sp_area = spinup_results.get(best_bias, {}).get(rgi_id, {}).get('area_km2', np.nan)

        forward_results[rgi_id] = {
            'best_bias': best_bias,
            'start_year': start_year,
            'rgi_area_km2': rgi_area,
            'lia_area_km2': lia_area,
            'sp_area_km2': sp_area,
            'fw_area_km2': area_end,
            'fw_volume_km3': vol_end,
            'fw_length_km': length_end,
            'sp_lia_ratio': sp_area / lia_area if lia_area > 0 else np.nan,
            'fw_rgi_ratio': area_end / rgi_area if rgi_area > 0 else np.nan,
        }
        n_ok += 1

    except Exception as e:
        print("  {} 正演失败: {}".format(rgi_id, str(e)[:80]))
        n_fail += 1

print("\n  正演完成: 成功={}, 失败={}".format(n_ok, n_fail))


# %% Cell 8 — 约束2: 正演终点 vs RGI + 综合分析
# ============================================================

print("\n[Step 8] 约束2分析: 正演终点 vs RGI面积")
print("=" * 100)

df_fw = pd.DataFrame(forward_results).T
df_fw.index.name = 'rgi_id'

if len(df_fw) == 0:
    print("  ⚠️ 没有正演结果，请检查前面的步骤")
else:
    # 打印逐冰川结果
    print("\n{:18s} {:>7s} {:>7s} {:>7s} {:>7s} {:>7s} {:>7s} {:>7s} {:>7s}".format(
        'RGI_ID', 'bias', 'Y_start', 'LIA', 'SP', 'FW', 'RGI', 'SP/LIA', 'FW/RGI'))
    print("-" * 100)

    for rgi_id, r in df_fw.iterrows():
        flag = ''
        if RGI_MATCH_TOL[0] <= r['fw_rgi_ratio'] <= RGI_MATCH_TOL[1]:
            flag = ' ✓'
        print("{:18s} {:7.2f} {:7.0f} {:7.2f} {:7.2f} {:7.2f} {:7.2f} {:7.3f} {:7.3f}{}".format(
            rgi_id, r['best_bias'], r['start_year'],
            r['lia_area_km2'], r['sp_area_km2'], r['fw_area_km2'], r['rgi_area_km2'],
            r['sp_lia_ratio'], r['fw_rgi_ratio'], flag))

    # 统计
    print("\n" + "=" * 100)
    print("综合统计:")
    print("  正演冰川数: {}".format(len(df_fw)))
    print("\n  约束1 (SP/LIA ratio):")
    print("    均值={:.3f}, 中位数={:.3f}, std={:.3f}".format(
        df_fw['sp_lia_ratio'].mean(), df_fw['sp_lia_ratio'].median(), df_fw['sp_lia_ratio'].std()))

    print("\n  约束2 (FW/RGI ratio):")
    print("    均值={:.3f}, 中位数={:.3f}, std={:.3f}".format(
        df_fw['fw_rgi_ratio'].mean(), df_fw['fw_rgi_ratio'].median(), df_fw['fw_rgi_ratio'].std()))

    n_match = len(df_fw[(df_fw['fw_rgi_ratio'] >= RGI_MATCH_TOL[0]) &
                         (df_fw['fw_rgi_ratio'] <= RGI_MATCH_TOL[1])])
    print("    在 {} 内: {}/{} = {:.1f}%".format(
        RGI_MATCH_TOL, n_match, len(df_fw), 100 * n_match / len(df_fw)))

    print("\n  best_bias 分布:")
    print("    均值={:.2f}, 中位数={:.2f}, std={:.2f}".format(
        df_fw['best_bias'].mean(), df_fw['best_bias'].median(), df_fw['best_bias'].std()))

    # 两个约束的一致性分析
    both_ok = df_fw[
        (df_fw['sp_lia_ratio'] >= LIA_MATCH_TOL[0]) &
        (df_fw['sp_lia_ratio'] <= LIA_MATCH_TOL[1]) &
        (df_fw['fw_rgi_ratio'] >= RGI_MATCH_TOL[0]) &
        (df_fw['fw_rgi_ratio'] <= RGI_MATCH_TOL[1])
    ]
    print("\n  同时满足两个约束: {}/{} = {:.1f}%".format(
        len(both_ok), len(df_fw), 100 * len(both_ok) / max(len(df_fw), 1)))

    # 保存
    df_fw.to_csv(os.path.join(OUTPUT_DIR, 'forward_results.csv'))
    print("\n  已保存: forward_results.csv")


# %% Cell 9 — 可视化
# ============================================================

if len(df_fw) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # --- (a) best_bias 分布直方图 ---
    ax = axes[0, 0]
    ax.hist(df_fw['best_bias'], bins=len(TEMP_BIASES), edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(df_fw['best_bias'].median(), color='red', ls='--', label='median={:.2f}'.format(df_fw['best_bias'].median()))
    ax.set_xlabel('Best Temperature Bias (K)')
    ax.set_ylabel('Count')
    ax.set_title('(a) LIA Cooling Estimate Distribution')
    ax.legend()

    # --- (b) SP/LIA ratio vs FW/RGI ratio 散点图 ---
    ax = axes[0, 1]
    sc = ax.scatter(df_fw['sp_lia_ratio'], df_fw['fw_rgi_ratio'],
                    c=df_fw['best_bias'], cmap='coolwarm', s=60, edgecolor='k', linewidth=0.5)
    ax.axhline(1.0, color='gray', ls='--', lw=0.8)
    ax.axvline(1.0, color='gray', ls='--', lw=0.8)
    ax.axhspan(RGI_MATCH_TOL[0], RGI_MATCH_TOL[1], alpha=0.1, color='green')
    ax.axvspan(LIA_MATCH_TOL[0], LIA_MATCH_TOL[1], alpha=0.1, color='blue')
    ax.set_xlabel('Constraint 1: SP / LIA area')
    ax.set_ylabel('Constraint 2: FW / RGI area')
    ax.set_title('(b) Dual Constraint Consistency')
    plt.colorbar(sc, ax=ax, label='Temp Bias (K)')

    # --- (c) FW/RGI ratio 分布 ---
    ax = axes[1, 0]
    ax.hist(df_fw['fw_rgi_ratio'], bins=15, edgecolor='black', alpha=0.7, color='darkorange')
    ax.axvline(1.0, color='red', ls='--', lw=1.5)
    ax.axvspan(RGI_MATCH_TOL[0], RGI_MATCH_TOL[1], alpha=0.15, color='green', label='target range')
    ax.set_xlabel('FW(2000) / RGI area ratio')
    ax.set_ylabel('Count')
    ax.set_title('(c) Forward Validation (Constraint 2)')
    ax.legend()

    # --- (d) best_bias vs LIA/RGI面积比 ---
    ax = axes[1, 1]
    ax.scatter(df_fw['lia_area_km2'] / df_fw['rgi_area_km2'], df_fw['best_bias'],
               s=60, edgecolor='k', linewidth=0.5, color='steelblue')
    ax.set_xlabel('LIA / RGI area ratio')
    ax.set_ylabel('Best Temperature Bias (K)')
    ax.set_title('(d) Cooling vs Glacier Change')

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, 'inverse_evolution_results.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("  图已保存: {}".format(fig_path))


# %% Cell 10 — 单冰川详细诊断（可选）
# ============================================================
# 选一条代表性冰川，画spinup+正演的完整时间序列
# ============================================================

if len(df_fw) > 0:
    # 选FW/RGI ratio最接近1.0的冰川
    best_idx = (df_fw['fw_rgi_ratio'] - 1.0).abs().idxmin()
    diag_rgi_id = best_idx
    diag_row = df_fw.loc[diag_rgi_id]
    diag_gdir = [g for g in valid_gdirs if g.rgi_id == diag_rgi_id][0]

    print("\n[诊断] 冰川: {}".format(diag_rgi_id))
    print("  best_bias={:.2f}, SP/LIA={:.3f}, FW/RGI={:.3f}".format(
        diag_row['best_bias'], diag_row['sp_lia_ratio'], diag_row['fw_rgi_ratio']))

    best_bias = diag_row['best_bias']
    sp_suffix = '_inv_spinup_bias{}'.format(
        str(best_bias).replace('.', 'p').replace('-', 'n'))
    fw_suffix = '_inv_forward'

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

    # Spinup演化
    try:
        fp_sp = diag_gdir.get_filepath('model_diagnostics', filesuffix=sp_suffix)
        with xr.open_dataset(fp_sp) as ds_sp:
            t_sp = ds_sp.time.values
            a_sp = ds_sp.area_m2.values / 1e6
            v_sp = ds_sp.volume_m3.values / 1e9
            l_sp = ds_sp.length_m.values / 1e3

        for i, (y, label) in enumerate([(a_sp, 'Area (km²)'), (v_sp, 'Volume (km³)'), (l_sp, 'Length (km)')]):
            axes[i].plot(t_sp, y, 'b-', lw=1, label='Spinup')
    except:
        pass

    # 正演演化
    try:
        fp_fw = diag_gdir.get_filepath('model_diagnostics', filesuffix=fw_suffix)
        with xr.open_dataset(fp_fw) as ds_fw:
            t_fw = ds_fw.time.values
            a_fw = ds_fw.area_m2.values / 1e6
            v_fw = ds_fw.volume_m3.values / 1e9
            l_fw = ds_fw.length_m.values / 1e3

        # 时间轴偏移: spinup结束 → 正演开始
        t_fw_shifted = t_fw + float(t_sp[-1]) + 1  # 接在spinup后面

        for i, (y, label) in enumerate([(a_fw, 'Area (km²)'), (v_fw, 'Volume (km³)'), (l_fw, 'Length (km)')]):
            axes[i].plot(t_fw_shifted, y, 'r-', lw=1, label='Forward')
            axes[i].set_ylabel(label)
    except:
        pass

    # 参考线
    rgi_area = diag_row['rgi_area_km2']
    lia_area = diag_row['lia_area_km2']
    axes[0].axhline(rgi_area, color='green', ls='--', lw=1, label='RGI={:.1f}'.format(rgi_area))
    axes[0].axhline(lia_area, color='purple', ls='--', lw=1, label='LIA={:.1f}'.format(lia_area))

    for ax in axes:
        ax.legend(loc='best', fontsize=8)
        ax.grid(True, alpha=0.3)

    axes[0].set_title('{} | bias={:.2f}K | SP/LIA={:.3f} | FW/RGI={:.3f}'.format(
        diag_rgi_id, best_bias, diag_row['sp_lia_ratio'], diag_row['fw_rgi_ratio']))
    axes[-1].set_xlabel('Time (spinup years → calendar years)')

    plt.tight_layout()
    diag_path = os.path.join(OUTPUT_DIR, 'diagnostic_{}.png'.format(diag_rgi_id.replace('.', '_')))
    plt.savefig(diag_path, dpi=150, bbox_inches='tight')
    plt.show()
    print("  诊断图已保存: {}".format(diag_path))


# %% Cell 11 — 实验总结
# ============================================================

print("\n" + "=" * 70)
print("实验总结")
print("=" * 70)

if len(df_fw) > 0:
    print("""
逆演化 + 正演验证结果:

1. LIA降温估计 (best_bias):
   均值 = {:.2f} K, 中位数 = {:.2f} K
   含义: LIA时期比现代(1970-2000)冷 {:.2f} K (中位数)

2. 约束1 (spinup面积 vs LIA遥感面积):
   SP/LIA ratio 均值 = {:.3f}
   在 {tol1} 容差内: {n1} / {total} = {p1:.1f}%

3. 约束2 (正演终点 vs RGI观测面积):
   FW/RGI ratio 均值 = {:.3f}
   在 {tol2} 容差内: {n2} / {total} = {p2:.1f}%

4. 双约束同时满足:
   {n_both} / {total} = {p_both:.1f}%

5. 下一步:
   - 如果双约束一致性好: 扩大到全量冰川，得到区域LIA降温估计
   - 如果约束2系统偏高: 重建气候的变暖趋势偏弱，可量化偏差
   - 如果约束2系统偏低: spinup过度增长或重建气候偏暖
""".format(
        df_fw['best_bias'].mean(), df_fw['best_bias'].median(),
        abs(df_fw['best_bias'].median()),
        df_fw['sp_lia_ratio'].mean(),
        tol1=LIA_MATCH_TOL,
        n1=len(df_fw[(df_fw['sp_lia_ratio'] >= LIA_MATCH_TOL[0]) & (df_fw['sp_lia_ratio'] <= LIA_MATCH_TOL[1])]),
        total=len(df_fw),
        p1=100 * len(df_fw[(df_fw['sp_lia_ratio'] >= LIA_MATCH_TOL[0]) & (df_fw['sp_lia_ratio'] <= LIA_MATCH_TOL[1])]) / len(df_fw),
        tol2=RGI_MATCH_TOL,
        n2=len(df_fw[(df_fw['fw_rgi_ratio'] >= RGI_MATCH_TOL[0]) & (df_fw['fw_rgi_ratio'] <= RGI_MATCH_TOL[1])]),
        p2=100 * len(df_fw[(df_fw['fw_rgi_ratio'] >= RGI_MATCH_TOL[0]) & (df_fw['fw_rgi_ratio'] <= RGI_MATCH_TOL[1])]) / len(df_fw),
        n_both=len(df_fw[
            (df_fw['sp_lia_ratio'] >= LIA_MATCH_TOL[0]) & (df_fw['sp_lia_ratio'] <= LIA_MATCH_TOL[1]) &
            (df_fw['fw_rgi_ratio'] >= RGI_MATCH_TOL[0]) & (df_fw['fw_rgi_ratio'] <= RGI_MATCH_TOL[1])
        ]),
        p_both=100 * len(df_fw[
            (df_fw['sp_lia_ratio'] >= LIA_MATCH_TOL[0]) & (df_fw['sp_lia_ratio'] <= LIA_MATCH_TOL[1]) &
            (df_fw['fw_rgi_ratio'] >= RGI_MATCH_TOL[0]) & (df_fw['fw_rgi_ratio'] <= RGI_MATCH_TOL[1])
        ]) / len(df_fw),
    ))